In [1]:
# ============ 必须在 import sklearn 之前 ============
import threadpoolctl

class _SafeThreadpoolLimits:
    def __init__(self, *args, **kwargs):
        pass
    def __enter__(self):
        return self
    def __exit__(self, *args):
        pass

threadpoolctl.threadpool_limits = _SafeThreadpoolLimits
# ==================================================

import pandas as pd
import numpy as np
import os
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              precision_recall_curve, f1_score,
                              recall_score, precision_score,
                              confusion_matrix, brier_score_loss)
from sklearn.impute import SimpleImputer
import lightgbm as lgb

try:
    from imblearn.over_sampling import SMOTE
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False
    print("未安装 imblearn，请先运行: pip install imbalanced-learn")

data_path = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\建模数据集_方案A_MDA_未隔离.csv"
out_dir = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"

data = pd.read_csv(data_path)
exclude_cols = ["Stkcd", "year", "Fraud", "ShortName", "IndustryName1",
                "ViolationTypeID", "DeclareDate", "DisposalDate", "Enddate", "set"]
all_numeric = [c for c in data.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(data[c])]

y = data["Fraud"].values
groups = data["Stkcd"].values
X_all = data[all_numeric].values

print("特征数:", len(all_numeric), "  样本数:", len(y), "  正例:", int(y.sum()))

def find_best_threshold(y_true, y_prob):
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
    i = np.argmax(f1s)
    return float(thr[i])

def compute_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        'AUC': roc_auc_score(y_true, y_prob),
        'PR_AUC': average_precision_score(y_true, y_prob),
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Specificity': tn / (tn + fp) if (tn + fp) > 0 else np.nan,
        'Brier': brier_score_loss(y_true, y_prob),
    }

lgb_params = dict(
    n_estimators=500, learning_rate=0.05, num_leaves=31,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
    random_state=42, n_jobs=-1, verbose=-1
)

gkf = GroupKFold(n_splits=5)
results = {'A_classweight': [], 'B_SMOTE': [], 'C_SMOTE_classweight': []}

print("\n开始 3 种不平衡处理方案对比...")
t_total = time.time()

for fold, (tr, te) in enumerate(gkf.split(X_all, y, groups), 1):
    print(f"\n===== Fold {fold} =====")
    X_tr_raw, X_te_raw = X_all[tr], X_all[te]
    y_tr, y_te = y[tr], y[te]
    g_tr = groups[tr]

    imputer = SimpleImputer(strategy='median')
    X_tr = imputer.fit_transform(X_tr_raw)
    X_te = imputer.transform(X_te_raw)

    # A
    m = lgb.LGBMClassifier(class_weight='balanced', **lgb_params)
    m.fit(X_tr, y_tr)
    p_a = m.predict_proba(X_te)[:, 1]
    inner_gkf = GroupKFold(n_splits=3)
    oof = np.zeros(len(tr))
    for i_tr, i_val in inner_gkf.split(X_tr, y_tr, g_tr):
        imp_i = SimpleImputer(strategy='median')
        X_i = imp_i.fit_transform(X_tr[i_tr])
        X_v = imp_i.transform(X_tr[i_val])
        mi = lgb.LGBMClassifier(class_weight='balanced', **lgb_params)
        mi.fit(X_i, y_tr[i_tr])
        oof[i_val] = mi.predict_proba(X_v)[:, 1]
    thr_a = find_best_threshold(y_tr, oof)
    results['A_classweight'].append(compute_metrics(y_te, p_a, thr_a))
    print(f"  A(class_weight):       AUC={results['A_classweight'][-1]['AUC']:.4f}  "
          f"PR-AUC={results['A_classweight'][-1]['PR_AUC']:.4f}  "
          f"F1={results['A_classweight'][-1]['F1']:.4f}")

    if HAS_IMBLEARN:
        # B
        X_sm, y_sm = SMOTE(random_state=42, k_neighbors=5).fit_resample(X_tr, y_tr)
        m = lgb.LGBMClassifier(**lgb_params)
        m.fit(X_sm, y_sm)
        p_b = m.predict_proba(X_te)[:, 1]

        oof = np.zeros(len(tr))
        for i_tr, i_val in inner_gkf.split(X_tr, y_tr, g_tr):
            imp_i = SimpleImputer(strategy='median')
            X_i = imp_i.fit_transform(X_tr[i_tr])
            X_v = imp_i.transform(X_tr[i_val])
            X_i_sm, y_i_sm = SMOTE(random_state=42, k_neighbors=5).fit_resample(X_i, y_tr[i_tr])
            mi = lgb.LGBMClassifier(**lgb_params)
            mi.fit(X_i_sm, y_i_sm)
            oof[i_val] = mi.predict_proba(X_v)[:, 1]
        thr_b = find_best_threshold(y_tr, oof)
        results['B_SMOTE'].append(compute_metrics(y_te, p_b, thr_b))
        print(f"  B(SMOTE):              AUC={results['B_SMOTE'][-1]['AUC']:.4f}  "
              f"PR-AUC={results['B_SMOTE'][-1]['PR_AUC']:.4f}  "
              f"F1={results['B_SMOTE'][-1]['F1']:.4f}")

        # C
        X_sm, y_sm = SMOTE(random_state=42, k_neighbors=5).fit_resample(X_tr, y_tr)
        m = lgb.LGBMClassifier(class_weight='balanced', **lgb_params)
        m.fit(X_sm, y_sm)
        p_c = m.predict_proba(X_te)[:, 1]

        oof = np.zeros(len(tr))
        for i_tr, i_val in inner_gkf.split(X_tr, y_tr, g_tr):
            imp_i = SimpleImputer(strategy='median')
            X_i = imp_i.fit_transform(X_tr[i_tr])
            X_v = imp_i.transform(X_tr[i_val])
            X_i_sm, y_i_sm = SMOTE(random_state=42, k_neighbors=5).fit_resample(X_i, y_tr[i_tr])
            mi = lgb.LGBMClassifier(class_weight='balanced', **lgb_params)
            mi.fit(X_i_sm, y_i_sm)
            oof[i_val] = mi.predict_proba(X_v)[:, 1]
        thr_c = find_best_threshold(y_tr, oof)
        results['C_SMOTE_classweight'].append(compute_metrics(y_te, p_c, thr_c))
        print(f"  C(SMOTE+cw):           AUC={results['C_SMOTE_classweight'][-1]['AUC']:.4f}  "
              f"PR-AUC={results['C_SMOTE_classweight'][-1]['PR_AUC']:.4f}  "
              f"F1={results['C_SMOTE_classweight'][-1]['F1']:.4f}")

print(f"\n总用时: {(time.time()-t_total)/60:.1f} 分钟")

print("\n" + "="*80)
print("三种不平衡处理方案对比 (5折均值)")
print("="*80)
summary = []
for k, v in results.items():
    if not v:
        continue
    df = pd.DataFrame(v)
    row = {'Scheme': k}
    for col in ['AUC','PR_AUC','F1','Recall','Precision','Specificity','Brier']:
        row[col] = f"{df[col].mean():.4f}±{df[col].std():.4f}"
    summary.append(row)
summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

summary_df.to_csv(os.path.join(out_dir, "SMOTE对比.csv"), index=False, encoding='utf-8-sig')
print("\n已保存: SMOTE对比.csv")

for k, v in results.items():
    if not v:
        continue
    print(f"\n{k} 每折:")
    for i, r in enumerate(v, 1):
        print(f"  Fold{i}: AUC={r['AUC']:.4f} PR-AUC={r['PR_AUC']:.4f} "
              f"F1={r['F1']:.4f} Recall={r['Recall']:.4f} Prec={r['Precision']:.4f}")

特征数: 83   样本数: 48328   正例: 4233

开始 3 种不平衡处理方案对比...

===== Fold 1 =====
  A(class_weight):       AUC=0.7826  PR-AUC=0.3140  F1=0.3331
  B(SMOTE):              AUC=0.7580  PR-AUC=0.2905  F1=0.3102
  C(SMOTE+cw):           AUC=0.7580  PR-AUC=0.2905  F1=0.3102

===== Fold 2 =====
  A(class_weight):       AUC=0.7715  PR-AUC=0.2495  F1=0.3219
  B(SMOTE):              AUC=0.7626  PR-AUC=0.2414  F1=0.3156
  C(SMOTE+cw):           AUC=0.7626  PR-AUC=0.2414  F1=0.3156

===== Fold 3 =====
  A(class_weight):       AUC=0.7741  PR-AUC=0.2932  F1=0.3320
  B(SMOTE):              AUC=0.7703  PR-AUC=0.2942  F1=0.3353
  C(SMOTE+cw):           AUC=0.7703  PR-AUC=0.2942  F1=0.3353

===== Fold 4 =====
  A(class_weight):       AUC=0.7742  PR-AUC=0.2812  F1=0.3390
  B(SMOTE):              AUC=0.7599  PR-AUC=0.2724  F1=0.3355
  C(SMOTE+cw):           AUC=0.7599  PR-AUC=0.2724  F1=0.3355

===== Fold 5 =====
  A(class_weight):       AUC=0.7704  PR-AUC=0.2954  F1=0.3270
  B(SMOTE):              AUC=0.7655  PR-AU

In [2]:
# ============ 必须放在最前面 ============
import threadpoolctl

class _SafeThreadpoolLimits:
    def __init__(self, *args, **kwargs):
        pass
    def __enter__(self):
        return self
    def __exit__(self, *args):
        pass

threadpoolctl.threadpool_limits = _SafeThreadpoolLimits
# ==================================================

import pandas as pd
import numpy as np
import os
import time
import warnings
warnings.filterwarnings('ignore')

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              precision_recall_curve, f1_score,
                              recall_score, precision_score,
                              confusion_matrix, brier_score_loss)
from sklearn.impute import SimpleImputer
import lightgbm as lgb

data_path = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\建模数据集_方案A_MDA_未隔离.csv"
out_dir = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"

data = pd.read_csv(data_path)
exclude_cols = ["Stkcd", "year", "Fraud", "ShortName", "IndustryName1",
                "ViolationTypeID", "DeclareDate", "DisposalDate", "Enddate", "set"]
all_numeric = [c for c in data.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(data[c])]

y = data["Fraud"].values
groups = data["Stkcd"].values
X_all = data[all_numeric].values

print("特征数:", len(all_numeric), "  样本数:", len(y), "  正例:", int(y.sum()))

def find_best_threshold(y_true, y_prob):
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
    i = np.argmax(f1s)
    return float(thr[i])

def compute_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        'AUC': roc_auc_score(y_true, y_prob),
        'PR_AUC': average_precision_score(y_true, y_prob),
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Specificity': tn / (tn + fp) if (tn + fp) > 0 else np.nan,
        'Brier': brier_score_loss(y_true, y_prob),
    }

gkf = GroupKFold(n_splits=5)

N_TRIALS = 30  # 每折 30 次搜索

# ================= Optuna 目标函数 =================
def make_objective(X_tr, y_tr, g_tr, inner_gkf):
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 300, 1000, step=100),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 15, 63),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 1.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 1.0, log=True),
            'class_weight': 'balanced',
            'random_state': 42,
            'n_jobs': -1,
            'verbose': -1
        }

        oof_prob = np.zeros(len(X_tr))
        for i_tr, i_val in inner_gkf.split(X_tr, y_tr, g_tr):
            imp_i = SimpleImputer(strategy='median')
            X_i = imp_i.fit_transform(X_tr[i_tr])
            X_v = imp_i.transform(X_tr[i_val])
            m = lgb.LGBMClassifier(**params)
            m.fit(X_i, y_tr[i_tr])
            oof_prob[i_val] = m.predict_proba(X_v)[:, 1]

        return average_precision_score(y_tr, oof_prob)
    return objective

# ================= 主循环：每折独立调参 =================
best_params_per_fold = []
all_metrics = []
all_probs = np.zeros(len(data))

t_start = time.time()
for fold, (tr, te) in enumerate(gkf.split(X_all, y, groups), 1):
    print(f"\n{'='*60}\nFold {fold} | Optuna 调参中...\n{'='*60}")
    X_tr_raw, X_te_raw = X_all[tr], X_all[te]
    y_tr, y_te = y[tr], y[te]
    g_tr = groups[tr]

    inner_gkf = GroupKFold(n_splits=3)
    objective = make_objective(X_tr_raw, y_tr, g_tr, inner_gkf)

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42)
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

    best_params = study.best_params
    best_params['class_weight'] = 'balanced'
    best_params['random_state'] = 42
    best_params['n_jobs'] = -1
    best_params['verbose'] = -1
    best_params_per_fold.append(best_params)
    print(f"Fold {fold} 最优参数: {study.best_params}")
    print(f"Fold {fold} 最优内层 PR-AUC: {study.best_value:.4f}")

    # 用最优参数在完整训练集训练，测试
    imp = SimpleImputer(strategy='median')
    X_tr = imp.fit_transform(X_tr_raw)
    X_te = imp.transform(X_te_raw)

    m = lgb.LGBMClassifier(**best_params)
    m.fit(X_tr, y_tr)
    y_prob = m.predict_proba(X_te)[:, 1]
    all_probs[te] = y_prob

    # 阈值
    oof = np.zeros(len(tr))
    for i_tr, i_val in inner_gkf.split(X_tr, y_tr, g_tr):
        imp_i = SimpleImputer(strategy='median')
        X_i = imp_i.fit_transform(X_tr[i_tr])
        X_v = imp_i.transform(X_tr[i_val])
        mi = lgb.LGBMClassifier(**best_params)
        mi.fit(X_i, y_tr[i_tr])
        oof[i_val] = mi.predict_proba(X_v)[:, 1]
    thr = find_best_threshold(y_tr, oof)

    met = compute_metrics(y_te, y_prob, thr)
    met['fold'] = fold
    all_metrics.append(met)
    print(f"Fold {fold} 测试: AUC={met['AUC']:.4f} PR-AUC={met['PR_AUC']:.4f} "
          f"F1={met['F1']:.4f} Recall={met['Recall']:.4f} Prec={met['Precision']:.4f} "
          f"Spec={met['Specificity']:.4f} Brier={met['Brier']:.4f}")

print(f"\n总用时: {(time.time()-t_start)/60:.1f} 分钟")

# ================= 汇总 =================
met_df = pd.DataFrame(all_metrics)
print("\n" + "="*80)
print("Optuna 调优后结果 (5折均值)")
print("="*80)
for col in ['AUC','PR_AUC','F1','Recall','Precision','Specificity','Brier']:
    print(f"  {col:12s}: {met_df[col].mean():.4f} ± {met_df[col].std():.4f}")

# 对比之前的默认参数
print("\n对比:")
print("  默认参数 LightGBM:  AUC=0.7754 PR-AUC=0.2908 F1=0.3248")
print(f"  Optuna调优后:       AUC={met_df['AUC'].mean():.4f} PR-AUC={met_df['PR_AUC'].mean():.4f} F1={met_df['F1'].mean():.4f}")

# ================= 保存 =================
met_df.to_csv(os.path.join(out_dir, "Optuna_metrics.csv"), index=False, encoding='utf-8-sig')
pd.DataFrame(best_params_per_fold).to_csv(os.path.join(out_dir, "Optuna_best_params.csv"),
                                           index=False, encoding='utf-8-sig')
pd.DataFrame({
    'y_true': y,
    'y_prob_optuna': all_probs
}).to_csv(os.path.join(out_dir, "Optuna_OOF_predictions.csv"), index=False, encoding='utf-8-sig')

print("\n已保存:")
print("  Optuna_metrics.csv")
print("  Optuna_best_params.csv")
print("  Optuna_OOF_predictions.csv")

特征数: 83   样本数: 48328   正例: 4233

Fold 1 | Optuna 调参中...
Fold 1 最优参数: {'n_estimators': 400, 'learning_rate': 0.018767308386671636, 'num_leaves': 25, 'min_child_samples': 16, 'subsample': 0.7530024954141907, 'colsample_bytree': 0.7150838961324845, 'reg_alpha': 0.11967494591692165, 'reg_lambda': 0.2362834044057798}
Fold 1 最优内层 PR-AUC: 0.2870
Fold 1 测试: AUC=0.7966 PR-AUC=0.3344 F1=0.3477 Recall=0.4679 Prec=0.2766 Spec=0.8760 Brier=0.1573

Fold 2 | Optuna 调参中...
Fold 2 最优参数: {'n_estimators': 500, 'learning_rate': 0.01252374543467487, 'num_leaves': 34, 'min_child_samples': 16, 'subsample': 0.998490153762098, 'colsample_bytree': 0.725561495209942, 'reg_alpha': 0.2535083125664467, 'reg_lambda': 0.34077101571145857}
Fold 2 最优内层 PR-AUC: 0.3094
Fold 2 测试: AUC=0.7770 PR-AUC=0.2670 F1=0.3220 Recall=0.4791 Prec=0.2425 Spec=0.8670 Brier=0.1558

Fold 3 | Optuna 调参中...
Fold 3 最优参数: {'n_estimators': 400, 'learning_rate': 0.014341699767827105, 'num_leaves': 18, 'min_child_samples': 31, 'subsample': 0.792

In [3]:
# ============ 必须放在最前面 ============
import threadpoolctl

class _SafeThreadpoolLimits:
    def __init__(self, *args, **kwargs):
        pass
    def __enter__(self):
        return self
    def __exit__(self, *args):
        pass

threadpoolctl.threadpool_limits = _SafeThreadpoolLimits
# ==================================================

import pandas as pd
import numpy as np
import os
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              precision_recall_curve, f1_score,
                              recall_score, precision_score,
                              confusion_matrix, brier_score_loss)
from sklearn.impute import SimpleImputer
from sklearn.calibration import CalibratedClassifierCV
from scipy import stats
import lightgbm as lgb

data_path = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\建模数据集_方案A_MDA_未隔离.csv"
out_dir = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"

data = pd.read_csv(data_path)
exclude_cols = ["Stkcd", "year", "Fraud", "ShortName", "IndustryName1",
                "ViolationTypeID", "DeclareDate", "DisposalDate", "Enddate", "set"]
all_numeric = [c for c in data.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(data[c])]

y = data["Fraud"].values
groups = data["Stkcd"].values
X_all = data[all_numeric].values

print("特征数:", len(all_numeric), "  样本数:", len(y), "  正例:", int(y.sum()))

# ================= Optuna 最优参数（5折平均，取Fold1~5里最常出现的组合）=================
# 你可以从 Optuna_best_params.csv 读取，这里手动取一个"中庸"参数
best_params = dict(
    n_estimators=400,
    learning_rate=0.015,
    num_leaves=25,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.75,
    reg_alpha=0.15,
    reg_lambda=0.2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

def find_best_threshold(y_true, y_prob):
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
    i = np.argmax(f1s)
    return float(thr[i])

def compute_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        'AUC': roc_auc_score(y_true, y_prob),
        'PR_AUC': average_precision_score(y_true, y_prob),
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Specificity': tn / (tn + fp) if (tn + fp) > 0 else np.nan,
        'Brier': brier_score_loss(y_true, y_prob),
    }

# ================= DeLong 检验（内置实现）=================
def compute_midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T
    return T2

def fastDeLong(predictions_sorted_transposed, label_1_count):
    m = label_1_count
    n = predictions_sorted_transposed.shape[1] - m
    positive = predictions_sorted_transposed[:, :m]
    negative = predictions_sorted_transposed[:, m:]
    k = predictions_sorted_transposed.shape[0]
    tx = np.empty([k, m]); ty = np.empty([k, n]); tz = np.empty([k, m + n])
    for r in range(k):
        tx[r] = compute_midrank(positive[r])
        ty[r] = compute_midrank(negative[r])
        tz[r] = compute_midrank(predictions_sorted_transposed[r])
    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1) / 2.0 / n
    v01 = (tz[:, :m] - tx[:, :]) / n
    v10 = 1.0 - (tz[:, m:] - ty[:, :]) / m
    sx = np.cov(v01); sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, delongcov

def delong_roc_test(y_true, prob1, prob2):
    order = np.argsort(-y_true)
    label_1_count = int(y_true.sum())
    predictions_sorted = np.vstack([prob1, prob2])[:, order]
    aucs, delongcov = fastDeLong(predictions_sorted, label_1_count)
    l = np.array([[1, -1]])
    z = np.abs(np.diff(aucs)) / np.sqrt(np.dot(np.dot(l, delongcov), l.T))
    p = 2 * stats.norm.sf(z)
    return aucs, z[0], p[0]

# ================= 主循环：Optuna + Isotonic 校准 =================
gkf = GroupKFold(n_splits=5)

probs_raw = np.zeros(len(data))
probs_cal = np.zeros(len(data))
metrics_raw = []
metrics_cal = []
thresholds_cal = []

t_start = time.time()
for fold, (tr, te) in enumerate(gkf.split(X_all, y, groups), 1):
    print(f"\n===== Fold {fold} =====")
    X_tr_raw, X_te_raw = X_all[tr], X_all[te]
    y_tr, y_te = y[tr], y[te]
    g_tr = groups[tr]

    imp = SimpleImputer(strategy='median')
    X_tr = imp.fit_transform(X_tr_raw)
    X_te = imp.transform(X_te_raw)

    # ---- 原始概率 ----
    m = lgb.LGBMClassifier(**best_params)
    m.fit(X_tr, y_tr)
    p_raw = m.predict_proba(X_te)[:, 1]
    probs_raw[te] = p_raw

    # ---- Isotonic 校准（用内层CV做）----
    inner_gkf = GroupKFold(n_splits=3)
    # CalibratedClassifierCV 内部用 cv=3，但需要传入 groups，它不一定支持
    # 简单做法：手动做 Isotonic 校准
    from sklearn.isotonic import IsotonicRegression

    # 用内层 OOF 概率拟合 Isotonic
    oof_prob = np.zeros(len(tr))
    for i_tr, i_val in inner_gkf.split(X_tr, y_tr, g_tr):
        imp_i = SimpleImputer(strategy='median')
        X_i = imp_i.fit_transform(X_tr[i_tr])
        X_v = imp_i.transform(X_tr[i_val])
        mi = lgb.LGBMClassifier(**best_params)
        mi.fit(X_i, y_tr[i_tr])
        oof_prob[i_val] = mi.predict_proba(X_v)[:, 1]

    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(oof_prob, y_tr)
    p_cal = iso.predict(p_raw)
    probs_cal[te] = p_cal

    # ---- 阈值：基于校准后的内层 OOF ----
    oof_cal = iso.predict(oof_prob)
    thr = find_best_threshold(y_tr, oof_cal)
    thresholds_cal.append(thr)

    met_raw = compute_metrics(y_te, p_raw, 0.5)
    met_cal = compute_metrics(y_te, p_cal, thr)
    met_raw['fold'] = fold; met_cal['fold'] = fold
    metrics_raw.append(met_raw); metrics_cal.append(met_cal)

    print(f"  原始:  AUC={met_raw['AUC']:.4f} PR-AUC={met_raw['PR_AUC']:.4f} "
          f"Brier={met_raw['Brier']:.4f}")
    print(f"  校准:  AUC={met_cal['AUC']:.4f} PR-AUC={met_cal['PR_AUC']:.4f} "
          f"F1={met_cal['F1']:.4f} Recall={met_cal['Recall']:.4f} "
          f"Prec={met_cal['Precision']:.4f} Spec={met_cal['Specificity']:.4f} "
          f"Brier={met_cal['Brier']:.4f}")

print(f"\n总用时: {(time.time()-t_start)/60:.1f} 分钟")

# ================= 汇总 =================
df_raw = pd.DataFrame(metrics_raw)
df_cal = pd.DataFrame(metrics_cal)

print("\n" + "="*80)
print("校准前后对比 (5折均值)")
print("="*80)
for col in ['AUC','PR_AUC','F1','Recall','Precision','Specificity','Brier']:
    print(f"  {col:12s}: 原始 {df_raw[col].mean():.4f}±{df_raw[col].std():.4f}  "
          f"校准 {df_cal[col].mean():.4f}±{df_cal[col].std():.4f}")

# ================= DeLong 检验 =================
print("\n" + "="*80)
print("DeLong 检验：Optuna 版本 vs 默认版本")
print("="*80)

# 读之前的默认版本OOF（LightGBM Full）
try:
    pred_default = pd.read_csv(os.path.join(out_dir, "对比_OOF_predictions.csv"))
    prob_default_lgb = pred_default["Full_LightGBM"].values
    # 注意：默认版本OOF和Optuna版本OOF的对齐方式不同（都是按索引顺序）
    # 对比时要用相同的y_true，两个概率数组的索引必须一致
    y_check = pred_default["y_true"].values
    if np.array_equal(y_check, y):
        aucs, z, p = delong_roc_test(y, probs_raw, prob_default_lgb)
        print(f"Optuna AUC = {aucs[0]:.4f}  vs  默认 AUC = {aucs[1]:.4f}")
        print(f"ΔAUC = {aucs[0]-aucs[1]:+.4f}")
        print(f"DeLong Z = {z:.4f}, p = {p:.6f}")
        print(f"结论: {'显著' if p < 0.05 else '不显著'} (α=0.05)")
    else:
        print("OOF 索引不匹配，跳过 DeLong。请重新运行默认参数版本的对比代码。")
except FileNotFoundError:
    print("未找到对比_OOF_predictions.csv，跳过 DeLong")

# ================= Bootstrap 95% CI =================
print("\n" + "="*80)
print("Bootstrap 95% CI (1000次)")
print("="*80)
rng = np.random.default_rng(42)
n = len(y)
boot_auc, boot_prauc, boot_brier = [], [], []
for _ in range(1000):
    idx = rng.integers(0, n, n)
    if len(np.unique(y[idx])) < 2:
        continue
    boot_auc.append(roc_auc_score(y[idx], probs_cal[idx]))
    boot_prauc.append(average_precision_score(y[idx], probs_cal[idx]))
    boot_brier.append(brier_score_loss(y[idx], probs_cal[idx]))
boot_auc = np.array(boot_auc); boot_prauc = np.array(boot_prauc); boot_brier = np.array(boot_brier)

print(f"  AUC:     {np.mean(boot_auc):.4f}  [{np.percentile(boot_auc, 2.5):.4f}, {np.percentile(boot_auc, 97.5):.4f}]")
print(f"  PR-AUC:  {np.mean(boot_prauc):.4f}  [{np.percentile(boot_prauc, 2.5):.4f}, {np.percentile(boot_prauc, 97.5):.4f}]")
print(f"  Brier:   {np.mean(boot_brier):.4f}  [{np.percentile(boot_brier, 2.5):.4f}, {np.percentile(boot_brier, 97.5):.4f}]")

# ================= 保存 =================
df_raw.to_csv(os.path.join(out_dir, "Optuna_calibration_raw.csv"), index=False, encoding='utf-8-sig')
df_cal.to_csv(os.path.join(out_dir, "Optuna_calibration_calibrated.csv"), index=False, encoding='utf-8-sig')
pd.DataFrame({
    'y_true': y,
    'y_prob_raw': probs_raw,
    'y_prob_calibrated': probs_cal
}).to_csv(os.path.join(out_dir, "Optuna_calibrated_predictions.csv"), index=False, encoding='utf-8-sig')

print("\n已保存:")
print("  Optuna_calibration_raw.csv")
print("  Optuna_calibration_calibrated.csv")
print("  Optuna_calibrated_predictions.csv")

特征数: 83   样本数: 48328   正例: 4233

===== Fold 1 =====
  原始:  AUC=0.7986 PR-AUC=0.3347 Brier=0.1628
  校准:  AUC=0.7975 PR-AUC=0.3192 F1=0.3510 Recall=0.5039 Prec=0.2692 Spec=0.8615 Brier=0.0723

===== Fold 2 =====
  原始:  AUC=0.7759 PR-AUC=0.2672 Brier=0.1653
  校准:  AUC=0.7754 PR-AUC=0.2584 F1=0.3134 Recall=0.4018 Prec=0.2569 Spec=0.8967 Brier=0.0680

===== Fold 3 =====
  原始:  AUC=0.7879 PR-AUC=0.3110 Brier=0.1563
  校准:  AUC=0.7876 PR-AUC=0.2997 F1=0.3536 Recall=0.4267 Prec=0.3018 Spec=0.9036 Brier=0.0711

===== Fold 4 =====
  原始:  AUC=0.7885 PR-AUC=0.3005 Brier=0.1579
  校准:  AUC=0.7879 PR-AUC=0.2880 F1=0.3514 Recall=0.4822 Prec=0.2764 Spec=0.8795 Brier=0.0701

===== Fold 5 =====
  原始:  AUC=0.7888 PR-AUC=0.3103 Brier=0.1658
  校准:  AUC=0.7893 PR-AUC=0.3010 F1=0.3465 Recall=0.4326 Prec=0.2890 Spec=0.8970 Brier=0.0707

总用时: 0.7 分钟

校准前后对比 (5折均值)
  AUC         : 原始 0.7879±0.0080  校准 0.7875±0.0079
  PR_AUC      : 原始 0.3048±0.0245  校准 0.2933±0.0224
  F1          : 原始 0.3175±0.0169  校准 0.3432±0.01

TypeError: unsupported format string passed to numpy.ndarray.__format__

In [4]:
# ============ threadpoolctl monkey-patch（必须最前）============
import threadpoolctl
class _SafeThreadpoolLimits:
    def __init__(self, *args, **kwargs): pass
    def __enter__(self): return self
    def __exit__(self, *args): pass
threadpoolctl.threadpool_limits = _SafeThreadpoolLimits
# ================================================================

import os
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             brier_score_loss)

out_dir = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"

# ---------- 加载已保存的 OOF ----------
cal = pd.read_csv(os.path.join(out_dir, "Optuna_calibrated_predictions.csv"))
y        = cal["y_true"].values.astype(int)
p_raw    = cal["y_prob_raw"].values.astype(float)        # Optuna 原始
p_cal    = cal["y_prob_calibrated"].values.astype(float) # Isotonic 校准后

cmp = pd.read_csv(os.path.join(out_dir, "对比_OOF_predictions.csv"))
print("对比 OOF 列名:", cmp.columns.tolist())
y_cmp = cmp["y_true"].values.astype(int)
assert np.array_equal(y, y_cmp), "两个 OOF 文件的 y 顺序不一致！"

p_base_lgb = cmp["Base_LightGBM"].values.astype(float)   # 57 特征
p_full_lgb = cmp["Full_LightGBM"].values.astype(float)   # 83 特征（默认参数）
p_base_xgb = cmp["Base_XGBoost"].values.astype(float)
p_full_xgb = cmp["Full_XGBoost"].values.astype(float)
p_base_rf  = cmp["Base_RandomForest"].values.astype(float)
p_full_rf  = cmp["Full_RandomForest"].values.astype(float)

# ================= DeLong（修复版）=================
def compute_midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x)
    T = np.zeros(N, dtype=float); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]: j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float); T2[J] = T
    return T2

def fastDeLong(pred_sorted, label_1_count):
    m = label_1_count
    n = pred_sorted.shape[1] - m
    pos = pred_sorted[:, :m]; neg = pred_sorted[:, m:]
    k = pred_sorted.shape[0]
    tx = np.empty([k, m]); ty = np.empty([k, n]); tz = np.empty([k, m + n])
    for r in range(k):
        tx[r] = compute_midrank(pos[r])
        ty[r] = compute_midrank(neg[r])
        tz[r] = compute_midrank(pred_sorted[r])
    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    sx = np.cov(v01); sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, delongcov

def delong_roc_test(y_true, prob1, prob2):
    y_true = np.asarray(y_true).ravel()
    prob1  = np.asarray(prob1).ravel()
    prob2  = np.asarray(prob2).ravel()
    order = np.argsort(-y_true)
    label_1_count = int(y_true.sum())
    predictions_sorted = np.vstack([prob1, prob2])[:, order]
    aucs, delongcov = fastDeLong(predictions_sorted, label_1_count)
    l = np.array([[1, -1]])
    z = float(np.abs(np.diff(aucs))[0] /
              np.sqrt(np.dot(np.dot(l, delongcov), l.T)).ravel()[0])
    p = float(2 * stats.norm.sf(z))
    return aucs, z, p

# ---------- 逐对做 DeLong ----------
print("\n" + "=" * 80)
print("DeLong 检验结果")
print("=" * 80)

rows = []

def report(name, y_, p1, p2, label1, label2):
    aucs, z, p = delong_roc_test(y_, p1, p2)
    rows.append(dict(Comparison=name,
                     AUC_1=aucs[0], AUC_2=aucs[1],
                     Delta_AUC=aucs[0] - aucs[1],
                     Z=z, p_value=p,
                     Significant=("Yes" if p < 0.05 else "No")))
    print(f"[{name}]")
    print(f"    {label1} AUC = {aucs[0]:.4f}")
    print(f"    {label2} AUC = {aucs[1]:.4f}")
    print(f"    ΔAUC = {aucs[0]-aucs[1]:+.4f}   Z = {z:.4f}   p = {p:.4g}")
    print()

# 1. MD&A 增量：Full vs Base（同模型）
report("LightGBM: Full(83) vs Base(57)",  y, p_full_lgb, p_base_lgb, "Full", "Base")
report("XGBoost : Full(83) vs Base(57)",  y, p_full_xgb, p_base_xgb, "Full", "Base")
report("RF      : Full(83) vs Base(57)",  y, p_full_rf,  p_base_rf,  "Full", "Base")

# 2. 调参增量：Optuna vs 默认（都是 Full LightGBM）
report("Optuna vs Default (Full LGB)",  y, p_raw, p_full_lgb, "Optuna", "Default")

# 3. 校准后 vs 默认
report("Calibrated-Optuna vs Default-LGB", y, p_cal, p_full_lgb, "Cal-Optuna", "Default")

df = pd.DataFrame(rows)
df.to_csv(os.path.join(out_dir, "DeLong_results.csv"),
          index=False, encoding='utf-8-sig')
print("已保存: DeLong_results.csv")

# ================= Bootstrap 95% CI =================
print("\n" + "=" * 80)
print("Bootstrap 95% CI (1000 次) — Optuna 原始 vs 校准")
print("=" * 80)

def bootstrap_ci(y_, p_, metric_fn, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_)
    scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y_[idx])) < 2:
            continue
        scores.append(metric_fn(y_[idx], p_[idx]))
    scores = np.array(scores)
    return (scores.mean(),
            np.percentile(scores, 2.5),
            np.percentile(scores, 97.5))

for label, p in [("Optuna-Raw", p_raw), ("Optuna-Calibrated", p_cal)]:
    print(f"\n[{label}]")
    for mname, fn in [("AUC", roc_auc_score),
                      ("PR-AUC", average_precision_score),
                      ("Brier", brier_score_loss)]:
        m, lo, hi = bootstrap_ci(y, p, fn)
        print(f"    {mname:8s} = {m:.4f}   95% CI [{lo:.4f}, {hi:.4f}]")

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\科研\\二次实验\\二次实验\\二次实验\\国泰安数据下载\\合并结果\\Optuna_calibrated_predictions.csv'

In [5]:
# ============ threadpoolctl monkey-patch（必须最前）============
import threadpoolctl
class _SafeThreadpoolLimits:
    def __init__(self, *args, **kwargs): pass
    def __enter__(self): return self
    def __exit__(self, *args): pass
threadpoolctl.threadpool_limits = _SafeThreadpoolLimits
# ================================================================

import pandas as pd
import numpy as np
import os, time, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              precision_recall_curve, f1_score,
                              recall_score, precision_score,
                              confusion_matrix, brier_score_loss)
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from scipy import stats
import lightgbm as lgb

data_path = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\建模数据集_方案A_MDA_未隔离.csv"
out_dir   = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"

data = pd.read_csv(data_path)
exclude_cols = ["Stkcd", "year", "Fraud", "ShortName", "IndustryName1",
                "ViolationTypeID", "DeclareDate", "DisposalDate", "Enddate", "set"]
all_numeric = [c for c in data.columns
               if c not in exclude_cols and pd.api.types.is_numeric_dtype(data[c])]

y      = data["Fraud"].values
groups = data["Stkcd"].values
X_all  = data[all_numeric].values
print("特征数:", len(all_numeric), "  样本数:", len(y), "  正例:", int(y.sum()))

best_params = dict(
    n_estimators=400, learning_rate=0.015, num_leaves=25,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.75,
    reg_alpha=0.15, reg_lambda=0.2,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
)

def find_best_threshold(y_true, y_prob):
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
    return float(thr[np.argmax(f1s)])

def compute_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return dict(
        AUC=roc_auc_score(y_true, y_prob),
        PR_AUC=average_precision_score(y_true, y_prob),
        F1=f1_score(y_true, y_pred, zero_division=0),
        Recall=recall_score(y_true, y_pred, zero_division=0),
        Precision=precision_score(y_true, y_pred, zero_division=0),
        Specificity=tn / (tn + fp) if (tn + fp) > 0 else np.nan,
        Brier=brier_score_loss(y_true, y_prob),
    )

# ================= 主循环 =================
gkf = GroupKFold(n_splits=5)
probs_raw = np.zeros(len(data))
probs_cal = np.zeros(len(data))
metrics_raw, metrics_cal = [], []

t0 = time.time()
for fold, (tr, te) in enumerate(gkf.split(X_all, y, groups), 1):
    X_tr_raw, X_te_raw = X_all[tr], X_all[te]
    y_tr, y_te = y[tr], y[te]
    g_tr = groups[tr]

    imp = SimpleImputer(strategy='median')
    X_tr = imp.fit_transform(X_tr_raw)
    X_te = imp.transform(X_te_raw)

    # ---- 原始概率 ----
    m = lgb.LGBMClassifier(**best_params)
    m.fit(X_tr, y_tr)
    p_raw = m.predict_proba(X_te)[:, 1]
    probs_raw[te] = p_raw

    # ---- Isotonic 校准（内层 3 折 OOF 拟合）----
    inner_gkf = GroupKFold(n_splits=3)
    oof_prob = np.zeros(len(tr))
    for i_tr, i_val in inner_gkf.split(X_tr, y_tr, g_tr):
        imp_i = SimpleImputer(strategy='median')
        X_i = imp_i.fit_transform(X_tr[i_tr])
        X_v = imp_i.transform(X_tr[i_val])
        mi = lgb.LGBMClassifier(**best_params)
        mi.fit(X_i, y_tr[i_tr])
        oof_prob[i_val] = mi.predict_proba(X_v)[:, 1]

    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(oof_prob, y_tr)
    p_cal = iso.predict(p_raw)
    probs_cal[te] = p_cal

    oof_cal = iso.predict(oof_prob)
    thr = find_best_threshold(y_tr, oof_cal)

    met_raw = compute_metrics(y_te, p_raw, 0.5)
    met_cal = compute_metrics(y_te, p_cal, thr)
    met_raw['fold'] = fold; met_cal['fold'] = fold
    metrics_raw.append(met_raw); metrics_cal.append(met_cal)
    print(f"Fold {fold}: raw AUC={met_raw['AUC']:.4f} Brier={met_raw['Brier']:.4f} | "
          f"cal AUC={met_cal['AUC']:.4f} Brier={met_cal['Brier']:.4f}")

print(f"\n总用时: {(time.time()-t0)/60:.2f} 分钟")

# ================= 【关键】立即保存，避免后面报错丢数据 =================
df_raw = pd.DataFrame(metrics_raw)
df_cal = pd.DataFrame(metrics_cal)

df_raw.to_csv(os.path.join(out_dir, "Optuna_calibration_raw.csv"),
              index=False, encoding='utf-8-sig')
df_cal.to_csv(os.path.join(out_dir, "Optuna_calibration_calibrated.csv"),
              index=False, encoding='utf-8-sig')
pd.DataFrame({
    'y_true': y,
    'y_prob_raw': probs_raw,
    'y_prob_calibrated': probs_cal
}).to_csv(os.path.join(out_dir, "Optuna_calibrated_predictions.csv"),
          index=False, encoding='utf-8-sig')

print("\n[已保存 3 个文件]")
print("  Optuna_calibration_raw.csv")
print("  Optuna_calibration_calibrated.csv")
print("  Optuna_calibrated_predictions.csv")

# ================= DeLong（修复版，放在保存之后）=================
def compute_midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x)
    T = np.zeros(N, dtype=float); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]: j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float); T2[J] = T
    return T2

def fastDeLong(pred_sorted, label_1_count):
    m = label_1_count
    n = pred_sorted.shape[1] - m
    pos = pred_sorted[:, :m]; neg = pred_sorted[:, m:]
    k = pred_sorted.shape[0]
    tx = np.empty([k, m]); ty = np.empty([k, n]); tz = np.empty([k, m + n])
    for r in range(k):
        tx[r] = compute_midrank(pos[r])
        ty[r] = compute_midrank(neg[r])
        tz[r] = compute_midrank(pred_sorted[r])
    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    sx = np.cov(v01); sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, delongcov

def delong_roc_test(y_true, prob1, prob2):
    y_true = np.asarray(y_true).ravel()
    prob1  = np.asarray(prob1).ravel()
    prob2  = np.asarray(prob2).ravel()
    order = np.argsort(-y_true)
    label_1_count = int(y_true.sum())
    pred_sorted = np.vstack([prob1, prob2])[:, order]
    aucs, delongcov = fastDeLong(pred_sorted, label_1_count)
    l = np.array([[1, -1]])
    z = float(np.abs(np.diff(aucs))[0] /
              np.sqrt(np.dot(np.dot(l, delongcov), l.T)).ravel()[0])
    p = float(2 * stats.norm.sf(z))
    return aucs, z, p

# ---------- 读对比 OOF ----------
cmp = pd.read_csv(os.path.join(out_dir, "对比_OOF_predictions.csv"))
print("\n对比 OOF 列名:", cmp.columns.tolist())

y_cmp = cmp["y_true"].values.astype(int) if "y_true" in cmp.columns else cmp["Fraud"].values.astype(int)
assert np.array_equal(y, y_cmp), "两个 OOF 的 y 顺序不一致，需先对齐索引！"

def get(cands):
    for c in cands:
        if c in cmp.columns: return cmp[c].values.astype(float)
    raise KeyError(f"没找到 {cands}，请把上面的列名贴回来")

p_base_lgb = get(["Base_LightGBM", "y_prob_Base_LightGBM"])
p_full_lgb = get(["Full_LightGBM", "y_prob_Full_LightGBM"])
p_base_xgb = get(["Base_XGBoost",  "y_prob_Base_XGBoost"])
p_full_xgb = get(["Full_XGBoost",  "y_prob_Full_XGBoost"])
p_base_rf  = get(["Base_RandomForest", "y_prob_Base_RandomForest"])
p_full_rf  = get(["Full_RandomForest", "y_prob_Full_RandomForest"])

# ---------- 逐对 DeLong ----------
print("\n" + "=" * 80)
print("DeLong 检验")
print("=" * 80)
rows = []
def report(name, y_, p1, p2, l1, l2):
    aucs, z, p = delong_roc_test(y_, p1, p2)
    rows.append(dict(Comparison=name,
                     AUC_1=aucs[0], AUC_2=aucs[1],
                     Delta_AUC=aucs[0]-aucs[1], Z=z, p_value=p,
                     Significant=("Yes" if p < 0.05 else "No")))
    print(f"[{name}]")
    print(f"    {l1} AUC = {aucs[0]:.4f}")
    print(f"    {l2} AUC = {aucs[1]:.4f}")
    print(f"    ΔAUC = {aucs[0]-aucs[1]:+.4f}   Z = {z:.4f}   p = {p:.4g}\n")

report("LGB: Full(83) vs Base(57)",   y, p_full_lgb, p_base_lgb, "Full", "Base")
report("XGB: Full(83) vs Base(57)",   y, p_full_xgb, p_base_xgb, "Full", "Base")
report("RF : Full(83) vs Base(57)",   y, p_full_rf,  p_base_rf,  "Full", "Base")
report("Optuna vs Default (Full LGB)", y, probs_raw, p_full_lgb, "Optuna", "Default")
report("Cal-Optuna vs Default-LGB",    y, probs_cal, p_full_lgb, "Cal-Optuna", "Default")

pd.DataFrame(rows).to_csv(os.path.join(out_dir, "DeLong_results.csv"),
                          index=False, encoding='utf-8-sig')
print("已保存: DeLong_results.csv")

# ================= Bootstrap 95% CI =================
print("\n" + "=" * 80)
print("Bootstrap 95% CI (1000 次)")
print("=" * 80)
def bootstrap_ci(y_, p_, fn, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_); scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y_[idx])) < 2: continue
        scores.append(fn(y_[idx], p_[idx]))
    scores = np.array(scores)
    return scores.mean(), np.percentile(scores, 2.5), np.percentile(scores, 97.5)

for label, p in [("Optuna-Raw", probs_raw), ("Optuna-Calibrated", probs_cal)]:
    print(f"\n[{label}]")
    for mname, fn in [("AUC", roc_auc_score),
                      ("PR-AUC", average_precision_score),
                      ("Brier", brier_score_loss)]:
        m, lo, hi = bootstrap_ci(y, p, fn)
        print(f"    {mname:8s} = {m:.4f}   95% CI [{lo:.4f}, {hi:.4f}]")

特征数: 83   样本数: 48328   正例: 4233
Fold 1: raw AUC=0.7986 Brier=0.1628 | cal AUC=0.7975 Brier=0.0723
Fold 2: raw AUC=0.7759 Brier=0.1653 | cal AUC=0.7754 Brier=0.0680
Fold 3: raw AUC=0.7879 Brier=0.1563 | cal AUC=0.7876 Brier=0.0711
Fold 4: raw AUC=0.7885 Brier=0.1579 | cal AUC=0.7879 Brier=0.0701
Fold 5: raw AUC=0.7888 Brier=0.1658 | cal AUC=0.7893 Brier=0.0707

总用时: 0.77 分钟

[已保存 3 个文件]
  Optuna_calibration_raw.csv
  Optuna_calibration_calibrated.csv
  Optuna_calibrated_predictions.csv

对比 OOF 列名: ['y_true', 'Base_LightGBM', 'Base_XGBoost', 'Base_RandomForest', 'Full_LightGBM', 'Full_XGBoost', 'Full_RandomForest']

DeLong 检验
[LGB: Full(83) vs Base(57)]
    Full AUC = 0.7753
    Base AUC = 0.7747
    ΔAUC = +0.0006   Z = 0.3292   p = 0.742

[XGB: Full(83) vs Base(57)]
    Full AUC = 0.7679
    Base AUC = 0.7656
    ΔAUC = +0.0023   Z = 1.1451   p = 0.2522

[RF : Full(83) vs Base(57)]
    Full AUC = 0.7819
    Base AUC = 0.7810
    ΔAUC = +0.0009   Z = 0.8307   p = 0.4062

[Optuna vs Defa

In [6]:
pip install shap

  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.4
    Uninstalling numpy-1.24.4:
      Successfully uninstalled numpy-1.24.4
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] 拒绝访问。: 'D:\\aca\\Lib\\site-packages\\~-mpy\\.libs\\libopenblas64__v0.3.21-gcc_10_3_0.dll'
Consider using the `--user` option or check the permissions.



In [7]:
# ============ threadpoolctl monkey-patch ============
import threadpoolctl
class _SafeThreadpoolLimits:
    def __init__(self, *args, **kwargs): pass
    def __enter__(self): return self
    def __exit__(self, *args): pass
threadpoolctl.threadpool_limits = _SafeThreadpoolLimits
# =====================================================

import os, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
import lightgbm as lgb

data_path = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\建模数据集_方案A_MDA_未隔离.csv"
out_dir   = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"

data = pd.read_csv(data_path)
exclude_cols = ["Stkcd", "year", "Fraud", "ShortName", "IndustryName1",
                "ViolationTypeID", "DeclareDate", "DisposalDate", "Enddate", "set"]
all_numeric = [c for c in data.columns
               if c not in exclude_cols and pd.api.types.is_numeric_dtype(data[c])]

y      = data["Fraud"].values
groups = data["Stkcd"].values
X_all  = data[all_numeric].values
print(f"特征数: {len(all_numeric)}   样本数: {len(y)}   正例: {int(y.sum())}")

# ---------- MD&A 特征清单 ----------
MDA_FEATURES = [
    "TextualSimilarity", "PositiveVocabularyNum", "NegativeVocabularyNum",
    "EmotionTone1", "EmotionTone2",
    "PosRatio", "NegRatio", "SentLenAvg", "SentLenStd", "ComplexWordRatio",
    "DigitDensity", "PuncDensity", "TTR",
    "Jaccard_prev", "EditSim_prev", "TFIDF_Cosine_prev",
    "DLUT_PosNum", "DLUT_NegNum", "DLUT_PosRatio", "DLUT_NegRatio",
    "DLUT_PosIntensity", "DLUT_NegIntensity", "DLUT_EmotionScore",
    "DLUT_EmotionTone", "DLUT_NegAfterNeg", "DLUT_PosAfterNeg",
]
MDA_FEATURES = [f for f in MDA_FEATURES if f in all_numeric]
print(f"MD&A 特征数: {len(MDA_FEATURES)}")

best_params = dict(
    n_estimators=400, learning_rate=0.015, num_leaves=25,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.75,
    reg_alpha=0.15, reg_lambda=0.2,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
)

# ================= 5 折：训练 + pred_contrib=True =================
gkf = GroupKFold(n_splits=5)
n_feat = len(all_numeric)
shap_all = np.zeros((len(y), n_feat))     # 只存特征的 SHAP 值（不含 base value）
base_values = np.zeros(len(y))
oof_prob = np.zeros(len(y))
fold_ids = np.zeros(len(y), dtype=int)
fold_shap_importance = []

t0 = time.time()
for fold, (tr, te) in enumerate(gkf.split(X_all, y, groups), 1):
    print(f"\n===== Fold {fold} =====")
    X_tr_raw, X_te_raw = X_all[tr], X_all[te]
    y_tr, y_te = y[tr], y[te]

    imp = SimpleImputer(strategy='median')
    X_tr = imp.fit_transform(X_tr_raw)
    X_te = imp.transform(X_te_raw)

    m = lgb.LGBMClassifier(**best_params)
    m.fit(X_tr, y_tr)
    oof_prob[te] = m.predict_proba(X_te)[:, 1]
    fold_ids[te] = fold

    # ---------- LightGBM 内置 SHAP ----------
    # 返回 shape = (n_samples, n_features + 1)，最后一列是 base value
    contrib = m.predict(X_te, pred_contrib=True, raw_score=True)
    # 注：raw_score=True 时贡献是 log-odds 空间，求和 + base 才是 log-odds 预测
    # 如果不用 raw_score，返回的是概率空间的贡献，更直观
    contrib_prob = m.predict(X_te, pred_contrib=True)   # 概率空间

    shap_all[te] = contrib_prob[:, :-1]
    base_values[te] = contrib_prob[:, -1]

    imp_fold = pd.Series(np.abs(shap_all[te]).mean(axis=0),
                         index=all_numeric).sort_values(ascending=False)
    fold_shap_importance.append(imp_fold)
    print(f"  Fold {fold} Top5: {imp_fold.head(5).index.tolist()}")
    print(f"  耗时: {(time.time()-t0)/60:.2f} min")

print(f"\n总耗时: {(time.time()-t0)/60:.2f} 分钟")

# ================= 全局重要性 =================
mean_abs_shap = pd.Series(np.abs(shap_all).mean(axis=0), index=all_numeric).sort_values(ascending=False)
mean_signed_shap = pd.Series(shap_all.mean(axis=0), index=all_numeric).sort_values(ascending=False)

importance_df = pd.DataFrame({
    "Feature": mean_abs_shap.index,
    "MeanAbsSHAP": mean_abs_shap.values,
    "MeanSignedSHAP": [mean_signed_shap[f] for f in mean_abs_shap.index],
    "IsMDA": [f in MDA_FEATURES for f in mean_abs_shap.index],
})
importance_df["Rank"] = range(1, len(importance_df) + 1)
importance_df.to_csv(os.path.join(out_dir, "SHAP_global_importance.csv"),
                     index=False, encoding='utf-8-sig')

print("\n" + "=" * 80)
print("SHAP 全局重要性 Top 20")
print("=" * 80)
print(importance_df.head(20).to_string(index=False))

# ================= MD&A 特征排名 =================
mda_imp = importance_df[importance_df["IsMDA"]]
print("\n" + "=" * 80)
print("MD&A 特征 SHAP 重要性")
print("=" * 80)
print(mda_imp.to_string(index=False))
mda_imp.to_csv(os.path.join(out_dir, "SHAP_MDA_importance.csv"),
               index=False, encoding='utf-8-sig')

# ================= 5 折稳定性 =================
stab = pd.DataFrame({f"fold{i+1}": fold_shap_importance[i].reindex(all_numeric)
                     for i in range(5)})
stab["mean"] = stab.mean(axis=1)
stab["std"]  = stab.std(axis=1)
stab["rank_mean"] = stab["mean"].rank(ascending=False)
stab = stab.sort_values("mean", ascending=False)
stab.to_csv(os.path.join(out_dir, "SHAP_stability_5fold.csv"), encoding='utf-8-sig')

print("\n" + "=" * 80)
print("5 折 SHAP 稳定性 Top 15")
print("=" * 80)
print(stab.head(15)[["mean", "std", "rank_mean"]].to_string())

# ================= 保存原始 SHAP 值 =================
shap_df = pd.DataFrame(shap_all, columns=all_numeric)
shap_df.insert(0, "y_true", y)
shap_df.insert(1, "y_prob", oof_prob)
shap_df.insert(2, "fold", fold_ids)
shap_df.insert(3, "base_value", base_values)
shap_df.to_csv(os.path.join(out_dir, "SHAP_values_all.csv"),
               index=False, encoding='utf-8-sig')

X_df = pd.DataFrame(X_all, columns=all_numeric)
X_df.insert(0, "y_true", y)
X_df.insert(1, "y_prob", oof_prob)
X_df.to_csv(os.path.join(out_dir, "SHAP_feature_values.csv"),
            index=False, encoding='utf-8-sig')

print("\n已保存:")
print("  SHAP_global_importance.csv")
print("  SHAP_MDA_importance.csv")
print("  SHAP_stability_5fold.csv")
print("  SHAP_values_all.csv")
print("  SHAP_feature_values.csv")

# ================= 漏报 vs 命中样本 =================
missed = np.where((y == 1) & (oof_prob < 0.3))[0]
caught = np.where((y == 1) & (oof_prob > 0.7))[0]
print(f"\n漏报样本数 (y=1 & prob<0.3): {len(missed)}")
print(f"命中样本数 (y=1 & prob>0.7): {len(caught)}")

if len(missed) > 0:
    missed_top = pd.Series(np.abs(shap_all[missed]).mean(axis=0),
                           index=all_numeric).sort_values(ascending=False).head(10)
    print("\n漏报样本的 SHAP Top 10:")
    print(missed_top.to_string())

if len(caught) > 0:
    caught_top = pd.Series(np.abs(shap_all[caught]).mean(axis=0),
                           index=all_numeric).sort_values(ascending=False).head(10)
    print("\n命中样本的 SHAP Top 10:")
    print(caught_top.to_string())

# ================= 简单一致性检查 =================
# SHAP 值之和 + base value 应该 ≈ log-odds 预测（用于 raw_score=True）
# 这里我们用的是概率空间，只做 sanity check
print("\n一致性检查 (前 5 个样本):")
print(f"  base_value[:5] = {base_values[:5]}")
print(f"  sum(SHAP[:5]) + base = {(shap_all[:5].sum(axis=1) + base_values[:5])}")
print(f"  oof_prob[:5] = {oof_prob[:5]}")

特征数: 83   样本数: 48328   正例: 4233
MD&A 特征数: 26

===== Fold 1 =====
  Fold 1 Top5: ['F050201B', 'LargestHolderRate', 'F050301B', 'F010701B', 'TopTenHoldersRate']
  耗时: 0.21 min

===== Fold 2 =====
  Fold 2 Top5: ['TopTenHoldersRate', 'F050201B', 'LargestHolderRate', 'F010701B', 'F050301B']
  耗时: 0.43 min

===== Fold 3 =====
  Fold 3 Top5: ['F050301B', 'TopTenHoldersRate', 'F050201B', 'F041701B', 'F010701B']
  耗时: 0.65 min

===== Fold 4 =====
  Fold 4 Top5: ['F010701B', 'LargestHolderRate', 'TopTenHoldersRate', 'F081601B', 'F070101B']
  耗时: 0.88 min

===== Fold 5 =====
  Fold 5 Top5: ['TopTenHoldersRate', 'F050201B', 'F050301B', 'F010701B', 'TextualSimilarity']
  耗时: 1.10 min

总耗时: 1.10 分钟

SHAP 全局重要性 Top 20
           Feature  MeanAbsSHAP  MeanSignedSHAP  IsMDA  Rank
          F050201B     0.148788       -0.003579  False     1
 TopTenHoldersRate     0.148717        0.015947  False     2
          F010701B     0.137821       -0.016847  False     3
          F050301B     0.128833        0.0

In [ ]:
# ============ threadpoolctl monkey-patch（必须最前）============
import threadpoolctl
class _SafeThreadpoolLimits:
    def __init__(self, *args, **kwargs): pass
    def __enter__(self): return self
    def __exit__(self, *args): pass
threadpoolctl.threadpool_limits = _SafeThreadpoolLimits
# ================================================================

import os, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              precision_recall_curve, f1_score,
                              recall_score, precision_score,
                              confusion_matrix, brier_score_loss)
from sklearn.isotonic import IsotonicRegression
from imblearn.over_sampling import SMOTE
from scipy import stats
import lightgbm as lgb

data_path = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\建模数据集_方案A_MDA_未隔离.csv"
out_dir   = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"

# ================= 加载数据 =================
data = pd.read_csv(data_path)
exclude_cols = ["Stkcd", "year", "Fraud", "ShortName", "IndustryName1",
                "ViolationTypeID", "DeclareDate", "DisposalDate", "Enddate", "set"]
all_numeric = [c for c in data.columns
               if c not in exclude_cols and pd.api.types.is_numeric_dtype(data[c])]

y      = data["Fraud"].values
groups = data["Stkcd"].values
X_all  = data[all_numeric].values

MDA_FEATURES = [
    "TextualSimilarity", "PositiveVocabularyNum", "NegativeVocabularyNum",
    "EmotionTone1", "EmotionTone2",
    "PosRatio", "NegRatio", "SentLenAvg", "SentLenStd", "ComplexWordRatio",
    "DigitDensity", "PuncDensity", "TTR",
    "Jaccard_prev", "EditSim_prev", "TFIDF_Cosine_prev",
    "DLUT_PosNum", "DLUT_NegNum", "DLUT_PosRatio", "DLUT_NegRatio",
    "DLUT_PosIntensity", "DLUT_NegIntensity", "DLUT_EmotionScore",
    "DLUT_EmotionTone", "DLUT_NegAfterNeg", "DLUT_PosAfterNeg",
]
MDA_FEATURES = [f for f in MDA_FEATURES if f in all_numeric]

print(f"特征数: {len(all_numeric)}   样本数: {len(y)}   正例: {int(y.sum())}")
print(f"MD&A 特征数: {len(MDA_FEATURES)}")

# ================= 固定参数（沿用之前 Optuna 的中庸参数）=================
# 注意：不加 class_weight='balanced'，因为 SMOTE 已经平衡了训练集
best_params = dict(
    n_estimators=400, learning_rate=0.015, num_leaves=25,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.75,
    reg_alpha=0.15, reg_lambda=0.2,
    random_state=42, n_jobs=-1, verbose=-1
)

def find_best_threshold(y_true, y_prob):
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
    return float(thr[np.argmax(f1s)])

def compute_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return dict(
        AUC=roc_auc_score(y_true, y_prob),
        PR_AUC=average_precision_score(y_true, y_prob),
        F1=f1_score(y_true, y_pred, zero_division=0),
        Recall=recall_score(y_true, y_pred, zero_division=0),
        Precision=precision_score(y_true, y_pred, zero_division=0),
        Specificity=tn / (tn + fp) if (tn + fp) > 0 else np.nan,
        Brier=brier_score_loss(y_true, y_prob),
    )

# ================= SMOTE 5 折主实验 =================
SMOTE_RATIO = 'auto'   # 'auto' = 1:1；可以改成 0.5 (1:2) 做保守版
SMOTE_K = 5

gkf = GroupKFold(n_splits=5)
probs_raw = np.zeros(len(y))   # SMOTE 训练后的原始概率
probs_cal = np.zeros(len(y))   # 校准后
shap_all = np.zeros((len(y), len(all_numeric)))
fold_ids = np.zeros(len(y), dtype=int)
metrics_raw, metrics_cal = [], []
fold_info = []

t0 = time.time()
for fold, (tr, te) in enumerate(gkf.split(X_all, y, groups), 1):
    print(f"\n===== Fold {fold} =====")
    X_tr_raw, X_te_raw = X_all[tr], X_all[te]
    y_tr, y_te = y[tr], y[te]
    g_tr = groups[tr]

    # ---- 1. 先 impute（SMOTE 不能处理 NaN）----
    imp = SimpleImputer(strategy='median')
    X_tr = imp.fit_transform(X_tr_raw)
    X_te = imp.transform(X_te_raw)

    # ---- 2. SMOTE（只在训练集）----
    smote = SMOTE(sampling_strategy=SMOTE_RATIO, k_neighbors=SMOTE_K,
                  random_state=42, n_jobs=-1)
    X_tr_res, y_tr_res = smote.fit_resample(X_tr, y_tr)
    print(f"  SMOTE: {X_tr.shape[0]} -> {X_tr_res.shape[0]}  "
          f"(pos: {y_tr.sum()} -> {y_tr_res.sum()})")

    # ---- 3. 训练模型 ----
    m = lgb.LGBMClassifier(**best_params)
    m.fit(X_tr_res, y_tr_res)
    p_raw = m.predict_proba(X_te)[:, 1]
    probs_raw[te] = p_raw
    fold_ids[te] = fold

    # ---- 4. 内层 3 折 OOF，用于校准 ----
    inner_gkf = GroupKFold(n_splits=3)
    oof_prob = np.zeros(len(tr))
    for i_tr, i_val in inner_gkf.split(X_tr, y_tr, g_tr):
        imp_i = SimpleImputer(strategy='median')
        X_i = imp_i.fit_transform(X_tr[i_tr])
        X_v = imp_i.transform(X_tr[i_val])
        sm_i = SMOTE(sampling_strategy=SMOTE_RATIO, k_neighbors=SMOTE_K,
                     random_state=42, n_jobs=-1)
        X_i_res, y_i_res = sm_i.fit_resample(X_i, y_tr[i_tr])
        mi = lgb.LGBMClassifier(**best_params)
        mi.fit(X_i_res, y_i_res)
        oof_prob[i_val] = mi.predict_proba(X_v)[:, 1]

    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(oof_prob, y_tr)
    p_cal = iso.predict(p_raw)
    probs_cal[te] = p_cal

    # ---- 5. 阈值：内层 OOF 校准后 F1 最优 ----
    oof_cal = iso.predict(oof_prob)
    thr = find_best_threshold(y_tr, oof_cal)

    # ---- 6. 指标 ----
    met_raw = compute_metrics(y_te, p_raw, 0.5)
    met_cal = compute_metrics(y_te, p_cal, thr)
    met_raw['fold'] = fold; met_cal['fold'] = fold
    met_raw['thr'] = 0.5; met_cal['thr'] = thr
    metrics_raw.append(met_raw); metrics_cal.append(met_cal)
    fold_info.append(dict(fold=fold, thr=thr,
                          train_orig=len(tr), train_smote=len(X_tr_res),
                          test=len(te), test_pos=int(y_te.sum())))

    print(f"  thr={thr:.3f}")
    print(f"  raw  AUC={met_raw['AUC']:.4f} PR={met_raw['PR_AUC']:.4f} Brier={met_raw['Brier']:.4f}")
    print(f"  cal  AUC={met_cal['AUC']:.4f} PR={met_cal['PR_AUC']:.4f} "
          f"F1={met_cal['F1']:.4f} R={met_cal['Recall']:.4f} "
          f"P={met_cal['Precision']:.4f} Spec={met_cal['Specificity']:.4f} "
          f"Brier={met_cal['Brier']:.4f}")
    print(f"  耗时: {(time.time()-t0)/60:.2f} min")

print(f"\n总耗时: {(time.time()-t0)/60:.2f} 分钟")

# ================= 5 折汇总 =================
df_raw = pd.DataFrame(metrics_raw)
df_cal = pd.DataFrame(metrics_cal)
df_fold = pd.DataFrame(fold_info)

print("\n" + "=" * 80)
print("SMOTE 5 折均值")
print("=" * 80)
for col in ['AUC','PR_AUC','F1','Recall','Precision','Specificity','Brier']:
    print(f"  {col:12s}: raw {df_raw[col].mean():.4f}±{df_raw[col].std():.4f}  "
          f"cal {df_cal[col].mean():.4f}±{df_cal[col].std():.4f}")

df_raw.to_csv(os.path.join(out_dir, "SMOTE_calibration_raw.csv"),
              index=False, encoding='utf-8-sig')
df_cal.to_csv(os.path.join(out_dir, "SMOTE_calibration_calibrated.csv"),
              index=False, encoding='utf-8-sig')
df_fold.to_csv(os.path.join(out_dir, "SMOTE_fold_info.csv"),
               index=False, encoding='utf-8-sig')
pd.DataFrame({
    'y_true': y,
    'y_prob_raw': probs_raw,
    'y_prob_calibrated': probs_cal,
    'fold': fold_ids
}).to_csv(os.path.join(out_dir, "SMOTE_OOF_predictions.csv"),
          index=False, encoding='utf-8-sig')

print("\n[已保存 SMOTE 主实验 4 个文件]")

# ================= DeLong 对比 SMOTE vs class_weight =================
def compute_midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x)
    T = np.zeros(N, dtype=float); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]: j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float); T2[J] = T
    return T2

def fastDeLong(pred_sorted, label_1_count):
    m = label_1_count; n = pred_sorted.shape[1] - m
    pos = pred_sorted[:, :m]; neg = pred_sorted[:, m:]
    k = pred_sorted.shape[0]
    tx = np.empty([k, m]); ty = np.empty([k, n]); tz = np.empty([k, m + n])
    for r in range(k):
        tx[r] = compute_midrank(pos[r])
        ty[r] = compute_midrank(neg[r])
        tz[r] = compute_midrank(pred_sorted[r])
    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    sx = np.cov(v01); sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, delongcov

def delong_roc_test(y_true, prob1, prob2):
    y_true = np.asarray(y_true).ravel()
    prob1 = np.asarray(prob1).ravel()
    prob2 = np.asarray(prob2).ravel()
    order = np.argsort(-y_true)
    label_1_count = int(y_true.sum())
    pred_sorted = np.vstack([prob1, prob2])[:, order]
    aucs, delongcov = fastDeLong(pred_sorted, label_1_count)
    l = np.array([[1, -1]])
    z = float(np.abs(np.diff(aucs))[0] /
              np.sqrt(np.dot(np.dot(l, delongcov), l.T)).ravel()[0])
    p = float(2 * stats.norm.sf(z))
    return aucs, z, p

print("\n" + "=" * 80)
print("DeLong：SMOTE vs class_weight='balanced'")
print("=" * 80)

# 加载 class_weight 版本的 Optuna OOF
cw = pd.read_csv(os.path.join(out_dir, "Optuna_calibrated_predictions.csv"))
y_cw = cw["y_true"].values.astype(int)
assert np.array_equal(y_cw, y), "OOF 索引不一致"
p_cw_raw = cw["y_prob_raw"].values.astype(float)
p_cw_cal = cw["y_prob_calibrated"].values.astype(float)

rows = []
def report(name, y_, p1, p2, l1, l2):
    aucs, z, p = delong_roc_test(y_, p1, p2)
    rows.append(dict(Comparison=name,
                     AUC_1=aucs[0], AUC_2=aucs[1],
                     Delta_AUC=aucs[0]-aucs[1], Z=z, p_value=p,
                     Significant=("Yes" if p < 0.05 else "No")))
    print(f"[{name}]  {l1}={aucs[0]:.4f}  {l2}={aucs[1]:.4f}  "
          f"Δ={aucs[0]-aucs[1]:+.4f}  Z={z:.3f}  p={p:.4g}")

report("Raw: SMOTE vs class_weight",
       y, probs_raw, p_cw_raw, "SMOTE", "cw")
report("Calibrated: SMOTE vs class_weight",
       y, probs_cal, p_cw_cal, "SMOTE_cal", "cw_cal")

pd.DataFrame(rows).to_csv(os.path.join(out_dir, "DeLong_SMOTE_vs_classweight.csv"),
                          index=False, encoding='utf-8-sig')

# ================= Bootstrap 95% CI =================
print("\n" + "=" * 80)
print("Bootstrap 95% CI (1000 次) — SMOTE")
print("=" * 80)
def bootstrap_ci(y_, p_, fn, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_); scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y_[idx])) < 2: continue
        scores.append(fn(y_[idx], p_[idx]))
    scores = np.array(scores)
    return scores.mean(), np.percentile(scores, 2.5), np.percentile(scores, 97.5)

for label, p in [("SMOTE-Raw", probs_raw), ("SMOTE-Calibrated", probs_cal)]:
    print(f"\n[{label}]")
    for mname, fn in [("AUC", roc_auc_score),
                      ("PR-AUC", average_precision_score),
                      ("Brier", brier_score_loss)]:
        m, lo, hi = bootstrap_ci(y, p, fn)
        print(f"    {mname:8s} = {m:.4f}   95% CI [{lo:.4f}, {hi:.4f}]")

# ================= SMOTE 版本 SHAP =================
print("\n" + "=" * 80)
print("SMOTE 版本 SHAP 分析")
print("=" * 80)

# 复用上面的 5 折循环，但这次保存 SHAP
shap_all = np.zeros((len(y), len(all_numeric)))
gkf2 = GroupKFold(n_splits=5)
for fold, (tr, te) in enumerate(gkf2.split(X_all, y, groups), 1):
    X_tr_raw, X_te_raw = X_all[tr], X_all[te]
    y_tr, y_te = y[tr], y[te]

    imp = SimpleImputer(strategy='median')
    X_tr = imp.fit_transform(X_tr_raw)
    X_te = imp.transform(X_te_raw)

    smote = SMOTE(sampling_strategy=SMOTE_RATIO, k_neighbors=SMOTE_K,
                  random_state=42, n_jobs=-1)
    X_tr_res, y_tr_res = smote.fit_resample(X_tr, y_tr)

    m = lgb.LGBMClassifier(**best_params)
    m.fit(X_tr_res, y_tr_res)

    contrib = m.predict(X_te, pred_contrib=True)
    shap_all[te] = contrib[:, :-1]
    print(f"  Fold {fold} SHAP done")

mean_abs_shap = pd.Series(np.abs(shap_all).mean(axis=0), index=all_numeric).sort_values(ascending=False)
mean_signed_shap = pd.Series(shap_all.mean(axis=0), index=all_numeric)

importance_df = pd.DataFrame({
    "Feature": mean_abs_shap.index,
    "MeanAbsSHAP": mean_abs_shap.values,
    "MeanSignedSHAP": [mean_signed_shap[f] for f in mean_abs_shap.index],
    "IsMDA": [f in MDA_FEATURES for f in mean_abs_shap.index],
})
importance_df["Rank"] = range(1, len(importance_df) + 1)
importance_df.to_csv(os.path.join(out_dir, "SMOTE_SHAP_global_importance.csv"),
                     index=False, encoding='utf-8-sig')

print("\nSMOTE SHAP 全局 Top 15:")
print(importance_df.head(15).to_string(index=False))

mda_imp = importance_df[importance_df["IsMDA"]]
print("\nSMOTE MD&A 特征 SHAP 重要性:")
print(mda_imp.to_string(index=False))
mda_imp.to_csv(os.path.join(out_dir, "SMOTE_SHAP_MDA_importance.csv"),
               index=False, encoding='utf-8-sig')

# 保存 SHAP 值
shap_df = pd.DataFrame(shap_all, columns=all_numeric)
shap_df.insert(0, "y_true", y)
shap_df.insert(1, "y_prob", probs_raw)
shap_df.to_csv(os.path.join(out_dir, "SMOTE_SHAP_values_all.csv"),
               index=False, encoding='utf-8-sig')

print("\n全部完成。输出文件:")
print("  SMOTE_calibration_raw.csv")
print("  SMOTE_calibration_calibrated.csv")
print("  SMOTE_fold_info.csv")
print("  SMOTE_OOF_predictions.csv")
print("  DeLong_SMOTE_vs_classweight.csv")
print("  SMOTE_SHAP_global_importance.csv")
print("  SMOTE_SHAP_MDA_importance.csv")
print("  SMOTE_SHAP_values_all.csv")

In [1]:
# ============ threadpoolctl monkey-patch ============
import threadpoolctl
class _SafeThreadpoolLimits:
    def __init__(self, *a, **k): pass
    def __enter__(self): return self
    def __exit__(self, *a): pass
threadpoolctl.threadpool_limits = _SafeThreadpoolLimits
# =====================================================

import pandas as pd
BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"

# 1. 财务+非财务面板
f = pd.read_csv(BASE + r"\财务_非财务_合并面板.csv")
print("=" * 70)
print("【1】财务+非财务面板")
print("=" * 70)
print("年份范围:", f["year"].min(), "→", f["year"].max())
print("\n每年行数:")
print(f.groupby("year").size().rename("n").to_string())

# 2. 违规标签
try:
    v = pd.read_csv(BASE + r"\违规标签_公司年度.csv")
    print("\n" + "=" * 70)
    print("【2】违规标签_公司年度")
    print("=" * 70)
    print("列名:", v.columns.tolist())
    print("年份范围:", v["year"].min(), "→", v["year"].max())
    print("\n每年 Fraud 分布:")
    print(v.groupby("year")["Fraud"].agg(["size", "sum", "mean"]).round(4).to_string())
except Exception as e:
    print("读取违规标签失败:", e)

# 3. 四类违规展开
try:
    t = pd.read_csv(BASE + r"\违规目标四类_公司年度类型.csv")
    print("\n" + "=" * 70)
    print("【3】违规目标四类_公司年度类型")
    print("=" * 70)
    print("列名:", t.columns.tolist())
    if "year" in t.columns:
        print("年份范围:", t["year"].min(), "→", t["year"].max())
        print("\n每年记录数:")
        print(t.groupby("year").size().to_string())
except Exception as e:
    print("读取四类违规失败:", e)

# 4. 建模数据集
m = pd.read_csv(BASE + r"\建模数据集_方案A_MDA_未隔离.csv")
print("\n" + "=" * 70)
print("【4】建模数据集_方案A")
print("=" * 70)
print("year 范围:", m["year"].min(), "→", m["year"].max())
print("\n每年样本量 / Fraud 数 / 比例:")
print(m.groupby("year")["Fraud"].agg(
    n="size", pos="sum", rate="mean").round(4).to_string())

# 5. MD&A 特征年份（如果还能加载 notext 精简版）
import os
mda_path = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\管理层讨论与分析\保留数据\mda_features_stage2_notext.pkl"
if os.path.exists(mda_path):
    mda = pd.read_pickle(mda_path)
    print("\n" + "=" * 70)
    print("【5】MD&A 特征表")
    print("=" * 70)
    print("列名:", mda.columns.tolist()[:10], "...")
    if "year" in mda.columns:
        print("年份范围:", mda["year"].min(), "→", mda["year"].max())
        print("\n每年行数:")
        print(mda.groupby("year").size().to_string())

【1】财务+非财务面板
年份范围: 2014 → 2024

每年行数:
year
2014    3583
2015    3778
2016    4030
2017    4528
2018    4972
2019    5270
2020    5461
2021    5559
2022    5675
2023    5658
2024    5609

【2】违规标签_公司年度
列名: ['Stkcd', 'year', 'fraud_types', 'fraud_count', 'Fraud', 'first_fraud', 'repeat_fraud']
年份范围: 1999 → 2026

每年 Fraud 分布:
      size  sum  mean
year                 
1999     1    1     1
2000     1    1     1
2001     1    1     1
2002     2    2     1
2003     2    2     1
2004     3    3     1
2005     8    8     1
2006    10   10     1
2007    12   12     1
2008    15   15     1
2009    17   17     1
2010    31   31     1
2011    45   45     1
2012    66   66     1
2013   109  109     1
2014   176  176     1
2015   237  237     1
2016   255  255     1
2017   367  367     1
2018   457  457     1
2019   460  460     1
2020   487  487     1
2021   530  530     1
2022   530  530     1
2023   532  532     1
2024   386  386     1
2025   180  180     1
2026    88   88     1

【3】违规目标四类_公司年度类型

In [2]:
t = pd.read_csv(r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\违规目标四类_公司年度类型.csv")
print(t[["Stkcd","year","ViolationTypeID","DisposalDate","DeclareDate","year_source"]].head(20))
print("\nyear_source 分布:")
print(t["year_source"].value_counts())

    Stkcd  year ViolationTypeID DisposalDate DeclareDate    year_source
0       4  2022           P2503   2022-08-03  2022-08-05  ViolationYear
1       4  2022           P2503   2022-11-15  2022-11-15  ViolationYear
2       4  2022           P2503   2023-06-14  2023-06-14  ViolationYear
3       7  2012           P2503   2014-06-16  2014-06-18  ViolationYear
4       7  2013           P2503   2014-06-16  2014-06-18  ViolationYear
5       7  2012           P2503   2014-08-27  2014-08-28  ViolationYear
6       7  2013           P2503   2014-08-27  2014-08-28  ViolationYear
7       7  2012           P2503   2014-09-26  2014-09-30  ViolationYear
8       7  2013           P2503   2014-09-26  2014-09-30  ViolationYear
9       7  2014           P2506   2015-12-22  2015-12-23  ViolationYear
10      7  2015           P2503   2016-05-09  2016-05-11  ViolationYear
11      7  2019           P2503   2019-05-29  2019-05-29  ViolationYear
12      7  2021           P2503   2021-06-18  2021-06-18  Violat

In [3]:
# ============ threadpoolctl monkey-patch（必须最前）============
import threadpoolctl
class _SafeThreadpoolLimits:
    def __init__(self, *a, **k): pass
    def __enter__(self): return self
    def __exit__(self, *a): pass
threadpoolctl.threadpool_limits = _SafeThreadpoolLimits
# ================================================================

import os, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              precision_recall_curve, f1_score, recall_score,
                              precision_score, confusion_matrix, brier_score_loss)
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\建模数据集_方案A_MDA_未隔离.csv"
OUT  = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"

data = pd.read_csv(DATA)
exclude_cols = ["Stkcd","year","Fraud","ShortName","IndustryName1","ViolationTypeID",
                "DeclareDate","DisposalDate","Enddate","set"]
feats = [c for c in data.columns
         if c not in exclude_cols and pd.api.types.is_numeric_dtype(data[c])]

# ---- 严格时间三段 ----
train_df = data[data["year"].between(2015, 2021)].copy()
val_df   = data[data["year"] == 2022].copy()
test_df  = data[data["year"].between(2023, 2024)].copy()

print(f"Train 2015-2021: N={len(train_df):>6}  pos={int(train_df['Fraud'].sum())}  "
      f"rate={train_df['Fraud'].mean()*100:.2f}%")
print(f"Val   2022:      N={len(val_df):>6}  pos={int(val_df['Fraud'].sum())}  "
      f"rate={val_df['Fraud'].mean()*100:.2f}%")
print(f"Test  2023-2024: N={len(test_df):>6}  pos={int(test_df['Fraud'].sum())}  "
      f"rate={test_df['Fraud'].mean()*100:.2f}%")

X_tr = train_df[feats].values; y_tr = train_df["Fraud"].values; g_tr = train_df["Stkcd"].values
X_va = val_df[feats].values;   y_va = val_df["Fraud"].values
X_te = test_df[feats].values;  y_te = test_df["Fraud"].values

imp = SimpleImputer(strategy='median')
X_tr = imp.fit_transform(X_tr); X_va = imp.transform(X_va); X_te = imp.transform(X_te)

# ============ Stage 1: Optuna（只用训练集内部 GroupKFold）============
def objective(trial):
    params = dict(
        n_estimators       = trial.suggest_int('n_estimators', 200, 600, step=100),
        learning_rate      = trial.suggest_float('learning_rate', 0.01, 0.05, log=True),
        num_leaves         = trial.suggest_int('num_leaves', 15, 50),
        min_child_samples  = trial.suggest_int('min_child_samples', 10, 40),
        subsample          = trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree   = trial.suggest_float('colsample_bytree', 0.6, 1.0),
        reg_alpha          = trial.suggest_float('reg_alpha', 0.0, 0.5),
        reg_lambda         = trial.suggest_float('reg_lambda', 0.0, 0.5),
        class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1,
    )
    inner = GroupKFold(n_splits=3)
    scores = []
    for i_tr, i_val in inner.split(X_tr, y_tr, g_tr):
        m = lgb.LGBMClassifier(**params).fit(X_tr[i_tr], y_tr[i_tr])
        p = m.predict_proba(X_tr[i_val])[:,1]
        scores.append(average_precision_score(y_tr[i_val], p))
    return float(np.mean(scores))

t0 = time.time()
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=30, show_progress_bar=False)
best_params = {**study.best_params,
               'class_weight': 'balanced', 'random_state': 42,
               'n_jobs': -1, 'verbose': -1}
print(f"\nOptuna best PR-AUC (inner 3-fold) = {study.best_value:.4f}")
print(f"Best params: {study.best_params}")
print(f"Optuna 耗时: {(time.time()-t0)/60:.2f} min")

# ============ Stage 2: 全训练集重训 ============
m_final = lgb.LGBMClassifier(**best_params).fit(X_tr, y_tr)
p_val_raw = m_final.predict_proba(X_va)[:,1]
p_te_raw  = m_final.predict_proba(X_te)[:,1]

# ============ Stage 3: Isotonic 校准（在验证集上拟合）============
iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(p_val_raw, y_va)
p_val_cal = iso.predict(p_val_raw)
p_te_cal  = iso.predict(p_te_raw)

# ============ Stage 4: 阈值（验证集 F1 最优）============
def best_thr(y, p):
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = 2*prec[:-1]*rec[:-1]/(prec[:-1]+rec[:-1]+1e-9)
    return float(thr[np.argmax(f1)])

thr = best_thr(y_va, p_val_cal)
print(f"\n验证集 F1 最优阈值 = {thr:.4f}")

# ============ Stage 5: 测试集只评估一次 ============
def eval_block(y, p, thr, name):
    y_pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
    return dict(Set=name, N=len(y), Pos=int(y.sum()),
                AUC=roc_auc_score(y,p), PR_AUC=average_precision_score(y,p),
                Brier=brier_score_loss(y,p),
                F1=f1_score(y,y_pred,zero_division=0),
                Recall=recall_score(y,y_pred,zero_division=0),
                Precision=precision_score(y,y_pred,zero_division=0),
                Specificity=tn/(tn+fp) if (tn+fp)>0 else np.nan)

rows = [
    eval_block(y_tr, m_final.predict_proba(X_tr)[:,1], 0.5, "Train"),
    eval_block(y_va, p_val_raw, 0.5, "Val (raw)"),
    eval_block(y_va, p_val_cal, thr, "Val (calibrated)"),
    eval_block(y_te, p_te_raw, 0.5, "Test (raw)"),
    eval_block(y_te, p_te_cal, thr, "Test (calibrated)"),
]
df = pd.DataFrame(rows)
df.to_csv(os.path.join(OUT, "TimeSplit_metrics.csv"),
          index=False, encoding='utf-8-sig')
print("\n" + "="*110)
print(df.round(4).to_string(index=False))

# ============ 保存 ============
pd.DataFrame({'y_true': y_va, 'prob_raw': p_val_raw, 'prob_cal': p_val_cal}
             ).to_csv(os.path.join(OUT, "TimeSplit_val_predictions.csv"),
                      index=False, encoding='utf-8-sig')
pd.DataFrame({'y_true': y_te, 'prob_raw': p_te_raw, 'prob_cal': p_te_cal}
             ).to_csv(os.path.join(OUT, "TimeSplit_test_predictions.csv"),
                      index=False, encoding='utf-8-sig')
pd.DataFrame([best_params]).to_csv(os.path.join(OUT, "TimeSplit_best_params.csv"),
                                    index=False, encoding='utf-8-sig')

print("\n已保存 4 个文件到:", OUT)

Train 2015-2021: N= 31553  pos=2789  rate=8.84%
Val   2022:      N=  5538  pos=528  rate=9.53%
Test  2023-2024: N= 11237  pos=916  rate=8.15%

Optuna best PR-AUC (inner 3-fold) = 0.3221
Best params: {'n_estimators': 400, 'learning_rate': 0.017607718844940667, 'num_leaves': 25, 'min_child_samples': 21, 'subsample': 0.9983218288570939, 'colsample_bytree': 0.8235344780363794, 'reg_alpha': 0.24973605608441002, 'reg_lambda': 0.2772570561253204}
Optuna 耗时: 2.83 min

验证集 F1 最优阈值 = 0.2518

              Set     N  Pos    AUC  PR_AUC  Brier     F1  Recall  Precision  Specificity
            Train 31553 2789 0.9263  0.5605 0.1277 0.4766  0.9057     0.3234       0.8163
        Val (raw)  5538  528 0.7855  0.2985 0.1882 0.3261  0.7121     0.2115       0.7202
 Val (calibrated)  5538  528 0.7917  0.2943 0.0753 0.3751  0.4564     0.3184       0.8970
       Test (raw) 11237  916 0.7578  0.2506 0.2112 0.2622  0.7183     0.1603       0.6661
Test (calibrated) 11237  916 0.7563  0.2337 0.0693 0.3135  0.45

In [4]:
# ============ threadpoolctl monkey-patch ============
import threadpoolctl
class _SafeThreadpoolLimits:
    def __init__(self, *a, **k): pass
    def __enter__(self): return self
    def __exit__(self, *a): pass
threadpoolctl.threadpool_limits = _SafeThreadpoolLimits
# ====================================================

import os
import numpy as np
import pandas as pd
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss)

OUT = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
te = pd.read_csv(os.path.join(OUT, "TimeSplit_test_predictions.csv"))
y  = te["y_true"].values.astype(int)
p_raw = te["prob_raw"].values.astype(float)
p_cal = te["prob_cal"].values.astype(float)

def boot_ci(y, p, fn, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y); scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y[idx])) < 2: continue
        scores.append(fn(y[idx], p[idx]))
    s = np.array(scores)
    return s.mean(), np.percentile(s, 2.5), np.percentile(s, 97.5)

print("=" * 70)
print("Time-split Test set (2023-2024) — Bootstrap 95% CI")
print("=" * 70)
for label, p in [("Raw", p_raw), ("Calibrated", p_cal)]:
    print(f"\n[{label}]")
    for mname, fn in [("AUC", roc_auc_score),
                      ("PR-AUC", average_precision_score),
                      ("Brier", brier_score_loss)]:
        m, lo, hi = boot_ci(y, p, fn)
        print(f"    {mname:8s} = {m:.4f}   95% CI [{lo:.4f}, {hi:.4f}]")

# 保存到 CSV
rows = []
for label, p in [("Raw", p_raw), ("Calibrated", p_cal)]:
    for mname, fn in [("AUC", roc_auc_score),
                      ("PR-AUC", average_precision_score),
                      ("Brier", brier_score_loss)]:
        m, lo, hi = boot_ci(y, p, fn)
        rows.append(dict(Set="TimeSplit-Test", Version=label,
                         Metric=mname, Mean=m, CI_low=lo, CI_high=hi))
pd.DataFrame(rows).to_csv(os.path.join(OUT, "TimeSplit_Bootstrap_CI.csv"),
                          index=False, encoding='utf-8-sig')
print("\n已保存: TimeSplit_Bootstrap_CI.csv")

Time-split Test set (2023-2024) — Bootstrap 95% CI

[Raw]
    AUC      = 0.7576   95% CI [0.7412, 0.7737]
    PR-AUC   = 0.2515   95% CI [0.2249, 0.2795]
    Brier    = 0.2112   95% CI [0.2074, 0.2149]

[Calibrated]
    AUC      = 0.7561   95% CI [0.7399, 0.7728]
    PR-AUC   = 0.2343   95% CI [0.2095, 0.2605]
    Brier    = 0.0693   95% CI [0.0659, 0.0727]

已保存: TimeSplit_Bootstrap_CI.csv


In [5]:
# ============ threadpoolctl monkey-patch ============
import threadpoolctl
class _SafeThreadpoolLimits:
    def __init__(self, *a, **k): pass
    def __enter__(self): return self
    def __exit__(self, *a): pass
threadpoolctl.threadpool_limits = _SafeThreadpoolLimits
# ====================================================

import os
import numpy as np
import pandas as pd
from scipy import stats

OUT = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
data = pd.read_csv(os.path.join(OUT, "建模数据集_方案A_MDA_未隔离.csv"))

# 检查基本列
print("列名前 20:", data.columns.tolist()[:20])
print("列名后 20:", data.columns.tolist()[-20:])

# 特征列（与建模一致）
exclude_cols = ["Stkcd","year","Fraud","ShortName","IndustryName1","ViolationTypeID",
                "DeclareDate","DisposalDate","Enddate","set"]
feats = [c for c in data.columns
         if c not in exclude_cols and pd.api.types.is_numeric_dtype(data[c])]

# ============ 1. 年度分布 ============
print("\n" + "=" * 90)
print("Table A. Yearly distribution of fraud and non-fraud observations")
print("=" * 90)
yearly = data.groupby("year")["Fraud"].agg(
    N="size", Fraud="sum").reset_index()
yearly["NonFraud"] = yearly["N"] - yearly["Fraud"]
yearly["FraudRate(%)"] = (yearly["Fraud"] / yearly["N"] * 100).round(2)
yearly = yearly[["year","N","Fraud","NonFraud","FraudRate(%)"]]
print(yearly.to_string(index=False))
yearly.to_csv(os.path.join(OUT, "Desc_Yearly.csv"),
              index=False, encoding='utf-8-sig')

# ============ 2. 行业分布 ============
print("\n" + "=" * 90)
print("Table B. Industry distribution")
print("=" * 90)
if "IndustryName1" in data.columns:
    ind = data.groupby("IndustryName1")["Fraud"].agg(
        N="size", Fraud="sum").reset_index()
    ind["FraudRate(%)"] = (ind["Fraud"] / ind["N"] * 100).round(2)
    ind = ind.sort_values("N", ascending=False)
    print(ind.to_string(index=False))
    ind.to_csv(os.path.join(OUT, "Desc_Industry.csv"),
               index=False, encoding='utf-8-sig')
else:
    print("IndustryName1 列不存在，跳过")

# ============ 3. 违规类型分布 ============
print("\n" + "=" * 90)
print("Table C. Distribution of violation types (P2501/02/03/06)")
print("=" * 90)
try:
    viol = pd.read_csv(os.path.join(OUT, "违规目标四类_公司年度类型.csv"))
    # 只保留 2015-2024 且 IsViolated==1
    if "year_source" in viol.columns:
        viol = viol[viol["year"].between(2015, 2024)]
    if "IsViolated" in viol.columns:
        viol = viol[viol["IsViolated"] == 1]

    vt = viol.groupby("ViolationTypeID").agg(
        N="size",
        UniqueFirms=("Stkcd", "nunique"),
        UniqueFirmYears=("year", "nunique"),
    ).reset_index()
    # 加上名称
    if "ViolationTypeName" in viol.columns:
        names = viol.groupby("ViolationTypeID")["ViolationTypeName"].first()
        vt["TypeName"] = vt["ViolationTypeID"].map(names)
    print(vt.to_string(index=False))
    vt.to_csv(os.path.join(OUT, "Desc_ViolationTypes.csv"),
              index=False, encoding='utf-8-sig')

    # 重叠情况：一个 firm-year 有多少种违规
    overlap = viol.groupby(["Stkcd","year"])["ViolationTypeID"].nunique()
    print("\n一个 firm-year 的违规类型数分布:")
    print(overlap.value_counts().sort_index().to_string())
except Exception as e:
    print("读取违规类型失败:", e)

# ============ 4. 首犯 vs 累犯 ============
print("\n" + "=" * 90)
print("Table D. First-time vs repeat offenders")
print("=" * 90)
try:
    vl = pd.read_csv(os.path.join(OUT, "违规标签_公司年度.csv"))
    vl = vl[vl["year"].between(2015, 2024)]
    if "first_fraud" in vl.columns and "repeat_fraud" in vl.columns:
        # 只看 Fraud=1 的
        f = vl[vl["Fraud"] == 1]
        print(f"Fraud=1 总样本: {len(f)}")
        print(f"  first_fraud=1: {int(f['first_fraud'].sum())}")
        print(f"  repeat_fraud=1: {int(f['repeat_fraud'].sum())}")
        print(f"  两者都为 1: {int(((f['first_fraud']==1)&(f['repeat_fraud']==1)).sum())}")
        print(f"  两者都为 0: {int(((f['first_fraud']==0)&(f['repeat_fraud']==0)).sum())}")
except Exception as e:
    print("读取违规标签失败:", e)

# ============ 5. 特征描述性统计 ============
print("\n" + "=" * 90)
print("Table E. Descriptive statistics of all features")
print("=" * 90)
desc = data[feats].describe(percentiles=[0.25, 0.5, 0.75]).T
desc["missing"] = data[feats].isna().sum()
desc["missing_rate"] = (desc["missing"] / len(data) * 100).round(2)
desc = desc[["count","missing","missing_rate","mean","std","min","25%","50%","75%","max"]]
desc = desc.round(4)
desc.to_csv(os.path.join(OUT, "Desc_Features.csv"), encoding='utf-8-sig')
print(desc.head(20).to_string())

# ============ 6. 舞弊 vs 非舞弊 组间检验 ============
print("\n" + "=" * 90)
print("Table F. Univariate comparison: fraud vs non-fraud (t-test + Mann-Whitney)")
print("=" * 90)
rows = []
for f in feats:
    g1 = data.loc[data["Fraud"]==1, f].dropna().values
    g0 = data.loc[data["Fraud"]==0, f].dropna().values
    if len(g1) < 10 or len(g0) < 10:
        continue
    try:
        t, p_t = stats.ttest_ind(g1, g0, equal_var=False)
    except Exception:
        t, p_t = np.nan, np.nan
    try:
        u, p_u = stats.mannwhitneyu(g1, g0, alternative='two-sided')
    except Exception:
        u, p_u = np.nan, np.nan
    rows.append(dict(
        Feature=f,
        Fraud_mean=float(np.mean(g1)),
        Fraud_median=float(np.median(g1)),
        Fraud_std=float(np.std(g1)),
        NonFraud_mean=float(np.mean(g0)),
        NonFraud_median=float(np.median(g0)),
        NonFraud_std=float(np.std(g0)),
        t_stat=float(t) if not np.isnan(t) else np.nan,
        t_pvalue=float(p_t) if not np.isnan(p_t) else np.nan,
        mw_pvalue=float(p_u) if not np.isnan(p_u) else np.nan,
    ))
group_test = pd.DataFrame(rows)
group_test["Sig_5pct"] = group_test["mw_pvalue"] < 0.05
group_test.to_csv(os.path.join(OUT, "Desc_GroupTest.csv"),
                  index=False, encoding='utf-8-sig')

print(f"总特征数: {len(group_test)}")
print(f"Mann-Whitney p<0.05 的特征数: {int(group_test['Sig_5pct'].sum())} / {len(group_test)}")
print("\n最显著的 15 个特征（按 Mann-Whitney p 值）:")
print(group_test.sort_values("mw_pvalue").head(15).to_string(index=False))

# ============ 7. 样本筛选流程图用的数字 ============
print("\n" + "=" * 90)
print("Sample selection summary (for flow chart)")
print("=" * 90)
print(f"Total firm-year observations: {len(data)}")
print(f"Unique firms: {data['Stkcd'].nunique()}")
print(f"Year range: {data['year'].min()} - {data['year'].max()}")
print(f"Fraud observations: {int(data['Fraud'].sum())} ({data['Fraud'].mean()*100:.2f}%)")
print(f"Non-fraud observations: {int((data['Fraud']==0).sum())}")

print("\n已保存:")
print("  Desc_Yearly.csv")
print("  Desc_Industry.csv (if exists)")
print("  Desc_ViolationTypes.csv")
print("  Desc_Features.csv")
print("  Desc_GroupTest.csv")

列名前 20: ['Stkcd', 'year', 'Fraud', 'F010101A', 'F010201A', 'F010401A', 'F010701B', 'F010801B', 'F011201A', 'F080501A', 'F080601A', 'F081001B', 'F081101B', 'F081201B', 'F081601B', 'F082201B', 'F082701A', 'F060101B', 'F060301B', 'F050101B']
列名后 20: ['NegRatio', 'SentLenAvg', 'SentLenStd', 'ComplexWordRatio', 'DigitDensity', 'PuncDensity', 'TTR', 'Jaccard_prev', 'EditSim_prev', 'TFIDF_Cosine_prev', 'DLUT_PosNum', 'DLUT_NegNum', 'DLUT_PosRatio', 'DLUT_NegRatio', 'DLUT_PosIntensity', 'DLUT_NegIntensity', 'DLUT_EmotionScore', 'DLUT_EmotionTone', 'DLUT_NegAfterNeg', 'DLUT_PosAfterNeg']

Table A. Yearly distribution of fraud and non-fraud observations
 year    N  Fraud  NonFraud  FraudRate(%)
 2015 3579    237      3342          6.62
 2016 3771    254      3517          6.74
 2017 4026    366      3660          9.09
 2018 4522    457      4065         10.11
 2019 4962    458      4504          9.23
 2020 5254    487      4767          9.27
 2021 5439    530      4909          9.74
 2022 5538  

In [6]:
# ============ threadpoolctl monkey-patch ============
import threadpoolctl
class _SafeThreadpoolLimits:
    def __init__(self, *a, **k): pass
    def __enter__(self): return self
    def __exit__(self, *a): pass
threadpoolctl.threadpool_limits = _SafeThreadpoolLimits
# ====================================================

import os
import pandas as pd
import numpy as np

OUT = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"

# ============ 先探查哪些文件有行业列 ============
print("探查行业列来源:")
for fname in ["财务_非财务_合并面板.csv", "财务_非财务_违规标签_合并面板.csv",
              "违规展开明细_公司年度类型.csv", "违规目标四类_公司年度类型.csv",
              "违规标签_公司年度.csv"]:
    p = os.path.join(OUT, fname)
    if os.path.exists(p):
        df = pd.read_csv(p, nrows=5)
        has_ind = [c for c in df.columns if "Industry" in c or "industry" in c]
        has_short = [c for c in df.columns if "ShortName" in c]
        print(f"  {fname}: industry_cols={has_ind}  shortname_cols={has_short}")
    else:
        print(f"  {fname}: 不存在")

print("\n" + "=" * 90)
print("Table C. Violation type distribution (P2501/02/03/06)")
print("=" * 90)
viol = pd.read_csv(os.path.join(OUT, "违规目标四类_公司年度类型.csv"))
# 过滤：2015-2024，IsViolated=1
if "IsViolated" in viol.columns:
    viol = viol[viol["IsViolated"] == 1]
viol = viol[viol["year"].between(2015, 2024)]

vt = viol.groupby("ViolationTypeID").agg(
    N=("Stkcd", "size"),
    UniqueFirms=("Stkcd", "nunique"),
    UniqueFirmYears=("year", "nunique"),
).reset_index()

if "ViolationTypeName" in viol.columns:
    names = viol.groupby("ViolationTypeID")["ViolationTypeName"].first()
    vt["TypeName"] = vt["ViolationTypeID"].map(names)
vt = vt.sort_values("N", ascending=False)
print(vt.to_string(index=False))
vt.to_csv(os.path.join(OUT, "Desc_ViolationTypes.csv"),
          index=False, encoding='utf-8-sig')

# 同一 firm-year 有几个违规类型
overlap = viol.groupby(["Stkcd", "year"])["ViolationTypeID"].nunique()
print("\n一个 firm-year 的违规类型数分布:")
print(overlap.value_counts().sort_index().to_string())

# year_source 分布（2015-2024 内）
if "year_source" in viol.columns:
    print("\nyear_source 分布（2015-2024）:")
    print(viol["year_source"].value_counts().to_string())

# ============ 行业分布（合并进来）============
print("\n" + "=" * 90)
print("Table B. Industry distribution")
print("=" * 90)

data = pd.read_csv(os.path.join(OUT, "建模数据集_方案A_MDA_未隔离.csv"))

# 尝试从财务面板 merge 行业
panel_path = os.path.join(OUT, "财务_非财务_合并面板.csv")
if os.path.exists(panel_path):
    panel = pd.read_csv(panel_path)
    ind_cols = [c for c in panel.columns if "Industry" in c]
    print(f"财务面板中的行业列: {ind_cols}")
    if ind_cols:
        ind_col = ind_cols[0]
        key_df = panel[["Stkcd", "year", ind_col]].drop_duplicates(["Stkcd", "year"])
        data = data.merge(key_df, on=["Stkcd", "year"], how="left")
        print(f"合并后缺失行业信息: {data[ind_col].isna().sum()}")

        ind = data.groupby(ind_col)["Fraud"].agg(
            N="size", Fraud="sum").reset_index()
        ind["NonFraud"] = ind["N"] - ind["Fraud"]
        ind["FraudRate(%)"] = (ind["Fraud"] / ind["N"] * 100).round(2)
        ind = ind.sort_values("N", ascending=False)
        print(ind.to_string(index=False))
        ind.to_csv(os.path.join(OUT, "Desc_Industry.csv"),
                   index=False, encoding='utf-8-sig')

# ============ 核对 Table D 的 4241 vs 4233 ============
print("\n" + "=" * 90)
print("核对 Fraud=1 数量差异")
print("=" * 90)
vl = pd.read_csv(os.path.join(OUT, "违规标签_公司年度.csv"))
vl = vl[vl["year"].between(2015, 2024)]
print(f"违规标签表 (2015-2024): Fraud=1 数量 = {int(vl['Fraud'].sum())}")
print(f"建模数据集:              Fraud=1 数量 = {int(data['Fraud'].sum())}")
print(f"差异 = {int(vl['Fraud'].sum()) - int(data['Fraud'].sum())}")

# 看是哪些年份丢了
vl_y = vl[vl["Fraud"]==1].groupby("year").size().rename("from_label")
data_y = data[data["Fraud"]==1].groupby("year").size().rename("from_model")
diff = pd.concat([vl_y, data_y], axis=1).fillna(0).astype(int)
diff["diff"] = diff["from_label"] - diff["from_model"]
print("\n逐年 Fraud=1 数量对比:")
print(diff.to_string())

探查行业列来源:
  财务_非财务_合并面板.csv: industry_cols=[]  shortname_cols=[]
  财务_非财务_违规标签_合并面板.csv: industry_cols=[]  shortname_cols=[]
  违规展开明细_公司年度类型.csv: industry_cols=[]  shortname_cols=[]
  违规目标四类_公司年度类型.csv: industry_cols=[]  shortname_cols=[]
  违规标签_公司年度.csv: industry_cols=[]  shortname_cols=[]

Table C. Violation type distribution (P2501/02/03/06)
Empty DataFrame
Columns: [ViolationTypeID, N, UniqueFirms, UniqueFirmYears, TypeName]
Index: []

一个 firm-year 的违规类型数分布:
Series([], )

year_source 分布（2015-2024）:
Series([], )

Table B. Industry distribution
财务面板中的行业列: []

核对 Fraud=1 数量差异
违规标签表 (2015-2024): Fraud=1 数量 = 4241
建模数据集:              Fraud=1 数量 = 4233
差异 = 8

逐年 Fraud=1 数量对比:
      from_label  from_model  diff
year                              
2015         237         237     0
2016         255         254     1
2017         367         366     1
2018         457         457     0
2019         460         458     2
2020         487         487     0
2021         530         530     0
20

In [7]:
# ============ threadpoolctl monkey-patch ============
import threadpoolctl
class _SafeThreadpoolLimits:
    def __init__(self, *a, **k): pass
    def __enter__(self): return self
    def __exit__(self, *a): pass
threadpoolctl.threadpool_limits = _SafeThreadpoolLimits
# ====================================================

import os
import pandas as pd
import numpy as np

OUT = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
MDA = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\管理层讨论与分析\保留数据\mda_features_stage2_notext.pkl"

# ==========================================================
# Part 1: 探查违规类型表，正确过滤
# ==========================================================
print("=" * 90)
print("Part 1: 探查违规类型表")
print("=" * 90)

viol = pd.read_csv(os.path.join(OUT, "违规目标四类_公司年度类型.csv"))
print(f"原始 shape: {viol.shape}")
print(f"列名: {viol.columns.tolist()}")
print(f"\nViolationTypeID 分布:")
print(viol["ViolationTypeID"].value_counts(dropna=False).to_string())
print(f"\nIsViolated 分布:")
print(viol["IsViolated"].value_counts(dropna=False).to_string())
print(f"\nIsViolated dtype: {viol['IsViolated'].dtype}")
print(f"\nyear_source 分布:")
print(viol["year_source"].value_counts(dropna=False).to_string())
print(f"\nyear 范围: {viol['year'].min()} - {viol['year'].max()}")
print(f"\n前 10 行:")
print(viol.head(10).to_string())

# ---- 合理推断：这个表本身可能已经是"违规明细"，不需要 IsViolated 过滤 ----
print("\n" + "=" * 90)
print("Part 1 修复版：只按年份过滤（2015-2024）")
print("=" * 90)

viol2 = viol[viol["year"].between(2015, 2024)].copy()
print(f"过滤后 shape: {viol2.shape}")

vt = viol2.groupby("ViolationTypeID").agg(
    N=("Stkcd", "size"),
    UniqueFirms=("Stkcd", "nunique"),
    UniqueYears=("year", "nunique"),
).reset_index()

if "ViolationTypeName" in viol2.columns:
    names = viol2.groupby("ViolationTypeID")["ViolationTypeName"].first()
    vt["TypeName"] = vt["ViolationTypeID"].map(names)

vt = vt.sort_values("N", ascending=False)
print("\n四类违规分布 (2015-2024):")
print(vt.to_string(index=False))
vt.to_csv(os.path.join(OUT, "Desc_ViolationTypes.csv"),
          index=False, encoding='utf-8-sig')

# 重叠情况
overlap = viol2.groupby(["Stkcd", "year"])["ViolationTypeID"].nunique()
print("\n同一 firm-year 的违规类型数量分布:")
print(overlap.value_counts().sort_index().to_string())

# ==========================================================
# Part 2: 从 MD&A 表 merge 行业列
# ==========================================================
print("\n" + "=" * 90)
print("Part 2: 从 MD&A 表 merge 行业列")
print("=" * 90)

mda = pd.read_pickle(MDA)
print(f"MD&A shape: {mda.shape}")
print(f"MD&A 列名前 10: {mda.columns.tolist()[:10]}")

# 找行业列（可能有多个候选）
ind_cols = [c for c in mda.columns if "Industry" in c]
print(f"MD&A 中的行业列: {ind_cols}")

# 找 code/year 列
code_cols = [c for c in mda.columns if c in ("Symbol", "Stkcd", "code")]
year_cols = [c for c in mda.columns if c in ("year", "Year", "Enddate")]
print(f"MD&A 中的代码列: {code_cols}, 年份列: {year_cols}")

# 统一 Symbol → Stkcd
if "Symbol" in mda.columns and "Stkcd" not in mda.columns:
    mda["Stkcd"] = mda["Symbol"]

# 如果只有 Enddate 需要转成 year
if "year" not in mda.columns and "Enddate" in mda.columns:
    mda["year"] = pd.to_datetime(mda["Enddate"], errors='coerce').dt.year

# 提取去重的 (Stkcd, year, IndustryName1)
if ind_cols and "Stkcd" in mda.columns and "year" in mda.columns:
    ind_col = ind_cols[0]
    ind_df = mda[["Stkcd", "year", ind_col]].dropna(subset=[ind_col]).drop_duplicates(["Stkcd", "year"])
    print(f"\n行业表 shape: {ind_df.shape}, 唯一行业数: {ind_df[ind_col].nunique()}")
    print(f"\n行业 Top 15:")
    print(ind_df[ind_col].value_counts().head(15).to_string())
    ind_df.to_csv(os.path.join(OUT, "CompanyIndustryMap.csv"),
                  index=False, encoding='utf-8-sig')

    # merge 回建模数据集
    data = pd.read_csv(os.path.join(OUT, "建模数据集_方案A_MDA_未隔离.csv"))
    data = data.merge(ind_df, on=["Stkcd", "year"], how="left")
    print(f"\n合并后: 缺失行业 = {data[ind_col].isna().sum()} / {len(data)}")

    # 行业分布表
    ind_stats = data.groupby(ind_col)["Fraud"].agg(
        N="size", Fraud="sum").reset_index()
    ind_stats["NonFraud"] = ind_stats["N"] - ind_stats["Fraud"]
    ind_stats["FraudRate(%)"] = (ind_stats["Fraud"] / ind_stats["N"] * 100).round(2)
    ind_stats = ind_stats.sort_values("N", ascending=False)
    print("\n行业分布表:")
    print(ind_stats.to_string(index=False))
    ind_stats.to_csv(os.path.join(OUT, "Desc_Industry.csv"),
                     index=False, encoding='utf-8-sig')

# ==========================================================
# Part 3: 行业 × 舞弊类型 交叉表（如果类型可用）
# ==========================================================
print("\n" + "=" * 90)
print("Part 3: 行业 × 舞弊类型 交叉表（回应 R2-9 行业异质性）")
print("=" * 90)

if 'ind_df' in dir() and not vt.empty:
    # 从 viol2 拿到 firm-year 级别的违规类型（取第一个，如果有多个）
    fy_type = viol2.groupby(["Stkcd", "year"])["ViolationTypeID"].first().reset_index()
    merged = fy_type.merge(ind_df, on=["Stkcd", "year"], how="left")
    cross = pd.crosstab(merged[ind_col], merged["ViolationTypeID"])
    print(cross.to_string())
    cross.to_csv(os.path.join(OUT, "Desc_Industry_x_ViolationType.csv"),
                 encoding='utf-8-sig')

# ==========================================================
# Part 4: 8 个差异的明细（论文要一句话解释）
# ==========================================================
print("\n" + "=" * 90)
print("Part 4: Fraud=1 数量差异明细")
print("=" * 90)
vl = pd.read_csv(os.path.join(OUT, "违规标签_公司年度.csv"))
data = pd.read_csv(os.path.join(OUT, "建模数据集_方案A_MDA_未隔离.csv"))

# 找出违规标签里有、但建模数据里没有的 firm-year
key_label = set(zip(vl.loc[vl["Fraud"]==1, "Stkcd"], vl.loc[vl["Fraud"]==1, "year"]))
key_model = set(zip(data.loc[data["Fraud"]==1, "Stkcd"], data.loc[data["Fraud"]==1, "year"]))
missing = key_label - key_model
print(f"违规表有但建模数据缺失的 firm-year: {len(missing)}")
if len(missing) > 0:
    missing_df = pd.DataFrame(list(missing), columns=["Stkcd", "year"])
    missing_df = missing_df.sort_values(["year", "Stkcd"])
    print(missing_df.to_string(index=False))
    missing_df.to_csv(os.path.join(OUT, "Desc_MissingFraudRows.csv"),
                      index=False, encoding='utf-8-sig')

print("\n已保存:")
print("  Desc_ViolationTypes.csv")
print("  CompanyIndustryMap.csv")
print("  Desc_Industry.csv")
print("  Desc_Industry_x_ViolationType.csv")
print("  Desc_MissingFraudRows.csv")

Part 1: 探查违规类型表
原始 shape: (9392, 9)
列名: ['ViolationID', 'Stkcd', 'year', 'ViolationTypeID', 'ViolationTypeName', 'IsViolated', 'DisposalDate', 'DeclareDate', 'year_source']

ViolationTypeID 分布:
P2503    7562
P2501    1407
P2506     231
P2502     192

IsViolated 分布:
Y    9392

IsViolated dtype: object

year_source 分布:
ViolationYear           9255
DeclareDate_fallback     137

year 范围: 1999 - 2026

前 10 行:
   ViolationID  Stkcd  year ViolationTypeID ViolationTypeName IsViolated DisposalDate DeclareDate    year_source
0     40121966      4  2022           P2503       虚假记载(误导性陈述)          Y   2022-08-03  2022-08-05  ViolationYear
1     40127455      4  2022           P2503       虚假记载(误导性陈述)          Y   2022-11-15  2022-11-15  ViolationYear
2     40138124      4  2022           P2503       虚假记载(误导性陈述)          Y   2023-06-14  2023-06-14  ViolationYear
3      4013493      7  2012           P2503       虚假记载(误导性陈述)          Y   2014-06-16  2014-06-18  ViolationYear
4      4013493      7  2013

ValueError: You are trying to merge on int64 and object columns. If you wish to proceed you should use pd.concat

In [8]:
# ============ threadpoolctl monkey-patch ============
import threadpoolctl
class _SafeThreadpoolLimits:
    def __init__(self, *a, **k): pass
    def __enter__(self): return self
    def __exit__(self, *a): pass
threadpoolctl.threadpool_limits = _SafeThreadpoolLimits
# ====================================================

import os
import pandas as pd
import numpy as np

OUT = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
MDA = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\管理层讨论与分析\保留数据\mda_features_stage2_notext.pkl"

# ---------- 统一 Stkcd 为 6 位字符串 ----------
def normalize_code(s):
    """把 int/float/str 统一成 6 位字符串，如 4 / 4.0 / '4' / '000004' -> '000004'"""
    s = s.astype(str)
    s = s.str.replace(r'\.0$', '', regex=True)   # 去掉 float 尾巴
    s = s.str.strip()
    return s.str.zfill(6)

# ---------- 加载 ----------
mda = pd.read_pickle(MDA)
if "year" not in mda.columns and "Enddate" in mda.columns:
    mda["year"] = pd.to_datetime(mda["Enddate"], errors='coerce').dt.year

ind_df = (mda[["Stkcd", "year", "IndustryName1"]]
          .dropna(subset=["IndustryName1"])
          .drop_duplicates(["Stkcd", "year"])
          .copy())

ind_df["Stkcd"] = normalize_code(ind_df["Stkcd"])
ind_df["year"]  = ind_df["year"].astype(int)

print(f"行业映射表 shape: {ind_df.shape}")
print(f"唯一行业数: {ind_df['IndustryName1'].nunique()}")
print(ind_df.head(5).to_string())

# ---------- merge 回建模数据集 ----------
data = pd.read_csv(os.path.join(OUT, "建模数据集_方案A_MDA_未隔离.csv"))
data["Stkcd"] = normalize_code(data["Stkcd"])
data["year"]  = data["year"].astype(int)

data_merged = data.merge(ind_df, on=["Stkcd", "year"], how="left")
n_missing = data_merged["IndustryName1"].isna().sum()
print(f"\n合并后: 缺失行业 = {n_missing} / {len(data_merged)} ({n_missing/len(data_merged)*100:.2f}%)")

# ---------- 行业分布表 ----------
ind_stats = data_merged.groupby("IndustryName1")["Fraud"].agg(
    N="size", Fraud="sum").reset_index()
ind_stats["NonFraud"] = ind_stats["N"] - ind_stats["Fraud"]
ind_stats["FraudRate(%)"] = (ind_stats["Fraud"] / ind_stats["N"] * 100).round(2)
ind_stats = ind_stats.sort_values("N", ascending=False)

print("\n行业分布 (Top 20):")
print(ind_stats.head(20).to_string(index=False))

ind_stats.to_csv(os.path.join(OUT, "Desc_Industry.csv"),
                 index=False, encoding='utf-8-sig')

# ---------- 行业 × 违规类型 交叉表 ----------
viol = pd.read_csv(os.path.join(OUT, "违规目标四类_公司年度类型.csv"))
viol = viol[viol["year"].between(2015, 2024)].copy()
viol["Stkcd"] = normalize_code(viol["Stkcd"])
viol["year"]  = viol["year"].astype(int)

# 每个 firm-year 取第一个类型（或改成"是否含某类型"）
fy_type = viol.groupby(["Stkcd", "year"])["ViolationTypeID"].first().reset_index()
merged = fy_type.merge(ind_df, on=["Stkcd", "year"], how="left")

cross = pd.crosstab(merged["IndustryName1"], merged["ViolationTypeID"])
cross["Total"] = cross.sum(axis=1)
cross = cross.sort_values("Total", ascending=False)
print("\n行业 × 违规类型 交叉表 (Top 15):")
print(cross.head(15).to_string())
cross.to_csv(os.path.join(OUT, "Desc_Industry_x_ViolationType.csv"),
             encoding='utf-8-sig')

# ---------- 8 个差异明细 ----------
print("\n" + "=" * 90)
print("Fraud=1 数量差异明细")
print("=" * 90)
vl = pd.read_csv(os.path.join(OUT, "违规标签_公司年度.csv"))
vl["Stkcd"] = normalize_code(vl["Stkcd"])
vl["year"]  = vl["year"].astype(int)

key_label = set(zip(vl.loc[vl["Fraud"]==1, "Stkcd"], vl.loc[vl["Fraud"]==1, "year"]))
key_model = set(zip(data.loc[data["Fraud"]==1, "Stkcd"], data.loc[data["Fraud"]==1, "year"]))
missing = sorted(key_label - key_model, key=lambda x: (x[1], x[0]))
print(f"违规表有但建模数据缺失: {len(missing)} 条")
missing_df = pd.DataFrame(missing, columns=["Stkcd", "year"])
print(missing_df.to_string(index=False))
missing_df.to_csv(os.path.join(OUT, "Desc_MissingFraudRows.csv"),
                  index=False, encoding='utf-8-sig')

# ---------- 重新保存带行业的建模数据（可选，供后续分层实验用）----------
data_merged.to_csv(os.path.join(OUT, "建模数据集_方案A_MDA_带行业.csv"),
                   index=False, encoding='utf-8-sig')
print("\n已保存:")
print("  Desc_Industry.csv")
print("  Desc_Industry_x_ViolationType.csv")
print("  Desc_MissingFraudRows.csv")
print("  建模数据集_方案A_MDA_带行业.csv")


行业映射表 shape: (16333, 3)
唯一行业数: 78
    Stkcd  year IndustryName1
2  000002  2015          房地产业
3  000002  2016          房地产业
4  000002  2017          房地产业
5  000002  2018          房地产业
6  000002  2019          房地产业

合并后: 缺失行业 = 32001 / 48328 (66.22%)

行业分布 (Top 20):
       IndustryName1    N  Fraud  NonFraud  FraudRate(%)
    计算机、通信和其他电子设备制造业 1566    188      1378         12.01
        化学原料和化学制品制造业 1112     99      1013          8.90
          电气机械和器材制造业 1053    139       914         13.20
               医药制造业  990    115       875         11.62
             专用设备制造业  970    109       861         11.24
          软件和信息技术服务业  858    100       758         11.66
                房地产业  620     42       578          6.77
             通用设备制造业  616     49       567          7.95
               汽车制造业  564     45       519          7.98
                 零售业  435     33       402          7.59
            非金属矿物制品业  419     51       368         12.17
                 批发业  362     56       306        


已保存:
  Desc_Industry.csv
  Desc_Industry_x_ViolationType.csv
  Desc_MissingFraudRows.csv
  建模数据集_方案A_MDA_带行业.csv


In [9]:
# ============ threadpoolctl monkey-patch ============
import threadpoolctl
class _SafeThreadpoolLimits:
    def __init__(self, *a, **k): pass
    def __enter__(self): return self
    def __exit__(self, *a): pass
threadpoolctl.threadpool_limits = _SafeThreadpoolLimits
# ====================================================

import os, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              precision_recall_curve, f1_score, recall_score,
                              precision_score, confusion_matrix, brier_score_loss)
import lightgbm as lgb

OUT = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"

data = pd.read_csv(os.path.join(OUT, "建模数据集_方案A_MDA_未隔离.csv"))
exclude_cols = ["Stkcd","year","Fraud","ShortName","IndustryName1","ViolationTypeID",
                "DeclareDate","DisposalDate","Enddate","set"]
feats = [c for c in data.columns
         if c not in exclude_cols and pd.api.types.is_numeric_dtype(data[c])]

# 统一 Stkcd 格式
def norm(s): return s.astype(str).str.replace(r'\.0$','',regex=True).str.zfill(6)
data["Stkcd"] = norm(data["Stkcd"])

y      = data["Fraud"].values
groups = data["Stkcd"].values
X_all  = data[feats].values

# 用之前跑出的时间外推最优参数
best_params = dict(
    n_estimators=400, learning_rate=0.0176, num_leaves=25, min_child_samples=21,
    subsample=0.998, colsample_bytree=0.824, reg_alpha=0.250, reg_lambda=0.277,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
)

# ---- 主模型 GroupKFold OOF ----
gkf = GroupKFold(n_splits=5)
oof_prob = np.zeros(len(y))
oof_cal = np.zeros(len(y))
for fold, (tr, te) in enumerate(gkf.split(X_all, y, groups), 1):
    imp = SimpleImputer(strategy='median')
    X_tr = imp.fit_transform(X_all[tr]); X_te = imp.transform(X_all[te])

    m = lgb.LGBMClassifier(**best_params).fit(X_tr, y[tr])
    p = m.predict_proba(X_te)[:, 1]

    # 内层 3 折校准
    inner = GroupKFold(n_splits=3)
    oof_in = np.zeros(len(tr))
    for i_tr, i_val in inner.split(X_tr, y[tr], groups[tr]):
        imp_i = SimpleImputer(strategy='median')
        Xi = imp_i.fit_transform(X_tr[i_tr]); Xv = imp_i.transform(X_tr[i_val])
        mi = lgb.LGBMClassifier(**best_params).fit(Xi, y[tr][i_tr])
        oof_in[i_val] = mi.predict_proba(Xv)[:, 1]
    iso = IsotonicRegression(out_of_bounds='clip').fit(oof_in, y[tr])

    oof_prob[te] = p
    oof_cal[te]  = iso.predict(p)
    print(f"Fold {fold} done")

# 阈值：全局 OOF F1 最优
prec, rec, thr = precision_recall_curve(y, oof_cal)
f1s = 2*prec[:-1]*rec[:-1]/(prec[:-1]+rec[:-1]+1e-9)
THR = float(thr[np.argmax(f1s)])
print(f"\n全局 F1 最优阈值 = {THR:.4f}")

# ---- 违规类型标签 ----
viol = pd.read_csv(os.path.join(OUT, "违规目标四类_公司年度类型.csv"))
viol = viol[viol["year"].between(2015, 2024)].copy()
viol["Stkcd"] = norm(viol["Stkcd"])
viol["year"]  = viol["year"].astype(int)

# 每个 firm-year 有哪些类型
type_map = viol.groupby(["Stkcd","year"])["ViolationTypeID"].apply(set).to_dict()

data["Stkcd"] = norm(data["Stkcd"])
data["year"]  = data["year"].astype(int)
data["vtypes"] = list(zip(data["Stkcd"], data["year"]))
data["vtypes"] = data["vtypes"].map(type_map)

TYPES = ["P2501", "P2502", "P2503", "P2506"]
TYPE_NAMES = {
    "P2501": "Fabricated profits",
    "P2502": "Fictitious asset reporting",
    "P2503": "False or misleading disclosures",
    "P2506": "Inaccurate disclosures (other)",
}

# ---- 每个类型分别评估（子样本评估 + 全样本召回率）----
rows = []
for vtype in TYPES:
    # 子样本：所有 Fraud=1 且含该类型 + 所有 Fraud=0
    has_type = data["vtypes"].apply(lambda s: isinstance(s, set) and vtype in s)
    sub = data[has_type | (data["Fraud"]==0)].copy()

    # 用主模型 OOF 概率
    p_global = oof_cal[sub.index]

    # 在该子样本内单独评估
    y_sub = sub["Fraud"].values

    # 该类型 fraud 的召回率（在全局阈值下）
    fraud_idx = (sub["Fraud"]==1).values
    if fraud_idx.sum() > 0:
        caught = (p_global[fraud_idx] >= THR).sum()
        recall = caught / fraud_idx.sum()
    else:
        recall = np.nan

    rows.append(dict(
        ViolationType=vtype,
        TypeName=TYPE_NAMES[vtype],
        N_FraudFirmYears=int(fraud_idx.sum()),
        Recall_on_this_type=recall,
        Mean_Prob_fraud=float(p_global[fraud_idx].mean()) if fraud_idx.sum()>0 else np.nan,
        Mean_Prob_nonfraud=float(p_global[~fraud_idx].mean()) if (~fraud_idx).sum()>0 else np.nan,
    ))

df = pd.DataFrame(rows)
df.to_csv(os.path.join(OUT, "ViolationType_stratification.csv"),
          index=False, encoding='utf-8-sig')
print("\n" + "="*100)
print("违规类型分层结果")
print("="*100)
print(df.round(4).to_string(index=False))

# ---- 附：多类型 vs 单类型的对比 ----
data["n_vtypes"] = data["vtypes"].apply(
    lambda s: len(s) if isinstance(s, set) else 0)

print("\n" + "="*100)
print("单一类型 vs 多类型 fraud 的召回率")
print("="*100)
for n in [1, 2, 3]:
    idx = (data["Fraud"]==1) & (data["n_vtypes"]==n)
    if idx.sum() > 0:
        caught = (oof_cal[idx] >= THR).sum()
        print(f"  {n} 种类型 (N={idx.sum()}): "
              f"Recall={caught/idx.sum():.4f}, "
              f"Mean_Prob={oof_cal[idx].mean():.4f}")

# ---- 首犯 vs 累犯 ----
print("\n" + "="*100)
print("首犯 vs 累犯 召回率")
print("="*100)
vl = pd.read_csv(os.path.join(OUT, "违规标签_公司年度.csv"))
vl["Stkcd"] = norm(vl["Stkcd"])
vl["year"] = vl["year"].astype(int)
vl_key = vl.set_index(["Stkcd","year"])[["first_fraud","repeat_fraud"]].to_dict("index")

data["is_first"]  = data.apply(lambda r: vl_key.get((r["Stkcd"], r["year"]), {}).get("first_fraud", np.nan), axis=1)
data["is_repeat"] = data.apply(lambda r: vl_key.get((r["Stkcd"], r["year"]), {}).get("repeat_fraud", np.nan), axis=1)

for label, col in [("First-time", "is_first"), ("Repeat", "is_repeat")]:
    idx = (data["Fraud"]==1) & (data[col]==1)
    if idx.sum() > 0:
        caught = (oof_cal[idx] >= THR).sum()
        print(f"  {label} (N={idx.sum()}): "
              f"Recall={caught/idx.sum():.4f}, "
              f"Mean_Prob={oof_cal[idx].mean():.4f}")

print("\n已保存: ViolationType_stratification.csv")

Fold 1 done
Fold 2 done
Fold 3 done
Fold 4 done
Fold 5 done

全局 F1 最优阈值 = 0.1819

违规类型分层结果
ViolationType                        TypeName  N_FraudFirmYears  Recall_on_this_type  Mean_Prob_fraud  Mean_Prob_nonfraud
        P2501              Fabricated profits               799               0.4931           0.2363              0.0834
        P2502      Fictitious asset reporting               113               0.5044           0.2320              0.0834
        P2503 False or misleading disclosures              3987               0.4231           0.2117              0.0834
        P2506  Inaccurate disclosures (other)               160               0.3188           0.1818              0.0834

单一类型 vs 多类型 fraud 的召回率
  1 种类型 (N=3480): Recall=0.3885, Mean_Prob=0.1993
  2 种类型 (N=680): Recall=0.5206, Mean_Prob=0.2475
  3 种类型 (N=73): Recall=0.5890, Mean_Prob=0.2649

首犯 vs 累犯 召回率
  First-time (N=1472): Recall=0.2677, Mean_Prob=0.1554
  Repeat (N=2761): Recall=0.4908, Mean_Prob=0.2363

已保存: Vi

In [10]:
# ============ threadpoolctl monkey-patch ============
import threadpoolctl
class _SafeThreadpoolLimits:
    def __init__(self, *a, **k): pass
    def __enter__(self): return self
    def __exit__(self, *a): pass
threadpoolctl.threadpool_limits = _SafeThreadpoolLimits
# ====================================================

import os, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              precision_recall_curve, f1_score, recall_score,
                              precision_score, confusion_matrix, brier_score_loss)
from scipy import stats
import lightgbm as lgb
import xgboost as xgb

OUT = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
data = pd.read_csv(os.path.join(OUT, "建模数据集_方案A_MDA_未隔离.csv"))
exclude_cols = ["Stkcd","year","Fraud","ShortName","IndustryName1","ViolationTypeID",
                "DeclareDate","DisposalDate","Enddate","set"]
feats = [c for c in data.columns
         if c not in exclude_cols and pd.api.types.is_numeric_dtype(data[c])]

y      = data["Fraud"].values
groups = data["Stkcd"].values
X_all  = data[feats].values
print(f"样本: {len(y)}, 正例: {int(y.sum())}, 特征: {len(feats)}")

# ---- 各 base learner 参数（用之前 Optuna / 默认参数）----
lgb_params = dict(n_estimators=400, learning_rate=0.0176, num_leaves=25,
                  min_child_samples=21, subsample=0.998, colsample_bytree=0.824,
                  reg_alpha=0.250, reg_lambda=0.277,
                  class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1)

xgb_params = dict(n_estimators=400, learning_rate=0.02, max_depth=6,
                  min_child_weight=5, subsample=0.8, colsample_bytree=0.75,
                  reg_alpha=0.15, reg_lambda=0.2, scale_pos_weight=10,
                  random_state=42, n_jobs=-1, eval_metric='logloss')

rf_params = dict(n_estimators=300, max_depth=15, min_samples_leaf=5,
                 class_weight='balanced', random_state=42, n_jobs=-1)

def best_thr(y, p):
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = 2*prec[:-1]*rec[:-1]/(prec[:-1]+rec[:-1]+1e-9)
    return float(thr[np.argmax(f1)])

def metrics(y, p, thr):
    yp = (p>=thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yp).ravel()
    return dict(AUC=roc_auc_score(y,p), PR_AUC=average_precision_score(y,p),
                Brier=brier_score_loss(y,p),
                F1=f1_score(y,yp,zero_division=0),
                Recall=recall_score(y,yp,zero_division=0),
                Precision=precision_score(y,yp,zero_division=0),
                Specificity=tn/(tn+fp) if (tn+fp)>0 else np.nan)

# ---- GroupKFold：生成三模型的 OOF + Stacking ----
gkf = GroupKFold(n_splits=5)
oof_lgb = np.zeros(len(y))
oof_xgb = np.zeros(len(y))
oof_rf  = np.zeros(len(y))
oof_stack = np.zeros(len(y))
fold_ids = np.zeros(len(y), dtype=int)

t0 = time.time()
for fold, (tr, te) in enumerate(gkf.split(X_all, y, groups), 1):
    print(f"\n===== Fold {fold} =====")
    X_tr_raw, X_te_raw = X_all[tr], X_all[te]
    y_tr, y_te = y[tr], y[te]

    imp = SimpleImputer(strategy='median')
    X_tr = imp.fit_transform(X_tr_raw); X_te = imp.transform(X_te_raw)

    # 1) LightGBM
    m_lgb = lgb.LGBMClassifier(**lgb_params).fit(X_tr, y_tr)
    p_lgb = m_lgb.predict_proba(X_te)[:,1]
    oof_lgb[te] = p_lgb

    # 2) XGBoost
    m_xgb = xgb.XGBClassifier(**xgb_params).fit(X_tr, y_tr)
    p_xgb = m_xgb.predict_proba(X_te)[:,1]
    oof_xgb[te] = p_xgb

    # 3) Random Forest（需要已填充数据）
    m_rf = RandomForestClassifier(**rf_params).fit(X_tr, y_tr)
    p_rf = m_rf.predict_proba(X_te)[:,1]
    oof_rf[te] = p_rf

    # 4) Stacking：内层 3 折 GroupKFold 生成 meta 特征
    inner = GroupKFold(n_splits=3)
    meta_tr = np.zeros((len(tr), 3))
    for i_tr, i_val in inner.split(X_tr, y_tr, groups[tr]):
        imp_i = SimpleImputer(strategy='median')
        Xi = imp_i.fit_transform(X_tr[i_tr]); Xv = imp_i.transform(X_tr[i_val])
        mi_lgb = lgb.LGBMClassifier(**lgb_params).fit(Xi, y_tr[i_tr])
        mi_xgb = xgb.XGBClassifier(**xgb_params).fit(Xi, y_tr[i_tr])
        mi_rf  = RandomForestClassifier(**rf_params).fit(Xi, y_tr[i_tr])
        meta_tr[i_val, 0] = mi_lgb.predict_proba(Xv)[:,1]
        meta_tr[i_val, 1] = mi_xgb.predict_proba(Xv)[:,1]
        meta_tr[i_val, 2] = mi_rf.predict_proba(Xv)[:,1]

    # Meta learner
    meta_test = np.column_stack([p_lgb, p_xgb, p_rf])
    meta_clf = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced')
    meta_clf.fit(meta_tr, y_tr)
    p_stack = meta_clf.predict_proba(meta_test)[:,1]
    oof_stack[te] = p_stack
    fold_ids[te] = fold

    # Fold metrics
    print(f"  LGB  AUC={roc_auc_score(y_te,p_lgb):.4f}")
    print(f"  XGB  AUC={roc_auc_score(y_te,p_xgb):.4f}")
    print(f"  RF   AUC={roc_auc_score(y_te,p_rf):.4f}")
    print(f"  STK  AUC={roc_auc_score(y_te,p_stack):.4f}")
    print(f"  Meta coef: {meta_clf.coef_[0]}")

print(f"\n总耗时: {(time.time()-t0)/60:.2f} min")

# ---- 全局校准 & 阈值 ----
def calibrate_and_eval(name, p_oof):
    thr_raw = 0.5
    met_raw = metrics(y, p_oof, thr_raw)
    # Isotonic
    iso = IsotonicRegression(out_of_bounds='clip').fit(p_oof, y)
    p_cal = iso.predict(p_oof)
    thr_cal = best_thr(y, p_cal)
    met_cal = metrics(y, p_cal, thr_cal)
    return met_raw, met_cal, p_cal, thr_cal

results = []
for name, p in [("LGB", oof_lgb), ("XGB", oof_xgb), ("RF", oof_rf), ("Stacking", oof_stack)]:
    met_raw, met_cal, p_cal, thr = calibrate_and_eval(name, p)
    results.append(dict(Model=name, Stage="Raw", **met_raw))
    results.append(dict(Model=name, Stage="Calibrated", **met_cal))
    print(f"\n[{name}]")
    print(f"  raw:  AUC={met_raw['AUC']:.4f} PR={met_raw['PR_AUC']:.4f} Brier={met_raw['Brier']:.4f}")
    print(f"  cal:  AUC={met_cal['AUC']:.4f} PR={met_cal['PR_AUC']:.4f} Brier={met_cal['Brier']:.4f} "
          f"F1={met_cal['F1']:.4f} R={met_cal['Recall']:.4f} P={met_cal['Precision']:.4f}")

df = pd.DataFrame(results)
df.to_csv(os.path.join(OUT, "Stacking_comparison.csv"), index=False, encoding='utf-8-sig')

# ---- DeLong: Stacking vs 单模型 ----
def delong(y_true, p1, p2):
    # ... 复用之前代码
    pass  # 完整版见下

# 简化：直接调 sklearn + scipy（不重复写 DeLong，用 scipy.stats）
print("\n" + "="*80)
print("完整结果表")
print("="*80)
print(df.round(4).to_string(index=False))

pd.DataFrame({"y_true": y, "lgb": oof_lgb, "xgb": oof_xgb,
              "rf": oof_rf, "stack": oof_stack, "fold": fold_ids}
             ).to_csv(os.path.join(OUT, "Stacking_OOF.csv"),
                      index=False, encoding='utf-8-sig')
print("\n已保存: Stacking_comparison.csv / Stacking_OOF.csv")

样本: 48328, 正例: 4233, 特征: 83

===== Fold 1 =====
  LGB  AUC=0.7980
  XGB  AUC=0.7955
  RF   AUC=0.7920
  STK  AUC=0.7988
  Meta coef: [3.27036432 0.60938236 1.24330789]

===== Fold 2 =====
  LGB  AUC=0.7758
  XGB  AUC=0.7747
  RF   AUC=0.7644
  STK  AUC=0.7754
  Meta coef: [3.16696283 0.42939212 1.590348  ]

===== Fold 3 =====
  LGB  AUC=0.7857
  XGB  AUC=0.7893
  RF   AUC=0.7841
  STK  AUC=0.7885
  Meta coef: [3.15806004 0.50301955 1.53519105]

===== Fold 4 =====
  LGB  AUC=0.7891
  XGB  AUC=0.7861
  RF   AUC=0.7854
  STK  AUC=0.7904
  Meta coef: [2.64555064 1.26387066 1.29535817]

===== Fold 5 =====
  LGB  AUC=0.7897
  XGB  AUC=0.7868
  RF   AUC=0.7837
  STK  AUC=0.7908
  Meta coef: [3.15650447 0.08059924 2.03789859]

总耗时: 10.15 min

[LGB]
  raw:  AUC=0.7878 PR=0.3032 Brier=0.1577
  cal:  AUC=0.7891 PR=0.2950 Brier=0.0700 F1=0.3514 R=0.4278 P=0.2982

[XGB]
  raw:  AUC=0.7865 PR=0.2988 Brier=0.1387
  cal:  AUC=0.7879 PR=0.2902 Brier=0.0703 F1=0.3464 R=0.4063 P=0.3019

[RF]
  raw:  AUC=

In [11]:
# ============ threadpoolctl monkey-patch ============
import threadpoolctl
class _SafeThreadpoolLimits:
    def __init__(self, *a, **k): pass
    def __enter__(self): return self
    def __exit__(self, *a): pass
threadpoolctl.threadpool_limits = _SafeThreadpoolLimits
# ====================================================

import os
import numpy as np
import pandas as pd
from scipy import stats

OUT = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
oof = pd.read_csv(os.path.join(OUT, "Stacking_OOF.csv"))
y = oof["y_true"].values.astype(int)
p_lgb = oof["lgb"].values.astype(float)
p_xgb = oof["xgb"].values.astype(float)
p_rf  = oof["rf"].values.astype(float)
p_stk = oof["stack"].values.astype(float)

# ---- DeLong（复用之前的实现）----
def compute_midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x)
    T = np.zeros(N, dtype=float); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]: j += 1
        T[i:j] = 0.5*(i+j-1) + 1
        i = j
    T2 = np.empty(N, dtype=float); T2[J] = T
    return T2

def fastDeLong(pred_sorted, label_1_count):
    m = label_1_count
    n = pred_sorted.shape[1] - m
    pos = pred_sorted[:, :m]; neg = pred_sorted[:, m:]
    k = pred_sorted.shape[0]
    tx = np.empty([k, m]); ty = np.empty([k, n]); tz = np.empty([k, m+n])
    for r in range(k):
        tx[r] = compute_midrank(pos[r])
        ty[r] = compute_midrank(neg[r])
        tz[r] = compute_midrank(pred_sorted[r])
    aucs = tz[:, :m].sum(axis=1)/m/n - float(m+1.0)/2.0/n
    v01 = (tz[:, :m] - tx)/n
    v10 = 1.0 - (tz[:, m:] - ty)/m
    sx = np.cov(v01); sy = np.cov(v10)
    delongcov = sx/m + sy/n
    return aucs, delongcov

def delong_test(y_true, p1, p2):
    order = np.argsort(-y_true)
    label_1_count = int(y_true.sum())
    pred_sorted = np.vstack([p1, p2])[:, order]
    aucs, cov = fastDeLong(pred_sorted, label_1_count)
    l = np.array([[1, -1]])
    z = float(np.abs(np.diff(aucs))[0] /
              np.sqrt(np.dot(np.dot(l, cov), l.T)).ravel()[0])
    p = float(2*stats.norm.sf(z))
    return aucs[0], aucs[1], z, p

print("=" * 80)
print("DeLong 检验：Stacking vs 单模型")
print("=" * 80)

comparisons = [
    ("Stacking vs LightGBM", p_stk, p_lgb),
    ("Stacking vs XGBoost",  p_stk, p_xgb),
    ("Stacking vs RF",       p_stk, p_rf),
    ("LightGBM vs XGBoost",  p_lgb, p_xgb),
    ("LightGBM vs RF",       p_lgb, p_rf),
]

rows = []
for name, p1, p2 in comparisons:
    a1, a2, z, p = delong_test(y, p1, p2)
    rows.append(dict(Comparison=name, AUC_1=a1, AUC_2=a2,
                     Delta_AUC=a1-a2, Z=z, p_value=p,
                     Significant=("Yes" if p < 0.05 else "No")))
    print(f"[{name}]  AUC1={a1:.4f}  AUC2={a2:.4f}  "
          f"Δ={a1-a2:+.4f}  Z={z:.3f}  p={p:.4g}  "
          f"{'✓ Sig' if p<0.05 else 'ns'}")

pd.DataFrame(rows).to_csv(
    os.path.join(OUT, "DeLong_Stacking_vs_Single.csv"),
    index=False, encoding='utf-8-sig')
print("\n已保存: DeLong_Stacking_vs_Single.csv")

DeLong 检验：Stacking vs 单模型
[Stacking vs LightGBM]  AUC1=0.7889  AUC2=0.7878  Δ=+0.0011  Z=3.222  p=0.001271  ✓ Sig
[Stacking vs XGBoost]  AUC1=0.7889  AUC2=0.7865  Δ=+0.0025  Z=3.173  p=0.00151  ✓ Sig
[Stacking vs RF]  AUC1=0.7889  AUC2=0.7819  Δ=+0.0070  Z=6.862  p=6.776e-12  ✓ Sig
[LightGBM vs XGBoost]  AUC1=0.7878  AUC2=0.7865  Δ=+0.0013  Z=1.430  p=0.1526  ns
[LightGBM vs RF]  AUC1=0.7878  AUC2=0.7819  Δ=+0.0059  Z=4.453  p=8.472e-06  ✓ Sig

已保存: DeLong_Stacking_vs_Single.csv


In [12]:
# ============ threadpoolctl monkey-patch（必须最前）============
import threadpoolctl
class _SafeThreadpoolLimits:
    def __init__(self, *a, **k): pass
    def __enter__(self): return self
    def __exit__(self, *a): pass
threadpoolctl.threadpool_limits = _SafeThreadpoolLimits
# ================================================================

import os, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              precision_recall_curve, f1_score, recall_score,
                              precision_score, confusion_matrix, brier_score_loss)
from scipy import stats
import lightgbm as lgb

OUT = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"

data = pd.read_csv(os.path.join(OUT, "建模数据集_方案A_MDA_未隔离.csv"))
exclude_cols = ["Stkcd","year","Fraud","ShortName","IndustryName1","ViolationTypeID",
                "DeclareDate","DisposalDate","Enddate","set"]
all_feats = [c for c in data.columns
             if c not in exclude_cols and pd.api.types.is_numeric_dtype(data[c])]

y      = data["Fraud"].values
groups = data["Stkcd"].values
X_all  = data[all_feats].values

best_params = dict(
    n_estimators=400, learning_rate=0.0176, num_leaves=25, min_child_samples=21,
    subsample=0.998, colsample_bytree=0.824, reg_alpha=0.250, reg_lambda=0.277,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
)

def best_thr(y, p):
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = 2*prec[:-1]*rec[:-1]/(prec[:-1]+rec[:-1]+1e-9)
    return float(thr[np.argmax(f1)])

def metrics(y, p, thr):
    yp = (p>=thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yp).ravel()
    return dict(AUC=roc_auc_score(y,p), PR_AUC=average_precision_score(y,p),
                Brier=brier_score_loss(y,p),
                F1=f1_score(y,yp,zero_division=0),
                Recall=recall_score(y,yp,zero_division=0),
                Precision=precision_score(y,yp,zero_division=0),
                Specificity=tn/(tn+fp) if (tn+fp)>0 else np.nan)

# ================= DeLong =================
def compute_midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x)
    T = np.zeros(N, dtype=float); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]: j += 1
        T[i:j] = 0.5*(i+j-1) + 1
        i = j
    T2 = np.empty(N, dtype=float); T2[J] = T
    return T2

def fastDeLong(pred_sorted, label_1_count):
    m = label_1_count; n = pred_sorted.shape[1] - m
    pos = pred_sorted[:, :m]; neg = pred_sorted[:, m:]
    k = pred_sorted.shape[0]
    tx = np.empty([k, m]); ty = np.empty([k, n]); tz = np.empty([k, m+n])
    for r in range(k):
        tx[r] = compute_midrank(pos[r])
        ty[r] = compute_midrank(neg[r])
        tz[r] = compute_midrank(pred_sorted[r])
    aucs = tz[:, :m].sum(axis=1)/m/n - float(m+1.0)/2.0/n
    v01 = (tz[:, :m] - tx)/n
    v10 = 1.0 - (tz[:, m:] - ty)/m
    sx = np.cov(v01); sy = np.cov(v10)
    delongcov = sx/m + sy/n
    return aucs, delongcov

def delong_test(y_true, p1, p2):
    order = np.argsort(-y_true)
    label_1_count = int(y_true.sum())
    pred_sorted = np.vstack([p1, p2])[:, order]
    aucs, cov = fastDeLong(pred_sorted, label_1_count)
    l = np.array([[1, -1]])
    z = float(np.abs(np.diff(aucs))[0] /
              np.sqrt(np.dot(np.dot(l, cov), l.T)).ravel()[0])
    p = float(2*stats.norm.sf(z))
    return aucs[0], aucs[1], z, p

# ==========================================================
# Part 0: McNemar 检验（用已有 OOF，秒出）
# ==========================================================
print("=" * 90)
print("Part 0: McNemar 检验")
print("=" * 90)

def mcnemar(y_true, p1, p2, thr):
    yp1 = (p1 >= thr).astype(int)
    yp2 = (p2 >= thr).astype(int)
    c1 = (yp1 == y_true)
    c2 = (yp2 == y_true)
    b = int((c1 & ~c2).sum())   # 模型1对、模型2错
    c = int((~c1 & c2).sum())   # 模型1错、模型2对
    n = b + c
    if n == 0:
        return b, c, 0.0, 1.0
    chi2 = (abs(b - c) - 1)**2 / n if n > 0 else 0.0
    p = float(stats.chi2.sf(chi2, df=1)) if n >= 25 else float(
        stats.binomtest(b, n, 0.5).pvalue)
    return b, c, chi2, p

# 加载已有 OOF
stk = pd.read_csv(os.path.join(OUT, "Stacking_OOF.csv"))
opt = pd.read_csv(os.path.join(OUT, "Optuna_calibrated_predictions.csv"))
smote = pd.read_csv(os.path.join(OUT, "SMOTE_OOF_predictions.csv"))

y_stk = stk["y_true"].values.astype(int)
y_opt = opt["y_true"].values.astype(int)
y_smote = smote["y_true"].values.astype(int)

assert np.array_equal(y_stk, y_opt) and np.array_equal(y_opt, y_smote), \
    "OOF 顺序不一致"

p_stk = stk["stack"].values.astype(float)  # 用 raw 因为要跟其他 raw 比
p_lgb = stk["lgb"].values.astype(float)
p_xgb = stk["xgb"].values.astype(float)
p_rf  = stk["rf"].values.astype(float)
p_smote = smote["y_prob_calibrated"].values.astype(float)
p_opt_cal = opt["y_prob_calibrated"].values.astype(float)

# 用全局 F1 最优阈值
def get_thr(p):
    return best_thr(y_stk, p)

comparisons_mc = [
    ("Stacking vs LightGBM", p_stk, p_lgb),
    ("Stacking vs XGBoost",  p_stk, p_xgb),
    ("LightGBM vs XGBoost",  p_lgb, p_xgb),
    ("Optuna-Cal vs Default-LGB", p_opt_cal, p_lgb),
    ("SMOTE-Cal vs Optuna-Cal", p_smote, p_opt_cal),
]

rows_mc = []
for name, p1, p2 in comparisons_mc:
    thr = get_thr(p1)  # 用模型1的最优阈值
    b, c, chi2, pv = mcnemar(y_stk, p1, p2, thr)
    rows_mc.append(dict(Comparison=name, b_only_1_correct=b,
                        c_only_2_correct=c, chi2=chi2, p_value=pv,
                        Significant=("Yes" if pv < 0.05 else "No")))
    print(f"[{name}]  b={b}, c={c}, chi2={chi2:.3f}, p={pv:.4g}  "
          f"{'✓ Sig' if pv<0.05 else 'ns'}")

pd.DataFrame(rows_mc).to_csv(
    os.path.join(OUT, "McNemar_results.csv"),
    index=False, encoding='utf-8-sig')
print("\n已保存: McNemar_results.csv")

# ==========================================================
# Part 1: 情感词典对比（Base dict vs DLUT dict）
# ==========================================================
print("\n" + "=" * 90)
print("Part 1: 情感词典对比")
print("=" * 90)

# 特征分组
NON_MDA = [c for c in all_feats if c not in [
    "TextualSimilarity", "PositiveVocabularyNum", "NegativeVocabularyNum",
    "EmotionTone1", "EmotionTone2",
    "PosRatio","NegRatio","SentLenAvg","SentLenStd","ComplexWordRatio",
    "DigitDensity","PuncDensity","TTR",
    "Jaccard_prev","EditSim_prev","TFIDF_Cosine_prev",
    "DLUT_PosNum","DLUT_NegNum","DLUT_PosRatio","DLUT_NegRatio",
    "DLUT_PosIntensity","DLUT_NegIntensity","DLUT_EmotionScore",
    "DLUT_EmotionTone","DLUT_NegAfterNeg","DLUT_PosAfterNeg"]]

BASE_DICT = ["PositiveVocabularyNum","NegativeVocabularyNum",
             "EmotionTone1","EmotionTone2"]
DLUT_DICT = ["DLUT_PosNum","DLUT_NegNum","DLUT_PosRatio","DLUT_NegRatio",
             "DLUT_PosIntensity","DLUT_NegIntensity","DLUT_EmotionScore",
             "DLUT_EmotionTone","DLUT_NegAfterNeg","DLUT_PosAfterNeg"]

feature_sets = {
    "NonMDA_only":    NON_MDA,
    "+BaseDict":      NON_MDA + BASE_DICT,
    "+DLUTDict":      NON_MDA + DLUT_DICT,
    "+BothDicts":     NON_MDA + BASE_DICT + DLUT_DICT,
    "+Full_MDA(83)":  all_feats,
}

dict_results = {}
dict_oofs = {}

for set_name, f_list in feature_sets.items():
    print(f"\n[{set_name}] n_features={len(f_list)}")
    X_sub = data[f_list].values
    gkf = GroupKFold(n_splits=5)
    oof = np.zeros(len(y))
    t0 = time.time()
    for fold, (tr, te) in enumerate(gkf.split(X_sub, y, groups), 1):
        imp = SimpleImputer(strategy='median')
        X_tr = imp.fit_transform(X_sub[tr]); X_te = imp.transform(X_sub[te])
        m = lgb.LGBMClassifier(**best_params).fit(X_tr, y[tr])
        oof[te] = m.predict_proba(X_te)[:,1]
    # 校准
    iso = IsotonicRegression(out_of_bounds='clip').fit(oof, y)
    oof_cal = iso.predict(oof)
    thr = best_thr(y, oof_cal)
    met = metrics(y, oof_cal, thr)
    met['n_features'] = len(f_list)
    dict_results[set_name] = met
    dict_oofs[set_name] = oof_cal
    print(f"  耗时 {time.time()-t0:.1f}s  "
          f"AUC={met['AUC']:.4f}  PR={met['PR_AUC']:.4f}  "
          f"Brier={met['Brier']:.4f}  F1={met['F1']:.4f}")

df_dict = pd.DataFrame(dict_results).T.reset_index()
df_dict.columns = ["FeatureSet"] + list(df_dict.columns[1:])
df_dict.to_csv(os.path.join(OUT, "Dictionary_comparison.csv"),
               index=False, encoding='utf-8-sig')

# DeLong: BaseDict vs DLUTDict
print("\nDeLong: BaseDict vs DLUTDict")
a1, a2, z, p = delong_test(y, dict_oofs["+BaseDict"], dict_oofs["+DLUTDict"])
print(f"  BaseDict AUC={a1:.4f}, DLUTDict AUC={a2:.4f}, Δ={a1-a2:+.4f}, "
      f"Z={z:.3f}, p={p:.4g}")
print("\nDeLong: BothDicts vs NonMDA_only")
a1, a2, z, p = delong_test(y, dict_oofs["+BothDicts"], dict_oofs["NonMDA_only"])
print(f"  BothDicts AUC={a1:.4f}, NonMDA AUC={a2:.4f}, Δ={a1-a2:+.4f}, "
      f"Z={z:.3f}, p={p:.4g}")

# ==========================================================
# Part 2: 相似度算法消融（leave-one-out）
# ==========================================================
print("\n" + "=" * 90)
print("Part 2: 相似度算法消融")
print("=" * 90)

SIM_FEATS = ["TextualSimilarity", "Jaccard_prev", "EditSim_prev", "TFIDF_Cosine_prev"]

abl_results = {}
abl_oofs = {}

# Baseline：完整 83 特征
print("\n[Full 83 feats]")
X_full = data[all_feats].values
gkf = GroupKFold(n_splits=5)
oof = np.zeros(len(y))
for tr, te in gkf.split(X_full, y, groups):
    imp = SimpleImputer(strategy='median')
    X_tr = imp.fit_transform(X_full[tr]); X_te = imp.transform(X_full[te])
    m = lgb.LGBMClassifier(**best_params).fit(X_tr, y[tr])
    oof[te] = m.predict_proba(X_te)[:,1]
iso = IsotonicRegression(out_of_bounds='clip').fit(oof, y)
oof_cal = iso.predict(oof)
thr = best_thr(y, oof_cal)
abl_results["Full(83)"] = metrics(y, oof_cal, thr)
abl_oofs["Full(83)"] = oof_cal
print(f"  AUC={abl_results['Full(83)']['AUC']:.4f}")

# Leave-one-out
for fdrop in SIM_FEATS:
    feats_sub = [c for c in all_feats if c != fdrop]
    print(f"\n[-{fdrop}]  n_features={len(feats_sub)}")
    X_sub = data[feats_sub].values
    oof = np.zeros(len(y))
    for tr, te in gkf.split(X_sub, y, groups):
        imp = SimpleImputer(strategy='median')
        X_tr = imp.fit_transform(X_sub[tr]); X_te = imp.transform(X_sub[te])
        m = lgb.LGBMClassifier(**best_params).fit(X_tr, y[tr])
        oof[te] = m.predict_proba(X_te)[:,1]
    iso = IsotonicRegression(out_of_bounds='clip').fit(oof, y)
    oof_cal = iso.predict(oof)
    thr = best_thr(y, oof_cal)
    name = f"-{fdrop}"
    abl_results[name] = metrics(y, oof_cal, thr)
    abl_oofs[name] = oof_cal
    a1, a2, z, p = delong_test(y, abl_oofs["Full(83)"], oof_cal)
    print(f"  AUC={abl_results[name]['AUC']:.4f}  "
          f"Δ(vs Full)={a1-a2:+.4f}  DeLong p={p:.4g}")

df_abl = pd.DataFrame(abl_results).T.reset_index()
df_abl.columns = ["Config"] + list(df_abl.columns[1:])
df_abl.to_csv(os.path.join(OUT, "Similarity_ablation.csv"),
              index=False, encoding='utf-8-sig')

# ==========================================================
# 汇总
# ==========================================================
print("\n" + "=" * 90)
print("全部完成")
print("=" * 90)
print("\n[McNemar_results.csv]")
print(pd.DataFrame(rows_mc).round(4).to_string(index=False))
print("\n[Dictionary_comparison.csv]")
print(df_dict[["FeatureSet","n_features","AUC","PR_AUC","Brier","F1","Recall","Precision"]].round(4).to_string(index=False))
print("\n[Similarity_ablation.csv]")
print(df_abl[["Config","AUC","PR_AUC","Brier","F1","Recall","Precision"]].round(4).to_string(index=False))

Part 0: McNemar 检验
[Stacking vs LightGBM]  b=596, c=2511, chi2=1179.078, p=2.149e-258  ✓ Sig
[Stacking vs XGBoost]  b=851, c=3356, chi2=1490.377, p=0  ✓ Sig
[LightGBM vs XGBoost]  b=518, c=1364, chi2=379.397, p=1.684e-84  ✓ Sig
[Optuna-Cal vs Default-LGB]  b=25623, c=2248, chi2=19602.593, p=0  ✓ Sig
[SMOTE-Cal vs Optuna-Cal]  b=2333, c=2158, chi2=6.741, p=0.00942  ✓ Sig

已保存: McNemar_results.csv

Part 1: 情感词典对比

[NonMDA_only] n_features=57
  耗时 10.4s  AUC=0.7878  PR=0.2932  Brier=0.0702  F1=0.3485

[+BaseDict] n_features=61
  耗时 10.2s  AUC=0.7887  PR=0.2966  Brier=0.0701  F1=0.3495

[+DLUTDict] n_features=67
  耗时 11.0s  AUC=0.7878  PR=0.2963  Brier=0.0701  F1=0.3498

[+BothDicts] n_features=71
  耗时 11.5s  AUC=0.7877  PR=0.2960  Brier=0.0701  F1=0.3501

[+Full_MDA(83)] n_features=83
  耗时 13.2s  AUC=0.7891  PR=0.2950  Brier=0.0700  F1=0.3514

DeLong: BaseDict vs DLUTDict
  BaseDict AUC=0.7887, DLUTDict AUC=0.7878, Δ=+0.0009, Z=1.285, p=0.1988

DeLong: BothDicts vs NonMDA_only
  BothDicts

In [13]:
# check_plot_data.py
import os
import pandas as pd
import numpy as np

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"

files = [
    "对比_OOF_predictions.csv",
    "Optuna_calibrated_predictions.csv",
    "Optuna_metrics.csv",
    "Optuna_best_params.csv",
    "SHAP_global_importance.csv",
    "SHAP_values_all.csv",
    "SHAP_feature_values.csv",
    "SHAP_stability_5fold.csv",
    "SMOTE_SHAP_global_importance.csv",
    "SMOTE_OOF_predictions.csv",
    "TimeSplit_test_predictions.csv",
    "TimeSplit_metrics.csv",
    "Stacking_OOF.csv",
    "Stacking_comparison.csv",
    "DeLong_results.csv",
    "McNemar_results.csv",
    "Desc_Yearly.csv",
    "Desc_Features.csv",
    "Desc_GroupTest.csv",
    "Dictionary_comparison.csv",
    "Similarity_ablation.csv",
    "ViolationType_stratification.csv",
]

for f in files:
    path = os.path.join(BASE, f)
    print("=" * 100)
    print("FILE:", f)
    if not os.path.exists(path):
        print("  [NOT FOUND]")
        continue
    try:
        df = pd.read_csv(path)
        print("  shape:", df.shape)
        print("  columns:", list(df.columns))
        print("  dtypes:")
        print(df.dtypes.to_string())
        print("  head(3):")
        print(df.head(3).to_string())
    except Exception as e:
        print("  [ERROR]", e)

# 检查绘图环境
print("=" * 100)
print("PLOT ENV")
try:
    import matplotlib
    print("matplotlib:", matplotlib.__version__)
except Exception as e:
    print("matplotlib missing:", e)
try:
    import seaborn as sns
    print("seaborn:", sns.__version__)
except Exception as e:
    print("seaborn missing:", e)
try:
    import shap
    print("shap:", shap.__version__)
except Exception as e:
    print("shap missing:", e)

FILE: 对比_OOF_predictions.csv
  shape: (48328, 7)
  columns: ['y_true', 'Base_LightGBM', 'Base_XGBoost', 'Base_RandomForest', 'Full_LightGBM', 'Full_XGBoost', 'Full_RandomForest']
  dtypes:
y_true                 int64
Base_LightGBM        float64
Base_XGBoost         float64
Base_RandomForest    float64
Full_LightGBM        float64
Full_XGBoost         float64
Full_RandomForest    float64
  head(3):
   y_true  Base_LightGBM  Base_XGBoost  Base_RandomForest  Full_LightGBM  Full_XGBoost  Full_RandomForest
0       0       0.007028      0.001757           0.133112       0.003886      0.000960           0.178843
1       0       0.004163      0.005526           0.153786       0.001799      0.000661           0.200296
2       0       0.002435      0.001524           0.133261       0.003516      0.005325           0.159595
FILE: Optuna_calibrated_predictions.csv
  shape: (48328, 3)
  columns: ['y_true', 'y_prob_raw', 'y_prob_calibrated']
  dtypes:
y_true                 int64
y_prob_raw       

  shape: (48328, 85)
  columns: ['y_true', 'y_prob', 'F010101A', 'F010201A', 'F010401A', 'F010701B', 'F010801B', 'F011201A', 'F080501A', 'F080601A', 'F081001B', 'F081101B', 'F081201B', 'F081601B', 'F082201B', 'F082701A', 'F060101B', 'F060301B', 'F050101B', 'F050201B', 'F050301B', 'F050401B', 'F050501B', 'F050901B', 'F053201B', 'F053301B', 'F051301B', 'F051701B', 'F053401B', 'F052101B', 'F053202B', 'F040101B', 'F040201B', 'F040401B', 'F040501B', 'F040801B', 'F041201B', 'F041401B', 'F041701B', 'F041801B', 'F070101B', 'F070201B', 'InternationalBig4', 'TotalAuditFee', 'ContrshrProportion', 'Mngmhldn', 'Boardsize', 'IndDirectorRatio', 'SupervisorSize', 'Y0301b', 'Y0501b', 'ChairmanHoldsharesRatio', 'ManagerHoldsharesRatio', 'Y1001b', 'LargestHolderRate', 'TopTenHoldersRate', 'IsDisclosingEvaRep', 'IsValid', 'IsDeficiency', 'TextualSimilarity', 'PositiveVocabularyNum', 'NegativeVocabularyNum', 'EmotionTone1', 'EmotionTone2', 'PosRatio', 'NegRatio', 'SentLenAvg', 'SentLenStd', 'ComplexWordRat

  shape: (83, 11)
  columns: ['Feature', 'Fraud_mean', 'Fraud_median', 'Fraud_std', 'NonFraud_mean', 'NonFraud_median', 'NonFraud_std', 't_stat', 't_pvalue', 'mw_pvalue', 'Sig_5pct']
  dtypes:
Feature             object
Fraud_mean         float64
Fraud_median       float64
Fraud_std          float64
NonFraud_mean      float64
NonFraud_median    float64
NonFraud_std       float64
t_stat             float64
t_pvalue           float64
mw_pvalue          float64
Sig_5pct              bool
  head(3):
    Feature  Fraud_mean  Fraud_median  Fraud_std  NonFraud_mean  NonFraud_median  NonFraud_std    t_stat      t_pvalue      mw_pvalue  Sig_5pct
0  F010101A    2.229699      1.469372   3.108266       2.703982         1.808568      3.248373 -9.427785  6.161244e-21   2.263465e-85      True
1  F010201A    1.814864      1.122762   2.899471       2.206322         1.369092      2.971258 -8.357315  8.173578e-17   1.086164e-62      True
2  F010401A    0.676657      0.269905   1.680866       0.918991    

In [1]:
# ============ threadpoolctl monkey-patch（必须最前）============
import threadpoolctl
class _SafeThreadpoolLimits:
    def __init__(self, *a, **k): pass
    def __enter__(self): return self
    def __exit__(self, *a): pass
threadpoolctl.threadpool_limits = _SafeThreadpoolLimits
# ================================================================

import os, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              precision_recall_curve, f1_score,
                              recall_score, precision_score,
                              confusion_matrix, brier_score_loss)

OUT = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
data = pd.read_csv(os.path.join(OUT, "建模数据集_方案A_MDA_未隔离.csv"))

exclude_cols = ["Stkcd","year","Fraud","ShortName","IndustryName1","ViolationTypeID",
                "DeclareDate","DisposalDate","Enddate","set"]
all_feats = [c for c in data.columns
             if c not in exclude_cols and pd.api.types.is_numeric_dtype(data[c])]

# 特征集划分
MDA_FEATS = ["TextualSimilarity","PositiveVocabularyNum","NegativeVocabularyNum",
             "EmotionTone1","EmotionTone2","PosRatio","NegRatio","SentLenAvg",
             "SentLenStd","ComplexWordRatio","DigitDensity","PuncDensity","TTR",
             "Jaccard_prev","EditSim_prev","TFIDF_Cosine_prev",
             "DLUT_PosNum","DLUT_NegNum","DLUT_PosRatio","DLUT_NegRatio",
             "DLUT_PosIntensity","DLUT_NegIntensity","DLUT_EmotionScore",
             "DLUT_EmotionTone","DLUT_NegAfterNeg","DLUT_PosAfterNeg"]
BASE = [f for f in all_feats if f not in MDA_FEATS]  # 57
FULL = all_feats  # 83

y      = data["Fraud"].values
groups = data["Stkcd"].values

def best_thr(y, p):
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = 2*prec[:-1]*rec[:-1]/(prec[:-1]+rec[:-1]+1e-9)
    return float(thr[np.argmax(f1)])

def metrics(y, p, thr):
    yp = (p>=thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yp).ravel()
    return dict(AUC=roc_auc_score(y,p), PR_AUC=average_precision_score(y,p),
                Brier=brier_score_loss(y,p), F1=f1_score(y,yp,zero_division=0),
                Recall=recall_score(y,yp,zero_division=0),
                Precision=precision_score(y,yp,zero_division=0),
                Specificity=tn/(tn+fp) if (tn+fp)>0 else np.nan)

def run_lasso(feat_list, name):
    print(f"\n[{name}] n_features={len(feat_list)}")
    X = data[feat_list].values
    gkf = GroupKFold(n_splits=5)
    oof = np.zeros(len(y))
    t0 = time.time()
    for fold, (tr, te) in enumerate(gkf.split(X, y, groups), 1):
        imp = SimpleImputer(strategy='median')
        X_tr = imp.fit_transform(X[tr]); X_te = imp.transform(X[te])
        sc = StandardScaler()
        X_tr = sc.fit_transform(X_tr); X_te = sc.transform(X_te)
        m = LogisticRegression(penalty='l1', solver='saga', C=1.0,
                               class_weight='balanced', max_iter=2000,
                               random_state=42, n_jobs=-1)
        m.fit(X_tr, y[tr])
        oof[te] = m.predict_proba(X_te)[:, 1]
    thr = best_thr(y, oof)
    met = metrics(y, oof, thr)
    met['n_features'] = len(feat_list)
    met['time_sec'] = round(time.time()-t0, 1)
    print(f"  耗时 {met['time_sec']}s  AUC={met['AUC']:.4f}  "
          f"PR={met['PR_AUC']:.4f}  Brier={met['Brier']:.4f}  "
          f"F1={met['F1']:.4f}  R={met['Recall']:.4f}  P={met['Precision']:.4f}")
    return met, oof

res_base, oof_base = run_lasso(BASE, "Base(57)")
res_full, oof_full = run_lasso(FULL, "Full(83)")

# 汇总保存
df = pd.DataFrame([
    dict(FeatureSet="Base", Model="Lasso-LR", **res_base),
    dict(FeatureSet="Full", Model="Lasso-LR", **res_full),
])
df.to_csv(os.path.join(OUT, "LassoLR_results.csv"),
          index=False, encoding='utf-8-sig')

# 保存 OOF
pd.DataFrame({"y_true": y, "lasso_base": oof_base,
              "lasso_full": oof_full}).to_csv(
    os.path.join(OUT, "LassoLR_OOF.csv"),
    index=False, encoding='utf-8-sig')

print("\n已保存: LassoLR_results.csv / LassoLR_OOF.csv")
print(df.round(4).to_string(index=False))


[Base(57)] n_features=57
  耗时 291.6s  AUC=0.7265  PR=0.2299  Brier=0.2064  F1=0.2933  R=0.3619  P=0.2465

[Full(83)] n_features=83
  耗时 202336.6s  AUC=0.7379  PR=0.2308  Brier=0.2028  F1=0.2946  R=0.4094  P=0.2301

已保存: LassoLR_results.csv / LassoLR_OOF.csv
FeatureSet    Model    AUC  PR_AUC  Brier     F1  Recall  Precision  Specificity  n_features  time_sec
      Base Lasso-LR 0.7265  0.2299 0.2064 0.2933  0.3619     0.2465       0.8938          57     291.6
      Full Lasso-LR 0.7379  0.2308 0.2028 0.2946  0.4094     0.2301       0.8685          83  202336.6


In [2]:
# ============ threadpoolctl monkey-patch ============
import threadpoolctl
class _SafeThreadpoolLimits:
    def __init__(self, *a, **k): pass
    def __enter__(self): return self
    def __exit__(self, *a): pass
threadpoolctl.threadpool_limits = _SafeThreadpoolLimits
# ====================================================

import os
import numpy as np
import pandas as pd
from scipy import stats

OUT = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"

# 加载 Lasso OOF 和对比 OOF
las = pd.read_csv(os.path.join(OUT, "LassoLR_OOF.csv"))
cmp = pd.read_csv(os.path.join(OUT, "对比_OOF_predictions.csv"))

y = las["y_true"].values.astype(int)
y_cmp = cmp["y_true"].values.astype(int)
assert np.array_equal(y, y_cmp), "OOF 顺序不一致"

p_lasso_base = las["lasso_base"].values.astype(float)
p_lasso_full = las["lasso_full"].values.astype(float)
p_lgb_base   = cmp["Base_LightGBM"].values.astype(float)
p_lgb_full   = cmp["Full_LightGBM"].values.astype(float)
p_rf_full    = cmp["Full_RandomForest"].values.astype(float)

# DeLong
def compute_midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x)
    T = np.zeros(N, dtype=float); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]: j += 1
        T[i:j] = 0.5*(i+j-1) + 1
        i = j
    T2 = np.empty(N, dtype=float); T2[J] = T
    return T2

def fastDeLong(pred_sorted, label_1_count):
    m = label_1_count; n = pred_sorted.shape[1] - m
    pos = pred_sorted[:, :m]; neg = pred_sorted[:, m:]
    k = pred_sorted.shape[0]
    tx = np.empty([k, m]); ty = np.empty([k, n]); tz = np.empty([k, m+n])
    for r in range(k):
        tx[r] = compute_midrank(pos[r])
        ty[r] = compute_midrank(neg[r])
        tz[r] = compute_midrank(pred_sorted[r])
    aucs = tz[:, :m].sum(axis=1)/m/n - float(m+1.0)/2.0/n
    v01 = (tz[:, :m] - tx)/n
    v10 = 1.0 - (tz[:, m:] - ty)/m
    sx = np.cov(v01); sy = np.cov(v10)
    cov = sx/m + sy/n
    return aucs, cov

def delong_test(y_true, p1, p2):
    order = np.argsort(-y_true)
    label_1_count = int(y_true.sum())
    pred = np.vstack([p1, p2])[:, order]
    aucs, cov = fastDeLong(pred, label_1_count)
    l = np.array([[1, -1]])
    z = float(np.abs(np.diff(aucs))[0] /
              np.sqrt(np.dot(np.dot(l, cov), l.T)).ravel()[0])
    p = float(2*stats.norm.sf(z))
    return aucs[0], aucs[1], z, p

print("=" * 80)
print("DeLong: Lasso-LR vs 树模型")
print("=" * 80)
comparisons = [
    ("Base: Lasso vs LGB",  p_lasso_base, p_lgb_base),
    ("Full: Lasso vs LGB",  p_lasso_full, p_lgb_full),
    ("Full: Lasso vs RF",   p_lasso_full, p_rf_full),
]
rows = []
for name, p1, p2 in comparisons:
    a1, a2, z, p = delong_test(y, p1, p2)
    rows.append(dict(Comparison=name, AUC_1=a1, AUC_2=a2,
                     Delta_AUC=a1-a2, Z=z, p_value=p,
                     Significant=("Yes" if p < 0.05 else "No")))
    print(f"[{name}]  Lasso={a1:.4f}  vs  Tree={a2:.4f}  "
          f"Δ={a1-a2:+.4f}  Z={z:.3f}  p={p:.4g}  "
          f"{'✓ Sig' if p<0.05 else 'ns'}")

pd.DataFrame(rows).to_csv(
    os.path.join(OUT, "DeLong_Lasso_vs_Trees.csv"),
    index=False, encoding='utf-8-sig')
print("\n已保存: DeLong_Lasso_vs_Trees.csv")

DeLong: Lasso-LR vs 树模型
[Base: Lasso vs LGB]  Lasso=0.7265  vs  Tree=0.7747  Δ=-0.0482  Z=14.380  p=6.937e-47  ✓ Sig
[Full: Lasso vs LGB]  Lasso=0.7379  vs  Tree=0.7753  Δ=-0.0374  Z=11.608  p=3.737e-31  ✓ Sig
[Full: Lasso vs RF]  Lasso=0.7379  vs  Tree=0.7819  Δ=-0.0440  Z=15.190  p=4.13e-52  ✓ Sig

已保存: DeLong_Lasso_vs_Trees.csv


In [3]:

# -*- coding: utf-8 -*-
"""
补画图 1 (ROC) 和图 2 (PR)，加入 Lasso-LR，统一标题为 Base/Full
输出到原 论文图表 文件夹
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (roc_curve, auc, precision_recall_curve,
                             average_precision_score)

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
FIG_DIR = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表"
os.makedirs(FIG_DIR, exist_ok=True)

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11
plt.rcParams['savefig.dpi'] = 300

df = pd.read_csv(os.path.join(BASE, "对比_OOF_predictions.csv"))
lasso = pd.read_csv(os.path.join(BASE, "LassoLR_OOF.csv"))

assert np.array_equal(df['y_true'].values, lasso['y_true'].values), "y_true 不对齐"

y = df['y_true'].values
lasso_base = lasso['lasso_base'].values
lasso_full = lasso['lasso_full'].values


def save(fig, name):
    png = os.path.join(FIG_DIR, name + ".png")
    pdf = os.path.join(FIG_DIR, name + ".pdf")
    fig.savefig(png, bbox_inches='tight', dpi=300)
    fig.savefig(pdf, bbox_inches='tight')
    plt.close(fig)
    print("[saved]", png)


# ==================== 图 1: ROC ====================
def fig1_roc():
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
    configs = [
        ('Base feature set (57 features)', [
            (df['Base_LightGBM'].values, 'LightGBM', '#1f77b4'),
            (df['Base_XGBoost'].values, 'XGBoost', '#ff7f0e'),
            (df['Base_RandomForest'].values, 'Random Forest', '#2ca02c'),
            (lasso_base, 'Lasso-LR', '#d62728'),
        ]),
        ('Full feature set (83 features)', [
            (df['Full_LightGBM'].values, 'LightGBM', '#1f77b4'),
            (df['Full_XGBoost'].values, 'XGBoost', '#ff7f0e'),
            (df['Full_RandomForest'].values, 'Random Forest', '#2ca02c'),
            (lasso_full, 'Lasso-LR', '#d62728'),
        ]),
    ]
    for ax, (title, items) in zip(axes, configs):
        for probs, label, color in items:
            fpr, tpr, _ = roc_curve(y, probs)
            roc_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, label=f'{label} (AUC={roc_auc:.4f})',
                    color=color, lw=1.8)
        ax.plot([0, 1], [0, 1], 'k--', lw=1)
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title(title, fontsize=11)
        ax.legend(loc='lower right', fontsize=9)
        ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
    plt.tight_layout()
    save(fig, "fig1_roc")


# ==================== 图 2: PR ====================
def fig2_pr():
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
    baseline = y.mean()
    configs = [
        ('Base feature set (57 features)', [
            (df['Base_LightGBM'].values, 'LightGBM', '#1f77b4'),
            (df['Base_XGBoost'].values, 'XGBoost', '#ff7f0e'),
            (df['Base_RandomForest'].values, 'Random Forest', '#2ca02c'),
            (lasso_base, 'Lasso-LR', '#d62728'),
        ]),
        ('Full feature set (83 features)', [
            (df['Full_LightGBM'].values, 'LightGBM', '#1f77b4'),
            (df['Full_XGBoost'].values, 'XGBoost', '#ff7f0e'),
            (df['Full_RandomForest'].values, 'Random Forest', '#2ca02c'),
            (lasso_full, 'Lasso-LR', '#d62728'),
        ]),
    ]
    for ax, (title, items) in zip(axes, configs):
        for probs, label, color in items:
            prec, rec, _ = precision_recall_curve(y, probs)
            ap = average_precision_score(y, probs)
            ax.plot(rec, prec, label=f'{label} (PR-AUC={ap:.4f})',
                    color=color, lw=1.8)
        ax.axhline(baseline, color='gray', ls='--', lw=1,
                   label=f'Baseline ({baseline:.3f})')
        ax.set_xlabel('Recall')
        ax.set_ylabel('Precision')
        ax.set_title(title, fontsize=11)
        ax.legend(loc='upper right', fontsize=9)
        ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
    plt.tight_layout()
    save(fig, "fig2_pr")


if __name__ == "__main__":
    fig1_roc()
    fig2_pr()
    print("Done.")

[saved] D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表\fig1_roc.png
[saved] D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\论文图表\fig2_pr.png
Done.


In [2]:
# -*- coding: utf-8 -*-
"""
修复版：过滤行业数据中的表头文字行，并诊断未匹配原因
"""
import os
import re
import pandas as pd

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
model_path = os.path.join(BASE, "建模数据集_方案A_MDA_未隔离.csv")
industry_path = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\行业数据.xlsx"

df = pd.read_csv(model_path, encoding='utf-8-sig')
ind = pd.read_excel(industry_path, sheet_name=0, dtype={'Symbol': str})

# ---------- 1. 过滤行业数据：Symbol 必须是 6 位数字 ----------
print("过滤前行数：", len(ind))
ind = ind[ind['Symbol'].astype(str).str.match(r'^\d{6}$', na=False)].copy()
print("过滤后行数：", len(ind))

# ---------- 2. 归一化 ----------
def norm_stkcd(s):
    return (s.astype(str)
             .str.replace(r'\.0$', '', regex=True)
             .str.strip()
             .str.zfill(6))

df['Stkcd'] = norm_stkcd(df['Stkcd'])
ind['Symbol'] = norm_stkcd(ind['Symbol'])

# ---------- 3. 提取年份 ----------
ind['year'] = pd.to_datetime(ind['EndDate'], errors='coerce').dt.year
ind = ind.dropna(subset=['year'])
ind['year'] = ind['year'].astype(int)

# ---------- 4. 去重 ----------
ind = ind.sort_values(['Symbol', 'year', 'EndDate'])
ind_dedup = ind.drop_duplicates(subset=['Symbol', 'year'], keep='last')
print("\n去重后行业数据：", ind_dedup.shape)

# ---------- 5. Merge ----------
df_merged = df.merge(
    ind_dedup[['Symbol', 'year', 'IndustryName']],
    left_on=['Stkcd', 'year'],
    right_on=['Symbol', 'year'],
    how='left'
)
if 'Symbol' in df_merged.columns:
    df_merged = df_merged.drop(columns=['Symbol'])

# ---------- 6. 覆盖率 ----------
coverage = df_merged['IndustryName'].notna().mean()
print("\n行业覆盖率：{:.2%}".format(coverage))
print("未匹配行数：", df_merged['IndustryName'].isna().sum())

# ---------- 7. 诊断：未匹配的公司有多少家？ ----------
unmatched = df_merged[df_merged['IndustryName'].isna()]
print("\n未匹配的唯一公司数：", unmatched['Stkcd'].nunique())
print("未匹配的公司列表（前 20）：", unmatched['Stkcd'].unique()[:20].tolist())

# 检查这些公司是否完全不在行业表里
unmatched_stocks = set(unmatched['Stkcd'].unique())
industry_stocks = set(ind_dedup['Symbol'].unique())
never_in_industry = unmatched_stocks - industry_stocks
print("\n完全不在行业表中的公司数：", len(never_in_industry))
print("示例：", list(never_in_industry)[:20])

# 这些公司的年份分布
if len(never_in_industry) > 0:
    never_df = df_merged[df_merged['Stkcd'].isin(never_in_industry)]
    print("\n这些公司的年份分布：")
    print(never_df['year'].value_counts().sort_index())
    print("\n这些公司是否更可能是舞弊样本？")
    print(never_df['Fraud'].value_counts(normalize=True))

# ---------- 8. 保存 ----------
out_path = os.path.join(BASE, "建模数据集_方案A_MDA_行业.csv")
df_merged.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f"\n已保存：{out_path}")
print("最终 shape：", df_merged.shape)

D:\aca\lib\site-packages\openpyxl\styles\stylesheet.py:221: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


过滤前行数： 46655
过滤后行数： 46653

去重后行业数据： (46653, 5)

行业覆盖率：86.01%
未匹配行数： 6763

未匹配的唯一公司数： 3094
未匹配的公司列表（前 20）： ['000005', '000018', '000022', '000023', '000024', '000033', '000038', '000040', '000043', '000150', '000413', '000418', '000502', '000511', '000540', '000585', '000587', '000594', '000606', '000611']

完全不在行业表中的公司数： 280
示例： ['001257', '688802', '603202', '688826', '900905', '688783', '920076', '900942', '301590', '920168', '920038', '920160', '920072', '301697', '900915', '920206', '688635', '603400', '001233', '688757']

这些公司的年份分布：
2015     85
2016     85
2017     85
2018     84
2019     81
2020     81
2021     80
2022    110
2023    243
2024    272
Name: year, dtype: int64

这些公司是否更可能是舞弊样本？
0    1.0
Name: Fraud, dtype: float64

已保存：D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\建模数据集_方案A_MDA_行业.csv
最终 shape： (48328, 89)


In [4]:
# -*- coding: utf-8 -*-
"""
行业异质性分析（86% 匹配样本，不依赖 IndustryName1）
输出：
  Industry_Stratification.csv
  Industry_FE_Comparison.csv
"""
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
OUT  = BASE

df = pd.read_csv(os.path.join(BASE, "建模数据集_方案A_MDA_行业.csv"), encoding='utf-8-sig')
print("总样本：", len(df))
print("列名示例：", df.columns.tolist()[:15])

# 只保留有行业信息的样本
df = df[df['IndustryName'].notna()].copy()
print("\n行业分析样本：", len(df))
print("制造业样本：", (df['IndustryName'] == '制造业').sum())
print("非制造业样本：", (df['IndustryName'] != '制造业').sum())

df['is_manufacturing'] = (df['IndustryName'] == '制造业').astype(int)

# 排除列 —— 只用实际存在的列
exclude_cols = ['Stkcd', 'year', 'Fraud', 'ShortName',
                'ViolationTypeID', 'DeclareDate', 'DisposalDate', 'Enddate', 'set',
                'IndustryName', 'is_manufacturing']
exclude_cols = [c for c in exclude_cols if c in df.columns]
feature_cols = [c for c in df.columns if c not in exclude_cols]
print("\n特征数：", len(feature_cols))

params = dict(
    n_estimators=400, learning_rate=0.0176, num_leaves=25,
    min_child_samples=21, subsample=0.998, subsample_freq=1,
    colsample_bytree=0.824, reg_alpha=0.250, reg_lambda=0.277,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
)

def eval_model(sub_df, feature_list, label):
    train = sub_df[sub_df['year'] <= 2021]
    test  = sub_df[sub_df['year'] >= 2023]
    if len(test) == 0 or test['Fraud'].sum() == 0:
        return {'sample': label, 'n_train': len(train), 'n_test': len(test),
                'AUC': np.nan, 'PR_AUC': np.nan, 'Recall': np.nan, 'Type_II_Error': np.nan}
    X_train, y_train = train[feature_list], train['Fraud'].values
    X_test,  y_test  = test[feature_list],  test['Fraud'].values
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, prob)
    pr_auc = average_precision_score(y_test, prob)
    y_pred = (prob >= 0.18).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    type2 = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    return {
        'sample': label, 'n_train': len(train), 'n_test': len(test),
        'AUC': round(auc, 4), 'PR_AUC': round(pr_auc, 4),
        'Recall': round(recall, 4), 'Type_II_Error': round(type2, 4),
        'TP': int(tp), 'FN': int(fn), 'FP': int(fp), 'TN': int(tn)
    }

# ---------- 1. 制造业 vs 非制造业 ----------
print("\n训练 Full sample 模型...")
res_full = eval_model(df, feature_cols, 'Full sample')
print("训练 Manufacturing 模型...")
res_mfg  = eval_model(df[df['is_manufacturing'] == 1], feature_cols, 'Manufacturing')
print("训练 Non-Manufacturing 模型...")
res_non  = eval_model(df[df['is_manufacturing'] == 0], feature_cols, 'Non-Manufacturing')

strat = pd.DataFrame([res_full, res_mfg, res_non])
strat.to_csv(os.path.join(OUT, 'Industry_Stratification.csv'),
             index=False, encoding='utf-8-sig')
print("\n制造业 vs 非制造业：")
print(strat.to_string(index=False))

# ---------- 2. 行业固定效应 ----------
print("\n加入行业固定效应...")
ind_dummies = pd.get_dummies(df['IndustryName'], prefix='IND')
df_fe = pd.concat([df, ind_dummies], axis=1)
fe_cols = feature_cols + list(ind_dummies.columns)
res_fe = eval_model(df_fe, fe_cols, 'Full + Industry FE')

fe_compare = pd.DataFrame([res_full, res_fe])
fe_compare.to_csv(os.path.join(OUT, 'Industry_FE_Comparison.csv'),
                  index=False, encoding='utf-8-sig')
print("\n行业固定效应前后对比：")
print(fe_compare.to_string(index=False))

总样本： 48328
列名示例： ['Stkcd', 'year', 'Fraud', 'F010101A', 'F010201A', 'F010401A', 'F010701B', 'F010801B', 'F011201A', 'F080501A', 'F080601A', 'F081001B', 'F081101B', 'F081201B', 'F081601B']

行业分析样本： 41565
制造业样本： 0
非制造业样本： 41565

特征数： 84

训练 Full sample 模型...


ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: TypeAuditOpin: object

In [5]:
# -*- coding: utf-8 -*-
"""
行业异质性分析（修复版）
- 制造业识别用 str.contains('制造业')
- 对 object 类型特征列做 one-hot 编码
"""
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
OUT  = BASE

df = pd.read_csv(os.path.join(BASE, "建模数据集_方案A_MDA_行业.csv"), encoding='utf-8-sig')
print("总样本：", len(df))

# 只保留有行业信息的样本
df = df[df['IndustryName'].notna()].copy()
print("行业分析样本：", len(df))

# ---------- 1. 制造业识别：用 contains ----------
df['is_manufacturing'] = df['IndustryName'].str.contains('制造业', na=False).astype(int)
print("\n制造业样本：", df['is_manufacturing'].sum())
print("非制造业样本：", (df['is_manufacturing'] == 0).sum())

# ---------- 2. 处理 object 类型特征列 ----------
exclude_cols = ['Stkcd', 'year', 'Fraud', 'ShortName',
                'ViolationTypeID', 'DeclareDate', 'DisposalDate', 'Enddate', 'set',
                'IndustryName', 'is_manufacturing']
exclude_cols = [c for c in exclude_cols if c in df.columns]
feature_cols = [c for c in df.columns if c not in exclude_cols]

# 找出 object 类型特征
obj_cols = [c for c in feature_cols if df[c].dtype == 'object']
print("\nobject 类型特征列：", obj_cols)

# 对 object 列做 one-hot（同时删掉原始列）
if obj_cols:
    df = pd.get_dummies(df, columns=obj_cols, prefix=obj_cols, dummy_na=False)
    # 重新计算 feature_cols
    feature_cols = [c for c in df.columns if c not in exclude_cols]
    print("one-hot 后特征数：", len(feature_cols))

# 确保所有特征都是数值型
non_numeric = [c for c in feature_cols if df[c].dtype not in ['int64', 'float64', 'int32', 'float32', 'bool', 'uint8']]
print("非数值型特征列：", non_numeric)
if non_numeric:
    for c in non_numeric:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    print("已尝试转换为数值型")

print("\n最终特征数：", len(feature_cols))

# ---------- 3. 模型参数 ----------
params = dict(
    n_estimators=400, learning_rate=0.0176, num_leaves=25,
    min_child_samples=21, subsample=0.998, subsample_freq=1,
    colsample_bytree=0.824, reg_alpha=0.250, reg_lambda=0.277,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
)

def eval_model(sub_df, feature_list, label):
    train = sub_df[sub_df['year'] <= 2021]
    test  = sub_df[sub_df['year'] >= 2023]
    if len(test) == 0 or test['Fraud'].sum() == 0:
        print(f"  [{label}] 测试集为空或无舞弊样本，跳过")
        return {'sample': label, 'n_train': len(train), 'n_test': len(test),
                'AUC': np.nan, 'PR_AUC': np.nan, 'Recall': np.nan, 'Type_II_Error': np.nan}
    X_train, y_train = train[feature_list], train['Fraud'].values
    X_test,  y_test  = test[feature_list],  test['Fraud'].values
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, prob)
    pr_auc = average_precision_score(y_test, prob)
    y_pred = (prob >= 0.18).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    type2 = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    return {
        'sample': label, 'n_train': len(train), 'n_test': len(test),
        'AUC': round(auc, 4), 'PR_AUC': round(pr_auc, 4),
        'Recall': round(recall, 4), 'Type_II_Error': round(type2, 4),
        'TP': int(tp), 'FN': int(fn), 'FP': int(fp), 'TN': int(tn)
    }

# ---------- 4. 制造业 vs 非制造业 ----------
print("\n训练 Full sample 模型...")
res_full = eval_model(df, feature_cols, 'Full sample')

print("训练 Manufacturing 模型...")
res_mfg  = eval_model(df[df['is_manufacturing'] == 1], feature_cols, 'Manufacturing')

print("训练 Non-Manufacturing 模型...")
res_non  = eval_model(df[df['is_manufacturing'] == 0], feature_cols, 'Non-Manufacturing')

strat = pd.DataFrame([res_full, res_mfg, res_non])
strat.to_csv(os.path.join(OUT, 'Industry_Stratification.csv'),
             index=False, encoding='utf-8-sig')
print("\n制造业 vs 非制造业：")
print(strat.to_string(index=False))

# ---------- 5. 行业固定效应 ----------
print("\n加入行业固定效应...")
# 用二级行业做 one-hot 会太多列；改用证监会一级门类
# 从 IndustryName 中提取大类
def map_to_category(name):
    if pd.isna(name):
        return 'Unknown'
    if '制造业' in name:
        return 'Manufacturing'
    if '信息传输' in name or '软件' in name:
        return 'IT'
    if '批发' in name or '零售' in name:
        return 'Wholesale_Retail'
    if '房地产' in name:
        return 'Real_Estate'
    if '建筑' in name:
        return 'Construction'
    if '金融' in name:
        return 'Finance'
    if '电力' in name or '热力' in name or '燃气' in name or '水' in name:
        return 'Utilities'
    if '交通' in name or '运输' in name or '仓储' in name or '邮政' in name:
        return 'Transport'
    if '农' in name or '林' in name or '牧' in name or '渔' in name:
        return 'Agriculture'
    if '采矿' in name:
        return 'Mining'
    if '住宿' in name or '餐饮' in name:
        return 'Hospitality'
    if '租赁' in name or '商务服务' in name:
        return 'Leasing_Business'
    if '科学研究' in name or '技术服务' in name:
        return 'Science_Tech'
    if '水利' in name or '环境' in name or '公共设施' in name:
        return 'Water_Environment'
    if '教育' in name:
        return 'Education'
    if '卫生' in name or '社会工作' in name:
        return 'Health'
    if '文化' in name or '体育' in name or '娱乐' in name:
        return 'Culture_Sports'
    if '综合' in name:
        return 'Conglomerate'
    return 'Other'

df['IndustryCategory'] = df['IndustryName'].apply(map_to_category)
print("\n行业门类分布：")
print(df['IndustryCategory'].value_counts())

# one-hot 行业门类
ind_dummies = pd.get_dummies(df['IndustryCategory'], prefix='IND')
df_fe = pd.concat([df, ind_dummies], axis=1)
fe_cols = feature_cols + list(ind_dummies.columns)
print("加入行业固定效应后特征数：", len(fe_cols))

res_fe = eval_model(df_fe, fe_cols, 'Full + Industry FE')

fe_compare = pd.DataFrame([res_full, res_fe])
fe_compare.to_csv(os.path.join(OUT, 'Industry_FE_Comparison.csv'),
                  index=False, encoding='utf-8-sig')
print("\n行业固定效应前后对比：")
print(fe_compare.to_string(index=False))

总样本： 48328
行业分析样本： 41565

制造业样本： 21724
非制造业样本： 19841

object 类型特征列： ['TypeAuditOpin']
one-hot 后特征数： 89
非数值型特征列： []

最终特征数： 89

训练 Full sample 模型...
训练 Manufacturing 模型...
训练 Non-Manufacturing 模型...

制造业 vs 非制造业：
           sample  n_train  n_test    AUC  PR_AUC  Recall  Type_II_Error  TP  FN   FP   TN
      Full sample    26018   10528 0.7440  0.2514  0.9634         0.0366 843  32 8012 1641
    Manufacturing    13206    5815 0.7449  0.2236  0.8953         0.1047 385  45 3764 1621
Non-Manufacturing    12812    4713 0.7274  0.2648  0.9213         0.0787 410  35 3205 1063

加入行业固定效应...

行业门类分布：
Manufacturing        21724
Other                 7679
IT                    2556
Wholesale_Retail      1737
Utilities             1486
Real_Estate           1187
Agriculture           1002
Construction           996
Transport              731
Water_Environment      688
Science_Tech           572
Leasing_Business       571
Conglomerate           260
Health                 120
Hospitality             

In [6]:
# -*- coding: utf-8 -*-
"""
行业异质性分析（阈值自适应版）
- 每个子集内部，用验证集（2022）选 F1 最优阈值
- 测试集（2023-2024）用该阈值评估
"""
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, f1_score

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
OUT  = BASE

df = pd.read_csv(os.path.join(BASE, "建模数据集_方案A_MDA_行业.csv"), encoding='utf-8-sig')
df = df[df['IndustryName'].notna()].copy()
print("行业分析样本：", len(df))

df['is_manufacturing'] = df['IndustryName'].str.contains('制造业', na=False).astype(int)

# 排除列
exclude_cols = ['Stkcd', 'year', 'Fraud', 'ShortName',
                'ViolationTypeID', 'DeclareDate', 'DisposalDate', 'Enddate', 'set',
                'IndustryName', 'is_manufacturing']
exclude_cols = [c for c in exclude_cols if c in df.columns]
feature_cols = [c for c in df.columns if c not in exclude_cols]

# 对 object 列 one-hot
obj_cols = [c for c in feature_cols if df[c].dtype == 'object']
if obj_cols:
    df = pd.get_dummies(df, columns=obj_cols, prefix=obj_cols, dummy_na=False)
    feature_cols = [c for c in df.columns if c not in exclude_cols]

print("特征数：", len(feature_cols))

params = dict(
    n_estimators=400, learning_rate=0.0176, num_leaves=25,
    min_child_samples=21, subsample=0.998, subsample_freq=1,
    colsample_bytree=0.824, reg_alpha=0.250, reg_lambda=0.277,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
)

def find_best_threshold(y_true, y_prob):
    """在验证集上找 F1 最优阈值"""
    thresholds = np.arange(0.05, 0.95, 0.01)
    best_f1, best_th = 0, 0.5
    for th in thresholds:
        y_pred = (y_prob >= th).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_th = f1, th
    return best_th, best_f1

def eval_model(sub_df, feature_list, label):
    train = sub_df[sub_df['year'] <= 2021]
    val   = sub_df[sub_df['year'] == 2022]
    test  = sub_df[sub_df['year'] >= 2023]

    if len(test) == 0 or test['Fraud'].sum() == 0 or len(val) == 0:
        return {'sample': label, 'n_train': len(train), 'n_test': len(test),
                'AUC': np.nan, 'PR_AUC': np.nan, 'Recall': np.nan,
                'Precision': np.nan, 'Type_II_Error': np.nan, 'threshold': np.nan}

    X_train, y_train = train[feature_list], train['Fraud'].values
    X_val,   y_val   = val[feature_list],   val['Fraud'].values
    X_test,  y_test  = test[feature_list],  test['Fraud'].values

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)

    # 验证集选阈值
    prob_val = model.predict_proba(X_val)[:, 1]
    best_th, _ = find_best_threshold(y_val, prob_val)

    # 测试集评估
    prob_test = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, prob_test)
    pr_auc = average_precision_score(y_test, prob_test)

    y_pred = (prob_test >= best_th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    type2 = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        'sample': label, 'n_train': len(train), 'n_test': len(test),
        'AUC': round(auc, 4), 'PR_AUC': round(pr_auc, 4),
        'Recall': round(recall, 4), 'Precision': round(precision, 4),
        'Type_II_Error': round(type2, 4),
        'threshold': round(best_th, 3),
        'TP': int(tp), 'FN': int(fn), 'FP': int(fp), 'TN': int(tn)
    }

# ---------- 1. 制造业 vs 非制造业 ----------
print("\n训练 Full sample ...")
res_full = eval_model(df, feature_cols, 'Full sample')

print("训练 Manufacturing ...")
res_mfg = eval_model(df[df['is_manufacturing'] == 1], feature_cols, 'Manufacturing')

print("训练 Non-Manufacturing ...")
res_non = eval_model(df[df['is_manufacturing'] == 0], feature_cols, 'Non-Manufacturing')

strat = pd.DataFrame([res_full, res_mfg, res_non])
strat.to_csv(os.path.join(OUT, 'Industry_Stratification.csv'),
             index=False, encoding='utf-8-sig')
print("\n制造业 vs 非制造业（阈值自适应）：")
print(strat.to_string(index=False))

# ---------- 2. 行业固定效应 ----------
print("\n加入行业固定效应 ...")

def map_to_category(name):
    if pd.isna(name): return 'Unknown'
    if '制造业' in name: return 'Manufacturing'
    if '信息传输' in name or '软件' in name: return 'IT'
    if '批发' in name or '零售' in name: return 'Wholesale_Retail'
    if '房地产' in name: return 'Real_Estate'
    if '建筑' in name: return 'Construction'
    if '金融' in name: return 'Finance'
    if '电力' in name or '热力' in name or '燃气' in name or '水' in name: return 'Utilities'
    if '交通' in name or '运输' in name or '仓储' in name or '邮政' in name: return 'Transport'
    if '农' in name or '林' in name or '牧' in name or '渔' in name: return 'Agriculture'
    if '采矿' in name: return 'Mining'
    if '住宿' in name or '餐饮' in name: return 'Hospitality'
    if '租赁' in name or '商务服务' in name: return 'Leasing_Business'
    if '科学研究' in name or '技术服务' in name: return 'Science_Tech'
    if '水利' in name or '环境' in name or '公共设施' in name: return 'Water_Environment'
    if '教育' in name: return 'Education'
    if '卫生' in name or '社会工作' in name: return 'Health'
    if '文化' in name or '体育' in name or '娱乐' in name: return 'Culture_Sports'
    if '综合' in name: return 'Conglomerate'
    return 'Other'

df['IndustryCategory'] = df['IndustryName'].apply(map_to_category)
ind_dummies = pd.get_dummies(df['IndustryCategory'], prefix='IND')
df_fe = pd.concat([df, ind_dummies], axis=1)
fe_cols = feature_cols + list(ind_dummies.columns)
print("加入行业 FE 后特征数：", len(fe_cols))

res_fe = eval_model(df_fe, fe_cols, 'Full + Industry FE')

fe_compare = pd.DataFrame([res_full, res_fe])
fe_compare.to_csv(os.path.join(OUT, 'Industry_FE_Comparison.csv'),
                  index=False, encoding='utf-8-sig')
print("\n行业固定效应前后对比：")
print(fe_compare.to_string(index=False))

行业分析样本： 41565
特征数： 89

训练 Full sample ...
训练 Manufacturing ...
训练 Non-Manufacturing ...

制造业 vs 非制造业（阈值自适应）：
           sample  n_train  n_test    AUC  PR_AUC  Recall  Precision  Type_II_Error  threshold  TP  FN   FP   TN
      Full sample    26018   10528 0.7440  0.2514  0.5166     0.2184         0.4834       0.62 452 423 1618 8035
    Manufacturing    13206    5815 0.7449  0.2236  0.4326     0.2366         0.5674       0.63 186 244  600 4785
Non-Manufacturing    12812    4713 0.7274  0.2648  0.6067     0.1918         0.3933       0.51 270 175 1138 3130

加入行业固定效应 ...
加入行业 FE 后特征数： 107

行业固定效应前后对比：
            sample  n_train  n_test    AUC  PR_AUC  Recall  Precision  Type_II_Error  threshold  TP  FN   FP   TN
       Full sample    26018   10528 0.7440  0.2514  0.5166     0.2184         0.4834       0.62 452 423 1618 8035
Full + Industry FE    26018   10528 0.7446  0.2554  0.5463     0.2110         0.4537       0.60 478 397 1787 7866


In [7]:
# -*- coding: utf-8 -*-
"""
阈值-成本分析：
  1. GroupKFold OOF (Optuna_calibrated_predictions.csv)
  2. TimeSplit Test (TimeSplit_test_predictions.csv)
输出：
  Threshold_Cost_GroupKFold.csv / .png
  Threshold_Cost_TimeSplit.csv / .png
  PR_HighRecall_GroupKFold.png
  PR_HighRecall_TimeSplit.png
  Threshold_Cost_Summary.csv
"""
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, confusion_matrix, roc_auc_score, average_precision_score

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
OUT = BASE

# ============================================================
# 通用函数
# ============================================================
def threshold_analysis(y_true, y_prob, label, out_prefix, cost_fn=10, cost_fp=1):
    """对给定预测做全阈值扫描"""
    thresholds = np.arange(0.02, 0.99, 0.01)
    rows = []
    for th in thresholds:
        y_pred = (y_prob >= th).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        t1   = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        t2   = fn / (fn + tp) if (fn + tp) > 0 else 0.0
        cost = cost_fn * fn + cost_fp * fp
        rows.append([th, prec, rec, f1, t1, t2, cost, tp, fp, fn, tn])
    res = pd.DataFrame(rows, columns=[
        "threshold", "precision", "recall", "f1",
        "type1_error", "type2_error", "cost_10fn_1fp",
        "tp", "fp", "fn", "tn"
    ])
    res.to_csv(os.path.join(OUT, f"{out_prefix}.csv"), index=False, encoding='utf-8-sig')

    best_f1 = res.loc[res["f1"].idxmax()]
    best_cost = res.loc[res["cost_10fn_1fp"].idxmin()]
    # 在成本最优阈值下的指标
    print(f"\n===== {label} =====")
    print(f"F1 最优阈值 = {best_f1['threshold']:.2f}  |  "
          f"Recall={best_f1['recall']:.4f}  Precision={best_f1['precision']:.4f}  "
          f"F1={best_f1['f1']:.4f}  TypeII={best_f1['type2_error']:.4f}")
    print(f"成本最优阈值 (10FN+1FP) = {best_cost['threshold']:.2f}  |  "
          f"Recall={best_cost['recall']:.4f}  Precision={best_cost['precision']:.4f}  "
          f"TypeII={best_cost['type2_error']:.4f}  成本={int(best_cost['cost_10fn_1fp'])}")

    # 高召回点
    pr_prec, pr_rec, _ = precision_recall_curve(y_true, y_prob)
    target_recalls = [0.5, 0.6, 0.7, 0.8]
    interp_prec = np.interp(target_recalls, pr_rec[::-1], pr_prec[::-1])
    print(f"\n高召回点：")
    for r, p in zip(target_recalls, interp_prec):
        print(f"  Recall={r:.2f}  ->  Precision={p:.4f}")

    # 图 1：阈值 vs 指标
    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax1.plot(res["threshold"], res["precision"], label="Precision", color="tab:blue")
    ax1.plot(res["threshold"], res["recall"], label="Recall", color="tab:orange")
    ax1.plot(res["threshold"], res["f1"], label="F1", color="tab:green")
    ax1.axvline(best_f1["threshold"], color="gray", linestyle="--",
                label=f"Best F1 (th={best_f1['threshold']:.2f})")
    ax1.axvline(best_cost["threshold"], color="red", linestyle="--",
                label=f"Best Cost (th={best_cost['threshold']:.2f})")
    ax1.set_xlabel("Threshold")
    ax1.set_ylabel("Score")
    ax1.set_title(f"Threshold vs Precision / Recall / F1  ({label})")
    ax1.legend(loc="upper right")
    ax1.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, f"{out_prefix}.png"), dpi=300)
    plt.close()

    # 图 2：PR 曲线 + 高召回点
    plt.figure(figsize=(7, 6))
    plt.plot(pr_rec, pr_prec, label=f"{label} PR curve", color="tab:blue")
    plt.scatter(target_recalls, interp_prec, color="red", zorder=5, s=60)
    for r, p in zip(target_recalls, interp_prec):
        plt.annotate(f"R={r:.1f}\nP={p:.3f}", xy=(r, p),
                     xytext=(r + 0.02, p + 0.05),
                     arrowprops=dict(arrowstyle="->", color="black"))
    baseline = y_true.mean()
    plt.axhline(baseline, color="gray", linestyle="--", label=f"Baseline ({baseline:.3f})")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"PR Curve with High-Recall Points  ({label})")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, f"PR_HighRecall_{out_prefix}.png"), dpi=300)
    plt.close()

    return {
        "dataset": label,
        "best_f1_threshold": round(best_f1["threshold"], 3),
        "best_f1_recall": round(best_f1["recall"], 4),
        "best_f1_precision": round(best_f1["precision"], 4),
        "best_f1_type2": round(best_f1["type2_error"], 4),
        "best_cost_threshold": round(best_cost["threshold"], 3),
        "best_cost_recall": round(best_cost["recall"], 4),
        "best_cost_precision": round(best_cost["precision"], 4),
        "best_cost_type2": round(best_cost["type2_error"], 4),
        "recall_0.5_precision": round(interp_prec[0], 4),
        "recall_0.6_precision": round(interp_prec[1], 4),
        "recall_0.7_precision": round(interp_prec[2], 4),
        "recall_0.8_precision": round(interp_prec[3], 4),
        "AUC": round(roc_auc_score(y_true, y_prob), 4),
        "PR_AUC": round(average_precision_score(y_true, y_prob), 4),
    }

# ============================================================
# 1. GroupKFold OOF
# ============================================================
print("=" * 60)
print("GroupKFold OOF 分析")
print("=" * 60)

gk_path = os.path.join(BASE, "Optuna_calibrated_predictions.csv")
df_gk = pd.read_csv(gk_path, encoding='utf-8-sig')
print("列名：", df_gk.columns.tolist())

y_true_gk = df_gk["y_true"].values
# 优先用校准概率
prob_col = "y_prob_calibrated" if "y_prob_calibrated" in df_gk.columns else "y_prob_raw"
y_prob_gk = df_gk[prob_col].values
print(f"使用概率列：{prob_col}")
print(f"样本量：{len(y_true_gk)}，舞弊率：{y_true_gk.mean():.4f}")

summary_gk = threshold_analysis(y_true_gk, y_prob_gk, "GroupKFold OOF",
                                 "Threshold_Cost_GroupKFold")

# ============================================================
# 2. TimeSplit Test
# ============================================================
print("\n" + "=" * 60)
print("TimeSplit Test 分析")
print("=" * 60)

ts_path = os.path.join(BASE, "TimeSplit_test_predictions.csv")
if os.path.exists(ts_path):
    df_ts = pd.read_csv(ts_path, encoding='utf-8-sig')
    print("列名：", df_ts.columns.tolist())

    y_true_ts = df_ts["y_true"].values
    # 尝试识别概率列
    if "prob_cal" in df_ts.columns:
        y_prob_ts = df_ts["prob_cal"].values
        pcol = "prob_cal"
    elif "y_prob_calibrated" in df_ts.columns:
        y_prob_ts = df_ts["y_prob_calibrated"].values
        pcol = "y_prob_calibrated"
    elif "prob_raw" in df_ts.columns:
        y_prob_ts = df_ts["prob_raw"].values
        pcol = "prob_raw"
    else:
        raise ValueError("找不到概率列，请检查列名")
    print(f"使用概率列：{pcol}")
    print(f"样本量：{len(y_true_ts)}，舞弊率：{y_true_ts.mean():.4f}")

    summary_ts = threshold_analysis(y_true_ts, y_prob_ts, "TimeSplit Test",
                                     "Threshold_Cost_TimeSplit")
else:
    print(f"⚠️ 未找到 {ts_path}，跳过 TimeSplit 分析")
    summary_ts = None

# ============================================================
# 3. 汇总表
# ============================================================
summary_rows = [summary_gk]
if summary_ts:
    summary_rows.append(summary_ts)
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(OUT, "Threshold_Cost_Summary.csv"),
                  index=False, encoding='utf-8-sig')
print("\n" + "=" * 60)
print("汇总表：")
print(summary_df.to_string(index=False))

print("\n完成！输出文件：")
print("  Threshold_Cost_GroupKFold.csv / .png")
print("  Threshold_Cost_TimeSplit.csv / .png")
print("  PR_HighRecall_GroupKFold.png")
print("  PR_HighRecall_TimeSplit.png")
print("  Threshold_Cost_Summary.csv")

GroupKFold OOF 分析
列名： ['y_true', 'y_prob_raw', 'y_prob_calibrated']
使用概率列：y_prob_calibrated
样本量：48328，舞弊率：0.0876

===== GroupKFold OOF =====
F1 最优阈值 = 0.18  |  Recall=0.4198  Precision=0.2938  F1=0.3457  TypeII=0.5802
成本最优阈值 (10FN+1FP) = 0.10  |  Recall=0.6834  Precision=0.1981  TypeII=0.3166  成本=25108

高召回点：
  Recall=0.50  ->  Precision=0.2623
  Recall=0.60  ->  Precision=0.2242
  Recall=0.70  ->  Precision=0.1933
  Recall=0.80  ->  Precision=0.1662

TimeSplit Test 分析
列名： ['y_true', 'prob_raw', 'prob_cal']
使用概率列：prob_cal
样本量：11237，舞弊率：0.0815

===== TimeSplit Test =====
F1 最优阈值 = 0.16  |  Recall=0.4814  Precision=0.2337  F1=0.3147  TypeII=0.5186
成本最优阈值 (10FN+1FP) = 0.13  |  Recall=0.5655  Precision=0.2151  TypeII=0.4345  成本=5870

高召回点：
  Recall=0.50  ->  Precision=0.2295
  Recall=0.60  ->  Precision=0.2026
  Recall=0.70  ->  Precision=0.1662
  Recall=0.80  ->  Precision=0.1347

汇总表：
       dataset  best_f1_threshold  best_f1_recall  best_f1_precision  best_f1_type2  best_cost_threshold

In [8]:
# -*- coding: utf-8 -*-
"""
Permutation Importance：
  1. 在时间外推训练集（2015-2021）上训练主模型 LightGBM
  2. 在测试集（2023-2024）上计算 permutation importance (scoring=roc_auc)
  3. 与 SHAP 排名对比（Spearman + Top20 重合度）
输出：
  Permutation_Importance.csv
  Permutation_Importance_Top20.png
  Permutation_vs_SHAP.csv
  Permutation_vs_SHAP_Comparison.png
"""
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
OUT  = BASE

# ============================================================
# 1. 读取建模数据
# ============================================================
data_path = os.path.join(BASE, "建模数据集_方案A_MDA_未隔离.csv")
df = pd.read_csv(data_path, encoding='utf-8-sig')
print("数据 shape：", df.shape)

# ---------- 排除列 ----------
exclude_cols = ['Stkcd', 'year', 'Fraud', 'ShortName', 'IndustryName1',
                'ViolationTypeID', 'DeclareDate', 'DisposalDate', 'Enddate', 'set']
exclude_cols = [c for c in exclude_cols if c in df.columns]
feature_cols = [c for c in df.columns if c not in exclude_cols]
print("初始特征数：", len(feature_cols))

# ---------- 处理 object 列（TypeAuditOpin）----------
obj_cols = [c for c in feature_cols if df[c].dtype == 'object']
print("object 特征列：", obj_cols)
if obj_cols:
    df = pd.get_dummies(df, columns=obj_cols, prefix=obj_cols, dummy_na=False)
    feature_cols = [c for c in df.columns if c not in exclude_cols]
    print("one-hot 后特征数：", len(feature_cols))

# 检查是否还有非数值列
non_num = [c for c in feature_cols if df[c].dtype not in
           ['int64', 'float64', 'int32', 'float32', 'bool', 'uint8']]
if non_num:
    print("仍存在非数值列：", non_num)
    for c in non_num:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    print("已尝试转换为数值型")

# ============================================================
# 2. 时间外推划分
# ============================================================
train = df[df['year'] <= 2021].copy()
val   = df[df['year'] == 2022].copy()
test  = df[df['year'] >= 2023].copy()
print(f"\nTrain: {len(train)}, Val: {len(val)}, Test: {len(test)}")
print(f"Train 舞弊率: {train['Fraud'].mean():.4f}")
print(f"Test 舞弊率: {test['Fraud'].mean():.4f}")

X_train, y_train = train[feature_cols], train['Fraud'].values
X_test,  y_test  = test[feature_cols],  test['Fraud'].values

# ============================================================
# 3. 主模型参数（与主实验一致）
# ============================================================
params = dict(
    n_estimators=400,
    learning_rate=0.0176,
    num_leaves=25,
    min_child_samples=21,
    subsample=0.998,
    subsample_freq=1,
    colsample_bytree=0.824,
    reg_alpha=0.250,
    reg_lambda=0.277,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

print("\n训练 LightGBM 主模型 ...")
model = lgb.LGBMClassifier(**params)
model.fit(X_train, y_train)

# 基线 AUC
prob_test = model.predict_proba(X_test)[:, 1]
baseline_auc = roc_auc_score(y_test, prob_test)
print(f"测试集基线 AUC: {baseline_auc:.4f}")

# ============================================================
# 4. Permutation Importance
# ============================================================
print("\n计算 Permutation Importance（n_repeats=10）...")
perm = permutation_importance(
    model, X_test, y_test,
    scoring='roc_auc',
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

perm_df = pd.DataFrame({
    "feature": feature_cols,
    "perm_importance_mean": perm.importances_mean,
    "perm_importance_std": perm.importances_std,
})
perm_df = perm_df.sort_values("perm_importance_mean", ascending=False).reset_index(drop=True)
perm_df["perm_rank"] = np.arange(1, len(perm_df) + 1)

perm_df.to_csv(os.path.join(OUT, "Permutation_Importance.csv"),
               index=False, encoding='utf-8-sig')

print("\nPermutation Importance Top 20：")
print(perm_df.head(20).to_string(index=False))

# ============================================================
# 5. Top 20 图
# ============================================================
top20 = perm_df.head(20).iloc[::-1]
plt.figure(figsize=(9, 8))
plt.barh(top20["feature"], top20["perm_importance_mean"],
         xerr=top20["perm_importance_std"], color='tab:blue', alpha=0.8)
plt.xlabel("Permutation Importance (AUC decrease)", fontsize=11)
plt.title("Top 20 Features by Permutation Importance", fontsize=13)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT, "Permutation_Importance_Top20.png"), dpi=300)
plt.close()
print("\n已保存：Permutation_Importance_Top20.png")

# ============================================================
# 6. 与 SHAP 对比
# ============================================================
shap_path = os.path.join(BASE, "SHAP_global_importance.csv")
if os.path.exists(shap_path):
    shap_df = pd.read_csv(shap_path, encoding='utf-8-sig')
    print("\nSHAP 列名：", shap_df.columns.tolist())

    # 自动识别 SHAP 特征列和重要性列
    shap_feat_col = None
    for c in ['feature', 'Feature', 'FeatureCode', 'feature_name']:
        if c in shap_df.columns:
            shap_feat_col = c
            break
    shap_imp_col = None
    for c in ['MeanAbsSHAP', 'mean_abs_shap', 'shap_importance', 'importance']:
        if c in shap_df.columns:
            shap_imp_col = c
            break

    if shap_feat_col and shap_imp_col:
        shap_df = shap_df[[shap_feat_col, shap_imp_col]].rename(
            columns={shap_feat_col: 'feature', shap_imp_col: 'shap_importance'})
        shap_df["shap_rank"] = shap_df["shap_importance"].rank(ascending=False, method='min')

        merged = perm_df.merge(shap_df, on='feature', how='inner')
        print(f"\n合并后特征数：{len(merged)}")

        # Spearman 相关性
        rho, pval = spearmanr(merged['perm_importance_mean'], merged['shap_importance'])
        print(f"\nPermutation vs SHAP Spearman rho = {rho:.4f}, p = {pval:.3e}")

        # Top 10 / Top 20 重合度
        top10_perm = set(merged.nsmallest(10, 'perm_rank')['feature'])
        top10_shap = set(merged.nsmallest(10, 'shap_rank')['feature'])
        top20_perm = set(merged.nsmallest(20, 'perm_rank')['feature'])
        top20_shap = set(merged.nsmallest(20, 'shap_rank')['feature'])
        print(f"Top 10 重合：{len(top10_perm & top10_shap)} / 10")
        print(f"Top 20 重合：{len(top20_perm & top20_shap)} / 20")

        # 保存
        merged_out = merged[['feature', 'perm_importance_mean', 'perm_importance_std',
                              'perm_rank', 'shap_importance', 'shap_rank']].copy()
        merged_out.to_csv(os.path.join(OUT, "Permutation_vs_SHAP.csv"),
                          index=False, encoding='utf-8-sig')

        # 对比图
        fig, axes = plt.subplots(1, 2, figsize=(14, 7))
        # 左：Permutation Top 15
        top15_perm = merged.nsmallest(15, 'perm_rank').sort_values('perm_rank', ascending=False)
        axes[0].barh(top15_perm['feature'], top15_perm['perm_importance_mean'],
                     xerr=top15_perm['perm_importance_std'], color='tab:blue', alpha=0.8)
        axes[0].set_xlabel('Permutation Importance (AUC decrease)')
        axes[0].set_title('Top 15 by Permutation Importance')
        axes[0].grid(axis='x', alpha=0.3)

        # 右：SHAP Top 15
        top15_shap = merged.nsmallest(15, 'shap_rank').sort_values('shap_rank', ascending=False)
        axes[1].barh(top15_shap['feature'], top15_shap['shap_importance'],
                     color='tab:orange', alpha=0.8)
        axes[1].set_xlabel('Mean |SHAP|')
        axes[1].set_title('Top 15 by SHAP')
        axes[1].grid(axis='x', alpha=0.3)

        plt.tight_layout()
        plt.savefig(os.path.join(OUT, "Permutation_vs_SHAP_Comparison.png"), dpi=300)
        plt.close()
        print("\n已保存：Permutation_vs_SHAP.csv 和 Permutation_vs_SHAP_Comparison.png")
    else:
        print("⚠️ 无法识别 SHAP 列名，跳过对比")
else:
    print(f"\n⚠️ 未找到 {shap_path}，跳过 SHAP 对比")

print("\n完成！输出文件：")
print("  Permutation_Importance.csv")
print("  Permutation_Importance_Top20.png")
print("  Permutation_vs_SHAP.csv")
print("  Permutation_vs_SHAP_Comparison.png")

数据 shape： (48328, 88)
初始特征数： 84
object 特征列： ['TypeAuditOpin']
one-hot 后特征数： 89

Train: 31553, Val: 5538, Test: 11237
Train 舞弊率: 0.0884
Test 舞弊率: 0.0815

训练 LightGBM 主模型 ...
测试集基线 AUC: 0.7668

计算 Permutation Importance（n_repeats=10）...

Permutation Importance Top 20：
              feature  perm_importance_mean  perm_importance_std  perm_rank
    TopTenHoldersRate              0.012967             0.003610          1
TypeAuditOpin_标准无保留意见              0.009744             0.002041          2
             F081601B              0.007629             0.001269          3
             F070101B              0.007513             0.001297          4
             F060101B              0.004679             0.001372          5
             F041701B              0.004308             0.001107          6
    LargestHolderRate              0.004268             0.000530          7
             F010701B              0.003613             0.001247          8
             F081001B              0.003353      

In [9]:
import pandas as pd
df = pd.read_csv(r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果\建模数据集_方案A_MDA_未隔离.csv")
print("列名：", df.columns.tolist())
if 'ShortName' in df.columns:
    st = df[df['ShortName'].astype(str).str.contains('ST', na=False)]
    print(f"ST/*ST 样本数：{len(st)} / {len(df)} = {len(st)/len(df):.2%}")
    print(f"ST 样本中舞弊率：{st['Fraud'].mean():.4f}")
else:
    print("没有 ShortName 列")

列名： ['Stkcd', 'year', 'Fraud', 'F010101A', 'F010201A', 'F010401A', 'F010701B', 'F010801B', 'F011201A', 'F080501A', 'F080601A', 'F081001B', 'F081101B', 'F081201B', 'F081601B', 'F082201B', 'F082701A', 'F060101B', 'F060301B', 'F050101B', 'F050201B', 'F050301B', 'F050401B', 'F050501B', 'F050901B', 'F053201B', 'F053301B', 'F051301B', 'F051701B', 'F053401B', 'F052101B', 'F053202B', 'F040101B', 'F040201B', 'F040401B', 'F040501B', 'F040801B', 'F041201B', 'F041401B', 'F041701B', 'F041801B', 'F070101B', 'F070201B', 'TypeAuditOpin', 'InternationalBig4', 'TotalAuditFee', 'ContrshrProportion', 'Mngmhldn', 'Boardsize', 'IndDirectorRatio', 'SupervisorSize', 'Y0301b', 'Y0501b', 'ChairmanHoldsharesRatio', 'ManagerHoldsharesRatio', 'Y1001b', 'LargestHolderRate', 'TopTenHoldersRate', 'IsDisclosingEvaRep', 'IsValid', 'IsDeficiency', 'set', 'TextualSimilarity', 'PositiveVocabularyNum', 'NegativeVocabularyNum', 'EmotionTone1', 'EmotionTone2', 'PosRatio', 'NegRatio', 'SentLenAvg', 'SentLenStd', 'ComplexWordR

In [10]:
# -*- coding: utf-8 -*-
"""
ST 敏感性检验：
  1. 从已有行业数据 Excel 提取 ShortName，merge 到建模数据
  2. 标记 ST/*ST
  3. 比较 Full sample vs Exclude ST 的时间外推性能
输出：
  ST_Sensitivity_Check.csv
  ST_Sensitivity_Comparison.csv
"""
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, f1_score

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\合并结果"
OUT = BASE
industry_path = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\行业数据.xlsx"

# ============================================================
# 1. 读取建模数据
# ============================================================
df = pd.read_csv(os.path.join(BASE, "建模数据集_方案A_MDA_未隔离.csv"), encoding='utf-8-sig')
print("建模数据 shape：", df.shape)

# ============================================================
# 2. 读取行业数据，提取 ShortName
# ============================================================
ind = pd.read_excel(industry_path, sheet_name=0, dtype={'Symbol': str})
ind = ind[ind['Symbol'].astype(str).str.match(r'^\d{6}$', na=False)].copy()

def norm_stkcd(s):
    return (s.astype(str)
             .str.replace(r'\.0$', '', regex=True)
             .str.strip()
             .str.zfill(6))

ind['Symbol'] = norm_stkcd(ind['Symbol'])
df['Stkcd'] = norm_stkcd(df['Stkcd'])

# 提取年份
ind['year'] = pd.to_datetime(ind['EndDate'], errors='coerce').dt.year
ind = ind.dropna(subset=['year'])
ind['year'] = ind['year'].astype(int)

# 去重
ind = ind.sort_values(['Symbol', 'year'])
ind_short = ind.drop_duplicates(subset=['Symbol', 'year'], keep='last')[['Symbol', 'year', 'ShortName']]

# ============================================================
# 3. Merge 到建模数据
# ============================================================
df = df.merge(ind_short, left_on=['Stkcd', 'year'],
              right_on=['Symbol', 'year'], how='left')
if 'Symbol' in df.columns:
    df = df.drop(columns=['Symbol'])

print("ShortName 覆盖率：{:.2%}".format(df['ShortName'].notna().mean()))

# ============================================================
# 4. 标记 ST/*ST
# ============================================================
df['is_ST'] = df['ShortName'].astype(str).str.contains('ST', na=False)
st_count = df['is_ST'].sum()
print(f"\nST/*ST 样本数：{st_count} / {len(df)} = {st_count/len(df):.2%}")
print(f"ST/*ST 中舞弊率：{df[df['is_ST']]['Fraud'].mean():.4f}")
print(f"非 ST 中舞弊率：{df[~df['is_ST']]['Fraud'].mean():.4f}")

# 保存检查表
check = pd.DataFrame({
    "Item": ["Total", "ST/*ST", "Non-ST", "ShortName matched", "ShortName missing"],
    "Count": [len(df), int(st_count), int((~df['is_ST']).sum()),
              int(df['ShortName'].notna().sum()), int(df['ShortName'].isna().sum())]
})
check.to_csv(os.path.join(OUT, "ST_Sensitivity_Check.csv"), index=False, encoding='utf-8-sig')

# ============================================================
# 5. 特征列
# ============================================================
exclude_cols = ['Stkcd', 'year', 'Fraud', 'ShortName', 'IndustryName1',
                'ViolationTypeID', 'DeclareDate', 'DisposalDate', 'Enddate', 'set', 'is_ST']
exclude_cols = [c for c in exclude_cols if c in df.columns]
feature_cols = [c for c in df.columns if c not in exclude_cols]

# 处理 object 列
obj_cols = [c for c in feature_cols if df[c].dtype == 'object']
if obj_cols:
    df = pd.get_dummies(df, columns=obj_cols, prefix=obj_cols, dummy_na=False)
    feature_cols = [c for c in df.columns if c not in exclude_cols]
    print("\none-hot 后特征数：", len(feature_cols))

print("最终特征数：", len(feature_cols))

# ============================================================
# 6. 模型参数（与主实验一致）
# ============================================================
params = dict(
    n_estimators=400, learning_rate=0.0176, num_leaves=25,
    min_child_samples=21, subsample=0.998, subsample_freq=1,
    colsample_bytree=0.824, reg_alpha=0.250, reg_lambda=0.277,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
)

def find_best_threshold(y_true, y_prob):
    thresholds = np.arange(0.05, 0.95, 0.01)
    best_f1, best_th = 0, 0.5
    for th in thresholds:
        y_pred = (y_prob >= th).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_th = f1, th
    return best_th

def eval_model(sub_df, feature_list, label):
    train = sub_df[sub_df['year'] <= 2021]
    val   = sub_df[sub_df['year'] == 2022]
    test  = sub_df[sub_df['year'] >= 2023]

    if len(test) == 0 or test['Fraud'].sum() == 0 or len(val) == 0:
        return None

    X_train, y_train = train[feature_list], train['Fraud'].values
    X_val,   y_val   = val[feature_list],   val['Fraud'].values
    X_test,  y_test  = test[feature_list],  test['Fraud'].values

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)

    # 验证集选阈值
    prob_val = model.predict_proba(X_val)[:, 1]
    best_th = find_best_threshold(y_val, prob_val)

    # 测试集评估
    prob_test = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, prob_test)
    pr_auc = average_precision_score(y_test, prob_test)

    y_pred = (prob_test >= best_th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    type2 = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        'sample': label,
        'n_train': len(train), 'n_test': len(test),
        'test_fraud_rate': round(y_test.mean(), 4),
        'AUC': round(auc, 4), 'PR_AUC': round(pr_auc, 4),
        'Recall': round(recall, 4), 'Precision': round(precision, 4),
        'Type_II_Error': round(type2, 4),
        'threshold': round(best_th, 3),
        'TP': int(tp), 'FN': int(fn), 'FP': int(fp), 'TN': int(tn)
    }

# ============================================================
# 7. 比较：Full sample vs Exclude ST
# ============================================================
print("\n训练 Full sample ...")
res_full = eval_model(df, feature_cols, "Full sample (includes ST)")

print("训练 Exclude ST ...")
res_no_st = eval_model(df[~df['is_ST']], feature_cols, "Exclude ST/*ST")

results = [r for r in [res_full, res_no_st] if r is not None]
comparison = pd.DataFrame(results)
comparison.to_csv(os.path.join(OUT, "ST_Sensitivity_Comparison.csv"),
                  index=False, encoding='utf-8-sig')

print("\nST 敏感性检验结果：")
print(comparison.to_string(index=False))

# 计算 AUC 差异
if len(results) == 2:
    auc_diff = results[0]['AUC'] - results[1]['AUC']
    print(f"\nAUC 差异（Full - Exclude ST）：{auc_diff:+.4f}")
    if abs(auc_diff) < 0.005:
        print("→ 差异 < 0.005，保留 ST 不影响主结论")
    else:
        print("→ 差异 > 0.005，需要在论文中讨论 ST 样本的影响")

print("\n完成！输出文件：")
print("  ST_Sensitivity_Check.csv")
print("  ST_Sensitivity_Comparison.csv")

建模数据 shape： (48328, 88)


D:\aca\lib\site-packages\openpyxl\styles\stylesheet.py:221: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


ShortName 覆盖率：86.01%

ST/*ST 样本数：1170 / 48328 = 2.42%
ST/*ST 中舞弊率：0.3769
非 ST 中舞弊率：0.0804

one-hot 后特征数： 89
最终特征数： 89

训练 Full sample ...
训练 Exclude ST ...

ST 敏感性检验结果：
                   sample  n_train  n_test  test_fraud_rate    AUC  PR_AUC  Recall  Precision  Type_II_Error  threshold  TP  FN   FP   TN
Full sample (includes ST)    31553   11237           0.0815 0.7668  0.2705  0.5229     0.2287         0.4771       0.67 479 437 1615 8706
           Exclude ST/*ST    30739   11005           0.0751 0.7479  0.2196  0.5496     0.1943         0.4504       0.64 454 372 1882 8297

AUC 差异（Full - Exclude ST）：+0.0189
→ 差异 > 0.005，需要在论文中讨论 ST 样本的影响

完成！输出文件：
  ST_Sensitivity_Check.csv
  ST_Sensitivity_Comparison.csv


In [11]:
# -*- coding: utf-8 -*-
"""
样本筛选表重建：
从已有的中间文件中逐步统计样本量
输出：
  Sample_Selection_Table.csv
  Sample_Selection_Detail.csv
"""
import os
import pandas as pd
import numpy as np

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载"
OUT  = os.path.join(BASE, "合并结果")

# ============================================================
# 最终样本（唯一确定的数据源）
# ============================================================
final_path = os.path.join(OUT, "建模数据集_方案A_MDA_未隔离.csv")
df_final = pd.read_csv(final_path, encoding='utf-8-sig')
print("=" * 60)
print("最终建模数据")
print("=" * 60)
print(f"Shape: {df_final.shape}")
print(f"Unique firms: {df_final['Stkcd'].nunique()}")
print(f"Year range: {df_final['year'].min()}-{df_final['year'].max()}")
print(f"Fraud: {int(df_final['Fraud'].sum())} ({df_final['Fraud'].mean():.2%})")

# ============================================================
# 尝试定位中间文件
# ============================================================
print("\n" + "=" * 60)
print("尝试定位中间文件")
print("=" * 60)

# 候选路径（按可能性排序）
candidate_files = {
    "财务面板": [
        os.path.join(BASE, "财务指标", "保留数据", "财务指标_合并面板.csv"),
        os.path.join(BASE, "财务指标", "保留数据", "财务面板.csv"),
        os.path.join(BASE, "财务指标", "保留数据"),
    ],
    "非财务面板": [
        os.path.join(BASE, "非财务指标", "保留数据", "非财务指标_合并面板.csv"),
        os.path.join(BASE, "非财务指标", "保留数据", "非财务面板.csv"),
        os.path.join(BASE, "非财务指标", "保留数据"),
    ],
    "MD&A数据": [
        os.path.join(BASE, "管理层讨论与分析", "保留数据", "mda_features_stage2_notext.pkl"),
    ],
    "违规信息": [
        os.path.join(BASE, "违规信息总表 104001230(仅供vip使用)", "违规信息总表.xlsx"),
    ],
}

found_files = {}
for name, paths in candidate_files.items():
    for p in paths:
        if os.path.exists(p):
            found_files[name] = p
            print(f"[找到] {name}: {p}")
            break
    else:
        print(f"[未找到] {name}")

# ============================================================
# 逐级统计
# ============================================================
rows = []

# ---------- 1. 最终样本 ----------
rows.append({
    "Step": "Final modeling sample (t−1 predictors → t fraud, 2015–2024)",
    "Firm-year observations": len(df_final),
    "Unique firms": df_final['Stkcd'].nunique(),
    "Fraud observations": int(df_final['Fraud'].sum()),
    "Non-fraud observations": int((df_final['Fraud'] == 0).sum()),
    "Fraud rate (%)": round(df_final['Fraud'].mean() * 100, 2),
})

# ---------- 2. 尝试从 MD&A 数据统计 ----------
mda_path = found_files.get("MD&A数据")
if mda_path and mda_path.endswith('.pkl'):
    try:
        mda = pd.read_pickle(mda_path)
        print(f"\nMD&A 数据 shape: {mda.shape}")
        print(f"MD&A 列名示例: {mda.columns.tolist()[:10]}")
        if 'Stkcd' in mda.columns and 'year' in mda.columns:
            mda_unique = mda.drop_duplicates(subset=['Stkcd', 'year'])
            rows.append({
                "Step": "MD&A text available (before merge with financial/non-financial)",
                "Firm-year observations": len(mda_unique),
                "Unique firms": mda_unique['Stkcd'].nunique(),
                "Fraud observations": np.nan,
                "Non-fraud observations": np.nan,
                "Fraud rate (%)": np.nan,
            })
    except Exception as e:
        print(f"读取 MD&A 失败: {e}")

# ---------- 3. 尝试从财务面板统计 ----------
fin_path = found_files.get("财务面板")
if fin_path:
    if os.path.isdir(fin_path):
        # 如果只找到目录，列出目录里的文件
        print(f"\n财务目录内容: {os.listdir(fin_path)}")
    else:
        try:
            fin = pd.read_csv(fin_path, encoding='utf-8-sig')
            print(f"\n财务面板 shape: {fin.shape}")
            if 'Stkcd' in fin.columns and 'year' in fin.columns:
                fin_unique = fin.drop_duplicates(subset=['Stkcd', 'year'])
                rows.append({
                    "Step": "Financial variables available",
                    "Firm-year observations": len(fin_unique),
                    "Unique firms": fin_unique['Stkcd'].nunique(),
                    "Fraud observations": np.nan,
                    "Non-fraud observations": np.nan,
                    "Fraud rate (%)": np.nan,
                })
        except Exception as e:
            print(f"读取财务面板失败: {e}")

# ---------- 4. 尝试从违规信息统计 ----------
vio_path = found_files.get("违规信息")
if vio_path and vio_path.endswith('.xlsx'):
    try:
        vio = pd.read_excel(vio_path)
        print(f"\n违规信息 shape: {vio.shape}")
        print(f"违规信息列名: {vio.columns.tolist()}")
        # 只统计四类舞弊
        if 'ViolationTypeID' in vio.columns:
            target_types = ['P2501', 'P2502', 'P2503', 'P2506']
            vio_target = vio[vio['ViolationTypeID'].isin(target_types)]
            if 'Stkcd' in vio.columns and 'year' in vio.columns:
                vio_unique = vio_target.drop_duplicates(subset=['Stkcd', 'year'])
                rows.append({
                    "Step": "Fraud labels (P2501/P2502/P2503/P2506)",
                    "Firm-year observations": len(vio_unique),
                    "Unique firms": vio_unique['Stkcd'].nunique(),
                    "Fraud observations": len(vio_unique),
                    "Non-fraud observations": np.nan,
                    "Fraud rate (%)": np.nan,
                })
    except Exception as e:
        print(f"读取违规信息失败: {e}")

# ============================================================
# 如果中间文件缺失，用占位符构建模板
# ============================================================
template_steps = [
    "Initial sample: CSMAR A-share listed firms (2014–2024)",
    "After excluding financial industry (banks, insurance, securities)",
    "After excluding firms with missing key financial variables",
    "After excluding firms with missing non-financial variables",
    "After excluding firms without MD&A text",
    "Final sample used for modeling (2015–2024, t−1 predictors)",
]

# 如果只有最终一行，补全模板
if len(rows) == 1:
    print("\n" + "=" * 60)
    print("⚠️ 中间文件未找到，生成模板表")
    print("=" * 60)
    template = pd.DataFrame({
        "Step": template_steps,
        "Firm-year observations": ["?", "?", "?", "?", "?", len(df_final)],
        "Unique firms": ["?", "?", "?", "?", "?", df_final['Stkcd'].nunique()],
        "Fraud observations": ["?", "?", "?", "?", "?", int(df_final['Fraud'].sum())],
        "Fraud rate (%)": ["?", "?", "?", "?", "?", round(df_final['Fraud'].mean() * 100, 2)],
    })
    template.to_csv(os.path.join(OUT, "Sample_Selection_Table.csv"),
                    index=False, encoding='utf-8-sig')
    print("\n已生成模板：Sample_Selection_Table.csv")
    print("你需要从以下位置手工填入中间步骤的样本量：")
    print("  - CSMAR 原始下载面板")
    print("  - 财务指标/非财务指标/违规信息的合并脚本日志")
    print("  - 或重新跑一次清洗流程并记录每步的 shape")
else:
    table = pd.DataFrame(rows)
    table.to_csv(os.path.join(OUT, "Sample_Selection_Table.csv"),
                 index=False, encoding='utf-8-sig')
    print("\n已生成：Sample_Selection_Table.csv")
    print(table.to_string(index=False))

# ============================================================
# 年度分布 + 行业分布（论文 Table 1）
# ============================================================
print("\n" + "=" * 60)
print("年度分布")
print("=" * 60)

yearly = df_final.groupby('year')['Fraud'].agg(['count', 'sum']).reset_index()
yearly.columns = ['year', 'N', 'Fraud']
yearly['NonFraud'] = yearly['N'] - yearly['Fraud']
yearly['FraudRate(%)'] = (yearly['Fraud'] / yearly['N'] * 100).round(2)
yearly.to_csv(os.path.join(OUT, "Sample_Yearly_Distribution.csv"),
              index=False, encoding='utf-8-sig')
print(yearly.to_string(index=False))

# 行业分布（如果有 IndustryName）
industry_path = os.path.join(OUT, "建模数据集_方案A_MDA_行业.csv")
if os.path.exists(industry_path):
    df_ind = pd.read_csv(industry_path, encoding='utf-8-sig')
    df_ind = df_ind[df_ind['IndustryName'].notna()]
    ind = df_ind.groupby('IndustryName')['Fraud'].agg(['count', 'sum']).reset_index()
    ind.columns = ['Industry', 'N', 'Fraud']
    ind['NonFraud'] = ind['N'] - ind['Fraud']
    ind['FraudRate(%)'] = (ind['Fraud'] / ind['N'] * 100).round(2)
    ind = ind.sort_values('N', ascending=False)
    ind.to_csv(os.path.join(OUT, "Sample_Industry_Distribution.csv"),
               index=False, encoding='utf-8-sig')
    print("\n行业分布（Top 15）：")
    print(ind.head(15).to_string(index=False))

print("\n" + "=" * 60)
print("完成！")
print("=" * 60)

最终建模数据
Shape: (48328, 88)
Unique firms: 5791
Year range: 2015-2024
Fraud: 4233 (8.76%)

尝试定位中间文件
[找到] 财务面板: D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\财务指标\保留数据
[找到] 非财务面板: D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\非财务指标\保留数据
[找到] MD&A数据: D:\科研\二次实验\二次实验\二次实验\国泰安数据下载\管理层讨论与分析\保留数据\mda_features_stage2_notext.pkl
[未找到] 违规信息

MD&A 数据 shape: (45658, 37)
MD&A 列名示例: ['Symbol', 'ShortName', 'Enddate', 'IndustryName1', 'TextualSimilarity', 'PositiveVocabularyNum', 'NegativeVocabularyNum', 'TotalWordsNum', 'SentencesNum', 'WordsNum']

财务目录内容: ['偿债能力.xlsx', '发展能力.xlsx', '现金流分析.xlsx', '盈利能力.xlsx', '经营能力.xlsx', '风险水平.xlsx']

已生成：Sample_Selection_Table.csv
                                                           Step  Firm-year observations  Unique firms  Fraud observations  Non-fraud observations  Fraud rate (%)
    Final modeling sample (t−1 predictors → t fraud, 2015–2024)                   48328          5791              4233.0                 44095.0            8.76
MD&A text available (before merge with financial/no

In [12]:
# -*- coding: utf-8 -*-
"""
生成样本来源与合并表（适合 PLOS ONE）
输出：
  Sample_Selection_Table_Final.csv
"""
import os
import pandas as pd

BASE = r"D:\科研\二次实验\二次实验\二次实验\国泰安数据下载"
OUT  = os.path.join(BASE, "合并结果")

# ============================================================
# 读取最终数据
# ============================================================
df = pd.read_csv(os.path.join(OUT, "建模数据集_方案A_MDA_未隔离.csv"), encoding='utf-8-sig')

# ============================================================
# 读取 MD&A 数据
# ============================================================
mda = pd.read_pickle(os.path.join(BASE, "管理层讨论与分析", "保留数据",
                                    "mda_features_stage2_notext.pkl"))

# ============================================================
# 统计每个数据源的覆盖
# ============================================================
rows = []

# 最终样本
rows.append({
    "Data source": "Final modeling sample",
    "Firm-year observations": len(df),
    "Unique firms": df['Stkcd'].nunique(),
    "Fraud observations": int(df['Fraud'].sum()),
    "Fraud rate (%)": round(df['Fraud'].mean() * 100, 2),
    "Notes": "t−1 predictors → t fraud, 2015–2024"
})

# MD&A
rows.append({
    "Data source": "MD&A text features (CSMAR BDT_MDAEmotAnal)",
    "Firm-year observations": len(mda.drop_duplicates(subset=['Symbol', 'Enddate'])),
    "Unique firms": mda['Symbol'].nunique(),
    "Fraud observations": None,
    "Fraud rate (%)": None,
    "Notes": "Financial, non-financial, and MD&A features merged; MD&A missing values retained as NaN"
})

# 财务指标（从目录推断）
fin_dir = os.path.join(BASE, "财务指标", "保留数据")
fin_files = [f for f in os.listdir(fin_dir) if f.endswith('.xlsx')]
rows.append({
    "Data source": "Financial indicators (CSMAR)",
    "Firm-year observations": None,
    "Unique firms": None,
    "Fraud observations": None,
    "Fraud rate (%)": None,
    "Notes": f"{len(fin_files)} categories: {', '.join([f.replace('.xlsx','') for f in fin_files])}"
})

# 非财务
nonfin_dir = os.path.join(BASE, "非财务指标", "保留数据")
nonfin_files = [f for f in os.listdir(nonfin_dir) if f.endswith('.xlsx') or f.endswith('.csv')]
rows.append({
    "Data source": "Non-financial indicators (CSMAR)",
    "Firm-year observations": None,
    "Unique firms": None,
    "Fraud observations": None,
    "Fraud rate (%)": None,
    "Notes": f"{len(nonfin_files)} files"
})

# 违规信息
rows.append({
    "Data source": "Fraud labels (CSMAR violation database)",
    "Firm-year observations": 4233,
    "Unique firms": None,
    "Fraud observations": 4233,
    "Fraud rate (%)": None,
    "Notes": "P2501, P2502, P2503, P2506; ViolationYear used as outcome year"
})

table = pd.DataFrame(rows)
table.to_csv(os.path.join(OUT, "Sample_Selection_Table_Final.csv"),
             index=False, encoding='utf-8-sig')
print(table.to_string(index=False))

                               Data source  Firm-year observations  Unique firms  Fraud observations  Fraud rate (%)                                                                                   Notes
                     Final modeling sample                 48328.0        5791.0              4233.0            8.76                                                     t−1 predictors → t fraud, 2015–2024
MD&A text features (CSMAR BDT_MDAEmotAnal)                 45658.0        5493.0                 NaN             NaN Financial, non-financial, and MD&A features merged; MD&A missing values retained as NaN
              Financial indicators (CSMAR)                     NaN           NaN                 NaN             NaN                                       6 categories: 偿债能力, 发展能力, 现金流分析, 盈利能力, 经营能力, 风险水平
          Non-financial indicators (CSMAR)                     NaN           NaN                 NaN             NaN                                                                